# EDA 1 — Structural Audit

_Cleaned & restructured version generated on 2026-03-29 13:21:23._

In [1]:
# --- Configuration -------------------------------------------------------
# All file paths are now parameterised; change DATA_DIR to point at your data.
from pathlib import Path
DATA_DIR = Path("data")  # <-- adjust this
TRAIN_FILE = DATA_DIR / "final_train_before_eda.csv"
TEST_FILE = DATA_DIR / "final_test_before_eda.csv"

# Step 0 Import libraries and file path

In [2]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT_DIR = Path(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk")

processed_dir = ROOT_DIR / "Data" / "Processed"

train_file = "final_train_before_eda.csv"
test_file = "final_test_before_eda.csv"

train_path = processed_dir / train_file
test_path = processed_dir / test_file

if not train_path.exists():
    raise FileNotFoundError(f"Train file not found: {train_path}")

if not test_path.exists():
    raise FileNotFoundError(f"Test file not found: {test_path}")

train_df = pd.read_csv(train_path, low_memory=False)
test_df = pd.read_csv(test_path, low_memory=False)

print(f"train_csv shape:{train_df.shape} ")
print(f"test_csv shape:{test_df.shape} ")

# target column 
TARGET = "TARGET"

train_csv shape:(307511, 584) 
test_csv shape:(48744, 583) 


# Step 1. Identify And Structural Audit

### 1.1 current Datatype and suggested datatype 

In [3]:
import pandas as pd
import numpy as np

def dtype_audit_report(df, category_threshold=0.05, category_max_unique=50):

    def is_integer_like_numeric(arr):
        arr = pd.Series(arr).dropna()
        if arr.empty:
            return False
        arr = pd.to_numeric(arr, errors="coerce")
        arr = arr.dropna()
        if arr.empty:
            return False
        return np.all(np.isclose(arr % 1, 0))

    def is_datetime_like_string(s):
        s = pd.Series(s).dropna().astype(str).str.strip()
        if s.empty:
            return False

        # avoid treating plain numeric columns like years / ids as datetime
        has_date_pattern = s.str.contains(r"[-/:]", regex=True).mean() >= 0.5
        if not has_date_pattern:
            return False

        dt_conv = pd.to_datetime(s, errors="coerce")
        return dt_conv.notna().mean() >= 0.9

    def suggest_dtype(series):
        s = series.dropna()

        if s.empty:
            return "keep_as_is (all values missing)"

        current_dtype = str(series.dtype)

        # =========================
        # Already typed columns
        # =========================
        if pd.api.types.is_bool_dtype(series):
            return "bool"

        if pd.api.types.is_datetime64_any_dtype(series):
            return "datetime64[ns]"

        if pd.api.types.is_integer_dtype(series):
            return current_dtype

        if pd.api.types.is_float_dtype(series):
            s_numeric = pd.to_numeric(s, errors="coerce").dropna()

            if s_numeric.empty:
                return current_dtype

            if is_integer_like_numeric(s_numeric):
                return "Int64"

            return "float64"

        # =========================
        # Object / string columns
        # =========================
        if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
            s_str = s.astype(str).str.strip()
            s_lower = s_str.str.lower()

            # Boolean-like check
            bool_map = {
                "true", "false", "yes", "no", "y", "n", "t", "f", "0", "1"
            }
            if s_lower.isin(bool_map).all():
                return "bool"

            # Numeric-like check
            numeric_conv = pd.to_numeric(s_str, errors="coerce")
            numeric_valid_ratio = numeric_conv.notna().mean()

            if numeric_valid_ratio >= 0.95:
                numeric_clean = numeric_conv.dropna()

                if numeric_clean.empty:
                    return "string"

                if is_integer_like_numeric(numeric_clean):
                    return "Int64"

                return "float64"

            # Datetime-like check
            if is_datetime_like_string(s_str):
                return "datetime64[ns]"

            # Category-like check
            nunique = s_str.nunique(dropna=True)
            non_null_count = len(s_str)
            unique_ratio = nunique / non_null_count if non_null_count > 0 else 0

            if nunique <= category_max_unique or unique_ratio <= category_threshold:
                return "category"

            return "string"

        return current_dtype

    report = pd.DataFrame({
        "column": df.columns,
        "current_dtype_report": [str(df[col].dtype) for col in df.columns],
        "suggested_dtype_report": [suggest_dtype(df[col]) for col in df.columns]
    })

    return report

In [4]:
dtype_report = dtype_audit_report(train_df)
dtype_report.head()

,column,current_dtype_report,suggested_dtype_report
0,SK_ID_CURR,int64,int64
1,TARGET,int64,int64
2,NAME_CONTRACT_TYPE,object,category
3,CODE_GENDER,object,category
4,FLAG_OWN_CAR,object,bool


### 1.2 Semantic_type_guess 

In [5]:
import re 
def semantic_type_guess_report(df, category_threshold=0.05, category_max_unique=50, text_min_avg_len=20):
    id_pattern = re.compile(r"(^id$|^id_|_id$|^sk_id|_sk_id|^key$|_key$)", flags=re.IGNORECASE)
    flag_pattern = re.compile(r"(flag|is_|has_|_bin$|_flag$)", flags=re.IGNORECASE)
    datetime_pattern = re.compile(r"(date|time|timestamp|dt|day|month|year)", flags=re.IGNORECASE)

    bool_text_values = {"yes", "no", "true", "false", "y", "n", "0", "1", "t", "f"}

    def _non_null(series):
        return series.dropna()

    def _clean_str(series):
        return series.dropna().astype(str).str.strip()

    def _unique_ratio(series):
        s = series.dropna()
        if s.empty:
            return 0.0
        return s.nunique(dropna=True) / len(s)

    def _is_integer_like_numeric(series):
        s = pd.to_numeric(series, errors="coerce").dropna()
        if s.empty:
            return False
        return np.isclose(s % 1, 0).all()

    def _is_datetime_like(series, threshold=0.8):
        s = _clean_str(series)
        if s.empty:
            return False

        # Avoid converting plain numeric ids / codes to datetime
        has_date_signal = (
            s.str.contains(r"[-/:]", regex=True).mean() >= 0.4
            or s.str.contains(r"[A-Za-z]", regex=True).mean() >= 0.4
        )

        if not has_date_signal:
            return False

        dt_test = pd.to_datetime(s, errors="coerce")
        return dt_test.notna().mean() >= threshold

    def _is_identifier_like(series, col_lower):
        s = _non_null(series)
        if s.empty:
            return False

        nunique = s.nunique(dropna=True)
        uniqueness = nunique / len(s)

        has_id_hint = bool(id_pattern.search(col_lower))

        # strict id-like: almost all unique and enough cardinality
        if has_id_hint and uniqueness >= 0.98 and nunique > min(category_max_unique, 20):
            return True

        return False

    def _is_binary_like_numeric(series):
        s = pd.to_numeric(series, errors="coerce").dropna()
        if s.empty:
            return False
        return s.nunique(dropna=True) == 2

    def _is_binary_like_text(series):
        s = _clean_str(series).str.lower()
        if s.empty:
            return False
        return s.isin(bool_text_values).mean() >= 0.95

    def _looks_like_free_text(series):
        s = _clean_str(series)
        if s.empty:
            return False

        avg_len = s.str.len().mean()
        nunique = s.nunique(dropna=True)
        ratio = nunique / len(s)

        has_space_ratio = s.str.contains(r"\s", regex=True).mean()
        long_value_ratio = (s.str.len() >= text_min_avg_len).mean()

        if avg_len >= text_min_avg_len and ratio >= 0.30:
            return True

        if long_value_ratio >= 0.40 and has_space_ratio >= 0.30:
            return True

        if nunique > category_max_unique and ratio > category_threshold and avg_len >= 12:
            return True

        return False

    def guess_semantic_type(col_name, series):
        s = _non_null(series)
        col_lower = str(col_name).lower()

        if s.empty:
            return "unknown_all_missing"

        nunique = s.nunique(dropna=True)
        unique_ratio = nunique / max(len(s), 1)

        # 1) Constant
        if nunique == 1:
            return "constant"

        # 2) Native bool
        if pd.api.types.is_bool_dtype(series):
            return "binary_flag"

        # 3) Native datetime
        if pd.api.types.is_datetime64_any_dtype(series):
            return "datetime"

        # 4) Name-based identifier hint
        if _is_identifier_like(series, col_lower):
            return "identifier"

        # 5) Name-based flag hint
        if flag_pattern.search(col_lower):
            if nunique <= 2:
                return "binary_flag"
            if nunique <= category_max_unique:
                return "categorical_flag"

        # 6) Name-based datetime hint
        if datetime_pattern.search(col_lower):
            if _is_datetime_like(series, threshold=0.75):
                return "datetime"

        # 7) Numeric columns
        if pd.api.types.is_numeric_dtype(series):
            numeric_s = pd.to_numeric(s, errors="coerce").dropna()
            if numeric_s.empty:
                return "unknown"

            # binary
            if numeric_s.nunique(dropna=True) == 2:
                return "binary_flag"

            # integer-like id
            if _is_identifier_like(numeric_s, col_lower):
                return "identifier"

            # integer dtype
            if pd.api.types.is_integer_dtype(series):
                if nunique <= category_max_unique or unique_ratio <= category_threshold:
                    return "categorical"
                return "numeric_discrete"

            # float dtype
            if pd.api.types.is_float_dtype(series):
                integer_like_ratio = np.isclose(numeric_s % 1, 0).mean()

                if nunique <= category_max_unique and unique_ratio <= 0.20:
                    return "categorical"

                if integer_like_ratio >= 0.95 and nunique <= category_max_unique:
                    return "categorical"

                if integer_like_ratio >= 0.98 and unique_ratio >= 0.98 and bool(id_pattern.search(col_lower)):
                    return "identifier"

                return "numeric_continuous"

        # 8) Object / string columns
        if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
            s_str = _clean_str(series)
            non_empty = s_str[s_str != ""]

            if non_empty.empty:
                return "unknown"

            non_empty_lower = non_empty.str.lower()
            non_empty_nunique = non_empty.nunique(dropna=True)
            non_empty_ratio = non_empty_nunique / len(non_empty)

            # bool-like text
            if _is_binary_like_text(non_empty):
                return "binary_flag"

            # datetime-like text
            if _is_datetime_like(non_empty, threshold=0.8):
                return "datetime"

            # numeric-like text
            num_test = pd.to_numeric(non_empty, errors="coerce")
            num_valid_ratio = num_test.notna().mean()

            if num_valid_ratio >= 0.95:
                num_clean = num_test.dropna()

                if num_clean.nunique(dropna=True) == 2:
                    return "binary_flag"

                if _is_identifier_like(num_clean, col_lower):
                    return "identifier"

                if _is_integer_like_numeric(num_clean):
                    if num_clean.nunique(dropna=True) <= category_max_unique or (num_clean.nunique(dropna=True) / len(num_clean)) <= category_threshold:
                        return "categorical"
                    return "numeric_as_text"

                if num_clean.nunique(dropna=True) <= category_max_unique:
                    return "categorical"

                return "numeric_as_text"

            # identifier-like text
            if _is_identifier_like(non_empty, col_lower):
                return "identifier"

            # free text vs categorical
            if _looks_like_free_text(non_empty):
                return "text"

            if non_empty_nunique <= category_max_unique or non_empty_ratio <= category_threshold:
                return "categorical"

            return "text"

        return "unknown"

    report = pd.DataFrame({
        "column": df.columns,
        "current_dtype_report": [str(df[col].dtype) for col in df.columns],
        "semantic_type_guess": [guess_semantic_type(col, df[col]) for col in df.columns]
    })

    return report

In [6]:
semantic_report = semantic_type_guess_report(train_df)
semantic_report.head(20)

,column,current_dtype_report,semantic_type_guess
0,SK_ID_CURR,int64,identifier
1,TARGET,int64,binary_flag
2,NAME_CONTRACT_TYPE,object,categorical
3,CODE_GENDER,object,categorical
4,FLAG_OWN_CAR,object,binary_flag
5,FLAG_OWN_REALTY,object,binary_flag
6,CNT_CHILDREN,int64,categorical
7,AMT_INCOME_TOTAL,float64,numeric_continuous
8,AMT_CREDIT,float64,numeric_continuous
9,AMT_ANNUITY,float64,numeric_continuous


### 1.3 id_like_flag , primaray key , foreign key

In [7]:
import pandas as pd
import numpy as np
import re

def key_like_report(df):
    # More precise ID-like column name detection
    id_regex = re.compile(
        r"(^id$|^id_|_id$|^sk_id$|^sk_id_|_sk_id$|^key$|^key_|_key$)",
        flags=re.IGNORECASE
    )

    def is_id_like(col_name):
        col = str(col_name).strip().lower()
        return bool(id_regex.search(col))

    def normalize_series(s):
        s = s.copy()

        # For object/string columns: trim and convert blank strings to NaN
        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            s = s.astype("string").str.strip()
            s = s.replace("", pd.NA)

        return s

    report_rows = []

    for col in df.columns:
        s = normalize_series(df[col])
        s_non_null = s.dropna()

        n_total = len(s)
        n_non_null = len(s_non_null)
        nunique = s_non_null.nunique(dropna=True)

        id_like_flag = is_id_like(col)

        # repeated values count
        n_repeated = n_non_null - nunique
        uniqueness_ratio = nunique / n_non_null if n_non_null > 0 else 0

        # Primary key-like:
        # - table empty না
        # - missing নেই
        # - সব non-null value unique
        is_primary_key_like = (
            n_total > 0
            and n_non_null == n_total
            and nunique == n_total
        )

        # Foreign key-like (single-table heuristic):
        # - column name id/key type
        # - not PK-like
        # - has repeated values
        # - has meaningful non-null values
        # - enough distinct values so it doesn't look like just binary / tiny categorical
        is_foreign_key_like = (
            id_like_flag
            and n_non_null > 0
            and not is_primary_key_like
            and n_repeated > 0
            and nunique >= 3
            and uniqueness_ratio < 0.98
        )

        report_rows.append({
            "column": col,
            "id_like_flag": id_like_flag,
            "is_primary_key_like": is_primary_key_like,
            "is_foreign_key_like": is_foreign_key_like
        })

    return pd.DataFrame(report_rows)

In [8]:
key_report = key_like_report(train_df)
key_report.head()

,column,id_like_flag,is_primary_key_like,is_foreign_key_like
0,SK_ID_CURR,True,True,False
1,TARGET,False,False,False
2,NAME_CONTRACT_TYPE,False,False,False
3,CODE_GENDER,False,False,False
4,FLAG_OWN_CAR,False,False,False


### 1.4 boolean_like_string_detection , numeric_like_object_detection ,date_like_object_detection



In [9]:
import pandas as pd
import numpy as np
import re

def object_pattern_detection_report(df, boolean_threshold=0.95, numeric_threshold=0.90, date_threshold=0.80):
    boolean_tokens = {
        "true", "false",
        "yes", "no",
        "y", "n",
        "t", "f",
        "1", "0"
    }

    null_like_tokens = {
        "", "na", "n/a", "null", "none", "nan", "missing", "unknown"
    }

    date_hint_pattern = re.compile(
        r"[-/]|:"
        r"|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec",
        flags=re.IGNORECASE
    )

    rows = []

    def normalize_object_series(series):
        s = series.dropna().astype(str).str.strip()
        s = s[s != ""]
        if s.empty:
            return s

        s_lower = s.str.lower()
        s = s[~s_lower.isin(null_like_tokens)]
        return s

    def prepare_numeric_strings(series):
        # remove commas and surrounding spaces for numeric detection
        return (
            series.astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )

    def has_date_signal(series):
        if series.empty:
            return False
        return series.str.contains(date_hint_pattern, regex=True).mean() >= 0.30

    for col in df.columns:
        s = df[col]
        current_dtype = str(s.dtype)

        boolean_like = False
        numeric_like = False
        date_like = False

        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            s_non_null = normalize_object_series(s)

            if len(s_non_null) > 0:
                s_lower = s_non_null.str.lower()

                # =========================
                # 1) Boolean-like detection
                # =========================
                bool_match_ratio = s_lower.isin(boolean_tokens).mean()
                boolean_like = bool_match_ratio >= boolean_threshold

                # =========================
                # 2) Numeric-like detection
                # Boolean-like হলে numeric overlap avoid করা হচ্ছে
                # =========================
                if not boolean_like:
                    s_num_ready = prepare_numeric_strings(s_non_null)
                    numeric_conv = pd.to_numeric(s_num_ready, errors="coerce")
                    numeric_match_ratio = numeric_conv.notna().mean()
                    numeric_like = numeric_match_ratio >= numeric_threshold

                # =========================
                # 3) Date-like detection
                # Pure numeric/object code-কে date ধরার risk কমানো হয়েছে
                # =========================
                if not boolean_like and not numeric_like:
                    if has_date_signal(s_lower):
                        date_conv = pd.to_datetime(s_non_null, errors="coerce")
                        date_match_ratio = date_conv.notna().mean()
                        date_like = date_match_ratio >= date_threshold

        rows.append({
            "column": col,
            "current_dtype_report": current_dtype,
            "boolean_like_string_detection": boolean_like,
            "numeric_like_object_detection": numeric_like,
            "date_like_object_detection": date_like
        })

    return pd.DataFrame(rows)

In [10]:
pattern_report = object_pattern_detection_report(train_df)
pattern_report.head()

,column,current_dtype_report,boolean_like_string_detection,numeric_like_object_detection,date_like_object_detection
0,SK_ID_CURR,int64,False,False,False
1,TARGET,int64,False,False,False
2,NAME_CONTRACT_TYPE,object,False,False,False
3,CODE_GENDER,object,False,False,False
4,FLAG_OWN_CAR,object,True,False,False


### 1.5 column content nature 

* text_vs_categorical_split
* mixed_type_flag

In [11]:
import pandas as pd
import numpy as np
import re

def column_content_nature_report(
    df,
    category_max_unique=30,
    category_ratio_threshold=0.10,
    text_min_avg_length=20,
    text_unique_ratio_threshold=0.30,
    mixed_lower_bound=0.20,
    mixed_upper_bound=0.80
):
    null_like_tokens = {
        "", "na", "n/a", "null", "none", "nan", "missing", "unknown"
    }

    bool_tokens = {
        "true", "false", "yes", "no", "y", "n", "t", "f", "0", "1"
    }

    date_hint_pattern = re.compile(
        r"[-/]|:|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec",
        flags=re.IGNORECASE
    )

    rows = []

    def clean_object_series(series):
        s = series.dropna().astype(str).str.strip()
        if s.empty:
            return s

        s_lower = s.str.lower()
        s = s[~s_lower.isin(null_like_tokens)]
        return s

    def numeric_ratio(series):
        if series.empty:
            return 0.0
        s_num = series.astype(str).str.replace(",", "", regex=False).str.strip()
        return pd.to_numeric(s_num, errors="coerce").notna().mean()

    def boolean_ratio(series):
        if series.empty:
            return 0.0
        return series.str.lower().isin(bool_tokens).mean()

    def date_ratio(series):
        if series.empty:
            return 0.0

        # Guard: if there is no visible date signal, do not try to over-parse
        has_date_signal = series.str.contains(date_hint_pattern, regex=True).mean()
        if has_date_signal < 0.30:
            return 0.0

        return pd.to_datetime(series, errors="coerce").notna().mean()

    def text_vs_categorical_decision(s_str):
        if s_str.empty:
            return np.nan, np.nan, "not_applicable"

        unique_count_local = s_str.nunique(dropna=True)
        unique_ratio_local = unique_count_local / len(s_str)
        avg_len_local = s_str.str.len().mean()
        has_space_ratio = s_str.str.contains(r"\s", regex=True).mean()

        # categorical-like: low cardinality + short labels
        if (
            (unique_count_local <= category_max_unique or unique_ratio_local <= category_ratio_threshold)
            and avg_len_local < text_min_avg_length
        ):
            return avg_len_local, unique_count_local, "categorical_like"

        # text-like: longer strings + more diversity
        if (
            avg_len_local >= text_min_avg_length
            and unique_ratio_local >= text_unique_ratio_threshold
        ) or (
            avg_len_local >= max(12, text_min_avg_length * 0.6)
            and has_space_ratio >= 0.30
            and unique_ratio_local >= 0.20
        ):
            return avg_len_local, unique_count_local, "text_like"

        return avg_len_local, unique_count_local, "borderline"

    def is_mixed(num_ratio, dt_ratio, bool_ratio, raw_python_type_count):
        numeric_mixed = mixed_lower_bound <= num_ratio <= mixed_upper_bound
        date_mixed = mixed_lower_bound <= dt_ratio <= mixed_upper_bound
        bool_mixed = mixed_lower_bound <= bool_ratio <= mixed_upper_bound

        # If more than one parsing pattern is partially true, likely mixed
        partial_hits = sum([numeric_mixed, date_mixed, bool_mixed])

        return (
            raw_python_type_count > 1
            or partial_hits >= 2
            or numeric_mixed
            or date_mixed
            or bool_mixed
        )

    for col in df.columns:
        s = df[col]
        current_dtype = str(s.dtype)

        non_null = s.dropna()
        non_null_count = len(non_null)

        # Default on original non-null values
        unique_count = non_null.nunique(dropna=True)
        unique_ratio = unique_count / non_null_count if non_null_count > 0 else np.nan

        avg_string_length = np.nan
        text_vs_categorical_split = "not_applicable"
        mixed_type_flag = False

        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            s_str = clean_object_series(s)

            if len(s_str) > 0:
                # Recompute based on cleaned meaningful string values
                cleaned_count = len(s_str)
                unique_count = s_str.nunique(dropna=True)
                unique_ratio = unique_count / cleaned_count if cleaned_count > 0 else np.nan

                avg_string_length, _, text_vs_categorical_split = text_vs_categorical_decision(s_str)

                num_ratio = numeric_ratio(s_str)
                dt_ratio = date_ratio(s_str)
                bool_ratio_val = boolean_ratio(s_str)

                raw_python_type_count = non_null.map(lambda x: type(x).__name__).nunique()

                mixed_type_flag = is_mixed(
                    num_ratio=num_ratio,
                    dt_ratio=dt_ratio,
                    bool_ratio=bool_ratio_val,
                    raw_python_type_count=raw_python_type_count
                )

        rows.append({
            "column": col,
            "current_dtype_report": current_dtype,
            "non_null_count": non_null_count,
            "unique_count": unique_count,
            "unique_ratio": unique_ratio,
            "avg_string_length": avg_string_length,
            "text_vs_categorical_split": text_vs_categorical_split,
            "mixed_type_flag": mixed_type_flag
        })

    return pd.DataFrame(rows)

In [12]:
content_nature_report = column_content_nature_report(train_df)
content_nature_report.head()

,column,current_dtype_report,non_null_count,unique_count,unique_ratio,avg_string_length,text_vs_categorical_split,mixed_type_flag
0,SK_ID_CURR,int64,307511,307511,1.000000,NaN,not_applicable,False
1,TARGET,int64,307511,2,0.000007,NaN,not_applicable,False
2,NAME_CONTRACT_TYPE,object,307511,2,0.000007,10.476064,categorical_like,False
3,CODE_GENDER,object,307511,3,0.000010,1.000026,categorical_like,True
4,FLAG_OWN_CAR,object,307511,2,0.000007,1.000000,categorical_like,False


### 1.6 String cleanliness report
* special_character_flag 
* whitespace_issue_flag 
* format_inconsistency_flag

In [13]:
import pandas as pd
import numpy as np
import re

def string_cleanliness_report(
    df,
    special_char_ratio_threshold=0.30,
    whitespace_issue_threshold=0.05,
    format_issue_threshold=0.20
):
    null_like_tokens = {
        "", "na", "n/a", "null", "none", "nan", "missing", "unknown"
    }

    # Precompiled regex patterns
    leading_trailing_pattern = re.compile(r"^\s|\s$")
    multiple_space_pattern = re.compile(r"\s{2,}")
    
    # Allow common structural separators; flag only more unusual symbols
    uncommon_special_pattern = re.compile(r"[^A-Za-z0-9\s\-_./:@]")
    
    lowercase_pattern = re.compile(r"[a-z\s]+")
    uppercase_pattern = re.compile(r"[A-Z\s]+")
    title_pattern = re.compile(r"([A-Z][a-z]*)(\s[A-Z][a-z]*)*")
    
    iso_date_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")
    slash_date_pattern = re.compile(r"\d{1,2}/\d{1,2}/\d{2,4}")
    dotted_date_pattern = re.compile(r"\d{1,2}\.\d{1,2}\.\d{2,4}")
    
    purely_numeric_pattern = re.compile(r"\d+(\.\d+)?")
    alpha_num_mix_pattern = re.compile(r"[A-Za-z].*\d|\d.*[A-Za-z]")
    date_hint_pattern = re.compile(r"[-/.]|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec", flags=re.IGNORECASE)

    rows = []

    def clean_object_series(series):
        s = series.dropna().astype(str)
        if s.empty:
            return s

        s_strip = s.str.strip()
        s = s[~s_strip.str.lower().isin(null_like_tokens)]
        s = s[s.str.strip() != ""]
        return s

    def safe_mean(boolean_series):
        return boolean_series.mean() if len(boolean_series) > 0 else np.nan

    def is_structured_string_column(s_trim):
        """
        Free text vs structured string heuristic.
        Format inconsistency is more meaningful only for structured strings.
        """
        if s_trim.empty:
            return False

        avg_len = s_trim.str.len().mean()
        has_space_ratio = s_trim.str.contains(r"\s", regex=True).mean()
        unique_ratio = s_trim.nunique(dropna=True) / len(s_trim)

        # Structured strings are usually shorter, less sentence-like
        return (
            avg_len <= 40
            and has_space_ratio <= 0.60
            and unique_ratio <= 0.98
        )

    def detect_case_mixed(s_trim):
        alpha_like = s_trim[s_trim.str.contains(r"[A-Za-z]", regex=True)]
        if alpha_like.empty:
            return False

        lower_ratio = safe_mean(alpha_like.str.fullmatch(lowercase_pattern, na=False))
        upper_ratio = safe_mean(alpha_like.str.fullmatch(uppercase_pattern, na=False))
        title_ratio = safe_mean(alpha_like.str.fullmatch(title_pattern, na=False))

        active_patterns = sum(r >= 0.05 for r in [lower_ratio, upper_ratio, title_ratio] if not pd.isna(r))
        return active_patterns >= 2

    def detect_date_pattern_mixed(s_trim):
        if s_trim.empty:
            return False

        date_hint_ratio = s_trim.str.contains(date_hint_pattern, regex=True).mean()
        if date_hint_ratio < 0.30:
            return False

        iso_ratio = safe_mean(s_trim.str.fullmatch(iso_date_pattern, na=False))
        slash_ratio = safe_mean(s_trim.str.fullmatch(slash_date_pattern, na=False))
        dotted_ratio = safe_mean(s_trim.str.fullmatch(dotted_date_pattern, na=False))

        active_patterns = sum(r >= 0.05 for r in [iso_ratio, slash_ratio, dotted_ratio] if not pd.isna(r))
        return active_patterns >= 2

    def detect_numeric_text_mixed(s_trim):
        if s_trim.empty:
            return False

        numeric_ratio = safe_mean(s_trim.str.fullmatch(purely_numeric_pattern, na=False))
        alnum_ratio = safe_mean(s_trim.str.contains(alpha_num_mix_pattern, regex=True))

        return (
            not pd.isna(numeric_ratio)
            and not pd.isna(alnum_ratio)
            and numeric_ratio >= format_issue_threshold
            and alnum_ratio >= format_issue_threshold
        )

    for col in df.columns:
        s = df[col]
        current_dtype = str(s.dtype)

        special_character_flag = False
        whitespace_issue_flag = False
        format_inconsistency_flag = False

        leading_or_trailing_space_ratio = np.nan
        multiple_space_ratio = np.nan
        special_character_ratio = np.nan

        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            s_str = clean_object_series(s)

            if len(s_str) > 0:
                # Keep original for whitespace checks
                s_non_empty = s_str.copy()
                s_trim = s_non_empty.str.strip()

                # 1) whitespace_issue_flag
                leading_trailing = s_non_empty.str.contains(leading_trailing_pattern, regex=True)
                multiple_spaces = s_non_empty.str.contains(multiple_space_pattern, regex=True)

                leading_or_trailing_space_ratio = safe_mean(leading_trailing)
                multiple_space_ratio = safe_mean(multiple_spaces)

                whitespace_issue_flag = (
                    (not pd.isna(leading_or_trailing_space_ratio) and leading_or_trailing_space_ratio >= whitespace_issue_threshold)
                    or
                    (not pd.isna(multiple_space_ratio) and multiple_space_ratio >= whitespace_issue_threshold)
                )

                # 2) special_character_flag
                uncommon_special_presence = s_trim.str.contains(uncommon_special_pattern, regex=True)
                special_character_ratio = safe_mean(uncommon_special_presence)

                special_character_flag = (
                    not pd.isna(special_character_ratio)
                    and special_character_ratio >= special_char_ratio_threshold
                )

                # 3) format_inconsistency_flag
                # Only apply strongly if column looks structured; avoid false positives on free text
                if is_structured_string_column(s_trim):
                    case_mixed = detect_case_mixed(s_trim)
                    date_pattern_mixed = detect_date_pattern_mixed(s_trim)
                    numeric_text_mixed = detect_numeric_text_mixed(s_trim)

                    format_inconsistency_flag = case_mixed or date_pattern_mixed or numeric_text_mixed
                else:
                    format_inconsistency_flag = False

        rows.append({
            "column": col,
            "current_dtype_report": current_dtype,
            "special_character_flag": special_character_flag,
            "whitespace_issue_flag": whitespace_issue_flag,
            "format_inconsistency_flag": format_inconsistency_flag,
            "leading_or_trailing_space_ratio": leading_or_trailing_space_ratio,
            "multiple_space_ratio": multiple_space_ratio,
            "special_character_ratio": special_character_ratio
        })

    return pd.DataFrame(rows)

In [14]:
string_clean_report = string_cleanliness_report(train_df)

print(string_clean_report.shape)
display(string_clean_report.head(10))

(584, 8)


,column,current_dtype_report,special_character_flag,whitespace_issue_flag,format_inconsistency_flag,leading_or_trailing_space_ratio,multiple_space_ratio,special_character_ratio
0,SK_ID_CURR,int64,False,False,False,NaN,NaN,NaN
1,TARGET,int64,False,False,False,NaN,NaN,NaN
2,NAME_CONTRACT_TYPE,object,False,False,False,0.0,0.0,0.0
3,CODE_GENDER,object,False,False,True,0.0,0.0,0.0
4,FLAG_OWN_CAR,object,False,False,True,0.0,0.0,0.0
5,FLAG_OWN_REALTY,object,False,False,True,0.0,0.0,0.0
6,CNT_CHILDREN,int64,False,False,False,NaN,NaN,NaN
7,AMT_INCOME_TOTAL,float64,False,False,False,NaN,NaN,NaN
8,AMT_CREDIT,float64,False,False,False,NaN,NaN,NaN
9,AMT_ANNUITY,float64,False,False,False,NaN,NaN,NaN


### 1.7 Schema comparison report 
* schema_mismatch_train_vs_test

In [15]:
import pandas as pd
import numpy as np

def schema_comparison_report(train_df, test_df):

    def simple_semantic_hint(series):
        """
        Very light semantic hint for mismatch support.
        """
        if series is None:
            return "missing"

        s = series.dropna()

        if len(s) == 0:
            return "all_missing"

        if pd.api.types.is_bool_dtype(series):
            return "boolean"

        if pd.api.types.is_datetime64_any_dtype(series):
            return "datetime"

        if pd.api.types.is_numeric_dtype(series):
            if s.nunique() == 2:
                return "binary_numeric"
            return "numeric"

        if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
            s_str = s.astype(str).str.strip()
            s_str = s_str[s_str != ""]

            if len(s_str) == 0:
                return "empty_text"

            num_ratio = pd.to_numeric(s_str, errors="coerce").notna().mean()
            date_ratio = pd.to_datetime(s_str, errors="coerce").notna().mean()

            if num_ratio >= 0.90:
                return "numeric_like_object"
            if date_ratio >= 0.80:
                return "date_like_object"

            avg_len = s_str.str.len().mean()
            uniq_ratio = s_str.nunique() / len(s_str)

            if s_str.nunique() <= 30 or uniq_ratio <= 0.10:
                return "categorical_text"
            if avg_len >= 20 and uniq_ratio >= 0.30:
                return "free_text"

            return "general_text"

        return "unknown"

    all_columns = sorted(set(train_df.columns).union(set(test_df.columns)))
    rows = []

    for col in all_columns:
        in_train = col in train_df.columns
        in_test = col in test_df.columns

        train_dtype = str(train_df[col].dtype) if in_train else "missing_in_train"
        test_dtype = str(test_df[col].dtype) if in_test else "missing_in_test"

        dtype_mismatch_flag = False
        schema_mismatch = False
        mismatch_reason = []

        train_series = train_df[col] if in_train else None
        test_series = test_df[col] if in_test else None

        if not in_train:
            schema_mismatch = True
            mismatch_reason.append("missing_in_train")

        if not in_test:
            schema_mismatch = True
            mismatch_reason.append("missing_in_test")

        if in_train and in_test:
            if train_dtype != test_dtype:
                dtype_mismatch_flag = True
                schema_mismatch = True
                mismatch_reason.append("dtype_mismatch")

            train_hint = simple_semantic_hint(train_series)
            test_hint = simple_semantic_hint(test_series)

            if train_hint != test_hint:
                meaningful_pairs = {
                    ("numeric", "general_text"),
                    ("general_text", "numeric"),
                    ("numeric", "free_text"),
                    ("free_text", "numeric"),
                    ("datetime", "general_text"),
                    ("general_text", "datetime"),
                    ("categorical_text", "free_text"),
                    ("free_text", "categorical_text"),
                    ("numeric", "date_like_object"),
                    ("date_like_object", "numeric"),
                    ("numeric", "numeric_like_object"),
                    ("numeric_like_object", "numeric")
                }

                if (train_hint, test_hint) in meaningful_pairs:
                    schema_mismatch = True
                    mismatch_reason.append(
                        f"semantic_shift:{train_hint}_vs_{test_hint}"
                    )

        rows.append({
            "column": col,
            "in_train": in_train,
            "in_test": in_test,
            "train_dtype": train_dtype,
            "test_dtype": test_dtype,
            "dtype_mismatch_flag": dtype_mismatch_flag,
            "schema_mismatch_train_vs_test": schema_mismatch,
            "mismatch_reason": ", ".join(mismatch_reason) if mismatch_reason else "no_issue"
        })

    return pd.DataFrame(rows)

In [16]:
schema_report = schema_comparison_report(train_df, test_df)

print(schema_report.shape)
display(schema_report.head(10))

(584, 8)


,column,in_train,in_test,train_dtype,test_dtype,dtype_mismatch_flag,schema_mismatch_train_vs_test,mismatch_reason
0,AMT_ANNUITY,True,True,float64,float64,False,False,no_issue
1,AMT_CREDIT,True,True,float64,float64,False,False,no_issue
2,AMT_GOODS_PRICE,True,True,float64,float64,False,False,no_issue
3,AMT_INCOME_TOTAL,True,True,float64,float64,False,False,no_issue
4,AMT_REQ_CREDIT_BUREAU_DAY,True,True,float64,float64,False,False,no_issue
5,AMT_REQ_CREDIT_BUREAU_HOUR,True,True,float64,float64,False,False,no_issue
6,AMT_REQ_CREDIT_BUREAU_MON,True,True,float64,float64,False,False,no_issue
7,AMT_REQ_CREDIT_BUREAU_QRT,True,True,float64,float64,False,False,no_issue
8,AMT_REQ_CREDIT_BUREAU_WEEK,True,True,float64,float64,False,False,no_issue
9,AMT_REQ_CREDIT_BUREAU_YEAR,True,True,float64,float64,False,False,no_issue


# merge

In [17]:
import pandas as pd
from functools import reduce

def add_prefix_except_key(df, prefix, key_col='column'):
    df = df.copy()

    if key_col not in df.columns:
        raise KeyError(f"'{key_col}' column not found in dataframe for prefix: {prefix}")

    df.columns = [key_col if col == key_col else f"{prefix}_{col}" for col in df.columns]
    return df


# =========================
# Prefixed report dataframes
# =========================
dtype_report_ = add_prefix_except_key(dtype_report, 'datatype')
semantic_report_ = add_prefix_except_key(semantic_report, 'semantic')
key_report_ = add_prefix_except_key(key_report, 'key')
pattern_report_ = add_prefix_except_key(pattern_report, 'pattern')
train_content_report_ = add_prefix_except_key(content_nature_report, 'train_content')
clean_report_ = add_prefix_except_key(string_clean_report, 'clean')

dfs_to_merge = [
    dtype_report_,
    semantic_report_,
    key_report_,
    pattern_report_,
    train_content_report_,
    clean_report_
]

if 'test_content_report' in globals():
    test_content_report_ = add_prefix_except_key(test_content_report, 'test_content')
    dfs_to_merge.append(test_content_report_)

dfs_to_merge = [
    df for df in dfs_to_merge
    if isinstance(df, pd.DataFrame) and not df.empty
]

# =========================
# Merge all reports into identity report
# =========================
identity_report = reduce(
    lambda left, right: pd.merge(left, right, on='column', how='outer'),
    dfs_to_merge
)

identity_report

,column,datatype_current_dtype_report,datatype_suggested_dtype_report,semantic_current_dtype_report,semantic_semantic_type_guess,key_id_like_flag,key_is_primary_key_like,key_is_foreign_key_like,pattern_current_dtype_report,pattern_boolean_like_string_detection,...,train_content_avg_string_length,train_content_text_vs_categorical_split,train_content_mixed_type_flag,clean_current_dtype_report,clean_special_character_flag,clean_whitespace_issue_flag,clean_format_inconsistency_flag,clean_leading_or_trailing_space_ratio,clean_multiple_space_ratio,clean_special_character_ratio
0,AMT_ANNUITY,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
1,AMT_CREDIT,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
2,AMT_GOODS_PRICE,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
3,AMT_INCOME_TOTAL,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
4,AMT_REQ_CREDIT_BUREAU_DAY,float64,Int64,float64,categorical,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,YEARS_BEGINEXPLUATATION_MEDI,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
580,YEARS_BEGINEXPLUATATION_MODE,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
581,YEARS_BUILD_AVG,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
582,YEARS_BUILD_MEDI,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN


In [18]:
train_content_report = column_content_nature_report(train_df)
test_content_report = column_content_nature_report(test_df)

In [19]:
print(identity_report.shape)
display(identity_report.head(10))

(584, 26)


,column,datatype_current_dtype_report,datatype_suggested_dtype_report,semantic_current_dtype_report,semantic_semantic_type_guess,key_id_like_flag,key_is_primary_key_like,key_is_foreign_key_like,pattern_current_dtype_report,pattern_boolean_like_string_detection,...,train_content_avg_string_length,train_content_text_vs_categorical_split,train_content_mixed_type_flag,clean_current_dtype_report,clean_special_character_flag,clean_whitespace_issue_flag,clean_format_inconsistency_flag,clean_leading_or_trailing_space_ratio,clean_multiple_space_ratio,clean_special_character_ratio
0,AMT_ANNUITY,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
1,AMT_CREDIT,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
2,AMT_GOODS_PRICE,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
3,AMT_INCOME_TOTAL,float64,float64,float64,numeric_continuous,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
4,AMT_REQ_CREDIT_BUREAU_DAY,float64,Int64,float64,categorical,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
5,AMT_REQ_CREDIT_BUREAU_HOUR,float64,Int64,float64,categorical,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
6,AMT_REQ_CREDIT_BUREAU_MON,float64,Int64,float64,categorical,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
7,AMT_REQ_CREDIT_BUREAU_QRT,float64,Int64,float64,categorical,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
8,AMT_REQ_CREDIT_BUREAU_WEEK,float64,Int64,float64,categorical,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN
9,AMT_REQ_CREDIT_BUREAU_YEAR,float64,Int64,float64,categorical,False,False,False,float64,False,...,NaN,not_applicable,False,float64,False,False,False,NaN,NaN,NaN


In [20]:
identity_report.to_csv(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\Identity_report.csv")

# Step 2: Missingness Audit

### 2.1 missing_summary_report
* missing_count 
* missing_percentage 
* missing_group 
* imputation_candidate_flag 

In [21]:
import pandas as pd
import numpy as np
from typing import Optional, Dict, Any

# Optional: for visualization
try:
    import missingno as msno
    MISSINGNO_AVAILABLE = True
except ImportError:
    MISSINGNO_AVAILABLE = False


def missing_summary_report(
    df: pd.DataFrame,
    low_threshold: float = 5.0,
    moderate_threshold: float = 20.0,
    high_threshold: float = 50.0,
    imputation_upper_threshold: float = 60.0,
    show_summary: bool = True,
    visualize: bool = False,
    figsize: tuple = (12, 6)
) -> pd.DataFrame:
    """
    Generate a comprehensive missing data summary report.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame to analyze
    low_threshold : float, default 5.0
        Threshold for low missing percentage
    moderate_threshold : float, default 20.0
        Threshold for moderate missing percentage
    high_threshold : float, default 50.0
        Threshold for high missing percentage
    imputation_upper_threshold : float, default 60.0
        Maximum missing percentage to consider for imputation
    show_summary : bool, default True
        Whether to print summary statistics
    visualize : bool, default False
        Whether to create visualization (requires missingno)
    figsize : tuple, default (12, 6)
        Figure size for visualization
        
    Returns
    -------
    pd.DataFrame
        Summary report with missing data statistics
        
    Examples
    --------
    >>> report = missing_summary_report(df)
    >>> report = missing_summary_report(df, low_threshold=10, visualize=True)
    """
    
    if df.empty:
        print("⚠️  DataFrame is empty")
        return pd.DataFrame()
    
    n_rows = len(df)
    n_cols = len(df.columns)
    
    # ============================================================
    # VECTORIZED CALCULATIONS (Performance Improvement)
    # ============================================================
    
    # Calculate all missing statistics at once
    missing_counts = df.isna().sum()
    missing_percentages = (missing_counts / n_rows * 100)
    
    # ============================================================
    # HELPER FUNCTIONS
    # ============================================================
    
    def assign_missing_group(missing_pct: float) -> str:
        """Categorize missing percentage into groups."""
        if missing_pct == 0:
            return "no_missing"
        elif missing_pct <= low_threshold:
            return "low_missing"
        elif missing_pct <= moderate_threshold:
            return "moderate_missing"
        elif missing_pct <= high_threshold:
            return "high_missing"
        elif missing_pct < 100:
            return "very_high_missing"
        else:
            return "all_missing"
    
    def get_imputation_strategy(missing_pct: float, dtype: str) -> Dict[str, Any]:
        """Determine appropriate imputation strategy based on missing % and dtype."""
        if missing_pct == 0:
            return {"strategy": "none", "recommendation": "No imputation needed"}
        
        if missing_pct >= 100:
            return {"strategy": "drop", "recommendation": "Drop column (100% missing)"}
        
        if missing_pct > imputation_upper_threshold:
            return {"strategy": "review", "recommendation": "Review column necessity"}
        
        # Determine strategy based on data type
        dtype_lower = str(dtype).lower()
        
        if 'int' in dtype_lower or 'float' in dtype_lower:
            if missing_pct <= low_threshold:
                return {
                    "strategy": "mean/median",
                    "recommendation": f"Mean (if normal) or Median (if skewed)"
                }
            else:
                return {
                    "strategy": "model-based",
                    "recommendation": "Consider KNN or IterativeImputer"
                }
        elif 'object' in dtype_lower or 'datetime' in dtype_lower:
            if missing_pct <= low_threshold:
                return {
                    "strategy": "mode/constant",
                    "recommendation": "Mode or 'Unknown'/'Missing' placeholder"
                }
            else:
                return {
                    "strategy": "model-based",
                    "recommendation": "Consider KNN or random forest imputation"
                }
        else:
            return {
                "strategy": "general",
                "recommendation": "Consider domain-specific approach"
            }
    
    def get_sample_non_missing(col: pd.Series, n: int = 3) -> list:
        """Get sample non-missing values from a column."""
        non_missing = col.dropna().head(n).tolist()
        # Convert to string for display, truncate long values
        return [str(v)[:30] + "..." if len(str(v)) > 30 else str(v) for v in non_missing]
    
    # ============================================================
    # BUILD REPORT DATAFRAME (Vectorized approach)
    # ============================================================
    
    report_data = []
    
    for col in df.columns:
        missing_count = missing_counts[col]
        missing_pct = missing_percentages[col]
        dtype = df[col].dtype
        unique_count = df[col].nunique(dropna=True)
        total_count = df[col].count()  # Non-missing count
        
        # Get imputation recommendation
        imputation_info = get_imputation_strategy(missing_pct, dtype)
        
        # Get sample values
        samples = get_sample_non_missing(df[col])
        
        # Build row
        report_data.append({
            "column": col,
            "dtype": str(dtype),
            "missing_count": missing_count,
            "missing_percentage": round(missing_pct, 2),
            "missing_group": assign_missing_group(missing_pct),
            "imputation_strategy": imputation_info["strategy"],
            "recommendation": imputation_info["recommendation"],
            "unique_values": unique_count,
            "non_missing_count": total_count,
            "completeness": round(100 - missing_pct, 2),
            "sample_values": samples
        })
    
    report_df = pd.DataFrame(report_data)
    
    # Sort by missing percentage (descending)
    report_df = report_df.sort_values(
        "missing_percentage", 
        ascending=False
    ).reset_index(drop=True)
    
    # ============================================================
    # SUMMARY STATISTICS
    # ============================================================
    
    if show_summary:
        _print_summary(report_df, n_rows, n_cols, imputation_upper_threshold)
    
    # ============================================================
    # VISUALIZATION
    # ============================================================
    
    if visualize:
        if MISSINGNO_AVAILABLE:
            _create_visualization(df, figsize)
        else:
            print("⚠️  Install missingno for visualization: pip install missingno")
    
    return report_df


def _print_summary(
    report_df: pd.DataFrame, 
    n_rows: int, 
    n_cols: int,
    imputation_threshold: float
) -> None:
    """Print formatted summary statistics."""
    
    total_cells = n_rows * n_cols
    total_missing = report_df["missing_count"].sum()
    overall_missing_pct = round((total_missing / total_cells * 100), 2) if total_cells > 0 else 0
    
    # Count columns by group
    group_counts = report_df["missing_group"].value_counts()
    
    print("\n" + "=" * 60)
    print("📊 MISSING DATA SUMMARY REPORT")
    print("=" * 60)
    
    print(f"\n📐 Dataset Dimensions: {n_rows:,} rows × {n_cols} columns")
    print(f"📈 Total Missing Values: {total_missing:,} ({overall_missing_pct}%)")
    
    print("\n📁 Columns by Missing Group:")
    print("-" * 40)
    
    group_order = [
        "no_missing", "low_missing", "moderate_missing", 
        "high_missing", "very_high_missing", "all_missing"
    ]
    
    for group in group_order:
        if group in group_counts.index:
            count = group_counts[group]
            pct = round(count / n_cols * 100, 1)
            bar = "█" * int(pct / 5)
            print(f"  {group:20s}: {count:3d} ({pct:5.1f}%) {bar}")
    
    # Columns needing attention
    print("\n⚠️  Columns Requiring Attention:")
    print("-" * 40)
    
    attention_needed = report_df[
        (report_df["missing_percentage"] > 0) & 
        (report_df["missing_percentage"] < 100)
    ].head(10)
    
    if len(attention_needed) > 0:
        for _, row in attention_needed.iterrows():
            print(f"  • {row['column']:25s}: {row['missing_percentage']:6.2f}% "
                  f"({row['imputation_strategy']})")
    else:
        print("  ✅ No columns require imputation!")
    
    # Columns to drop
    cols_to_drop = report_df[report_df["missing_percentage"] >= 100]["column"].tolist()
    if cols_to_drop:
        print(f"\n🗑️  Columns to Drop (100% missing): {cols_to_drop}")
    
    print("\n" + "=" * 60)


def _create_visualization(df: pd.DataFrame, figsize: tuple) -> None:
    """Create missing data visualizations using missingno."""
    
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Bar chart of missing percentages
    missing_pct = (df.isna().sum() / len(df) * 100)
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=True)
    
    if len(missing_pct) > 0:
        colors = ['#ff6b6b' if v > 50 else '#ffd93d' if v > 20 else '#6bcb77' 
                  for v in missing_pct.values]
        axes[0].barh(missing_pct.index, missing_pct.values, color=colors)
        axes[0].set_xlabel('Missing Percentage (%)')
        axes[0].set_title('Missing Values by Column')
        axes[0].axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
        axes[0].legend()
    
    # Matrix plot
    msno.matrix(df, ax=axes[1], fontsize=8)
    axes[1].set_title('Missing Data Matrix')
    
    plt.tight_layout()
    plt.show()


# ============================================================
# BONUS: Quick one-liner function
# ============================================================

def quick_missing_check(df: pd.DataFrame) -> pd.DataFrame:
    """
    Quick missing data check - simple one-liner version.
    
    Returns a DataFrame with column, dtype, missing count, and missing %.
    """
    return pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.values,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().sum() / len(df) * 100).round(2).values,
        "completeness": (100 - (df.isna().sum() / len(df) * 100)).round(2).values
    })


# ============================================================
# EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":

    
    
    print("🔍 Running Quick Missing Check...")
    print(quick_missing_check(train_df))
    
    print("\n" + "\n" + "🔍 Running Full Missing Summary Report...")
    train_missing_summary_report = missing_summary_report(
        train_df,
        show_summary=True,
        visualize=False  # Set to True if you have missingno installed
    )
    
    print("\n📋 Full Report DataFrame:")
    print(train_missing_summary_report.to_string())


🔍 Running Quick Missing Check...
                                         column    dtype  missing_count  \
0                                    SK_ID_CURR    int64              0   
1                                        TARGET    int64              0   
2                            NAME_CONTRACT_TYPE   object              0   
3                                   CODE_GENDER   object              0   
4                                  FLAG_OWN_CAR   object              0   
..                                          ...      ...            ...   
579         CC_NAME_CONTRACT_STATUS_Demand_MEAN  float64         220606   
580        CC_NAME_CONTRACT_STATUS_Refused_MEAN  float64         220606   
581  CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN  float64         220606   
582         CC_NAME_CONTRACT_STATUS_Signed_MEAN  float64         220606   
583            CC_NAME_CONTRACT_STATUS_nan_MEAN  float64         220606   

     missing_pct  completeness  
0           0.00        100.00  


In [22]:
train_missing_summary_report.head()

,column,dtype,missing_count,missing_percentage,missing_group,imputation_strategy,recommendation,unique_values,non_missing_count,completeness,sample_values
0,CC_PAYMENT_MIN_RATIO_MEAN,float64,248242,80.73,very_high_missing,review,Review column necessity,58279,59269,19.27,"[1.1777430423082598, 1.847013234549597, 5.1998..."
1,CC_PAYMENT_MIN_RATIO_MAX,float64,248242,80.73,very_high_missing,review,Review column necessity,47187,59269,19.27,"[6.165, 19.732017865475637, 20.494730370085147]"
2,CC_PAYMENT_TOTAL_RATIO_MAX,float64,247736,80.56,very_high_missing,review,Review column necessity,56912,59775,19.44,"[0.1663049533930368, 1125.7493597951343, 1.853..."
3,CC_PAYMENT_TOTAL_RATIO_MEAN,float64,247736,80.56,very_high_missing,review,Review column necessity,57349,59775,19.44,"[-2.593249949553239, 56.45259305556781, 0.1894..."
4,CC_AMT_PAYMENT_CURRENT_MAX,float64,246451,80.14,very_high_missing,review,Review column necessity,25753,61060,19.86,"[55485.0, 333000.0, 213750.0]"


### 2.2 missing_signal_report
* missing_vs_target_rate
* missing_vs_target_rate_gap_abs
* missing_indicator_recommended
* missing_pattern_guess

In [23]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.spatial.distance import jensenshannon

try:
    from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False


def _infer_target_type(y: pd.Series, max_unique_for_classification: int = 10) -> str:
    """
    Infer target type:
      - binary_classification
      - multiclass_classification
      - regression
    """
    y = y.dropna()

    if y.nunique() < 2:
        raise ValueError("Target column must contain at least 2 unique non-missing values.")

    # Binary classification
    if pd.api.types.is_bool_dtype(y) or y.nunique() == 2:
        return "binary_classification"

    # Numeric target: decide regression vs classification by uniqueness / integer-like values
    if pd.api.types.is_numeric_dtype(y):
        uniq = pd.unique(y)
        if len(uniq) <= max_unique_for_classification:
            try:
                uniq_float = np.asarray(uniq, dtype=float)
                if np.all(np.isclose(uniq_float, np.round(uniq_float))):
                    return "multiclass_classification"
            except Exception:
                pass
        return "regression"

    # Object/category/string => multiclass classification
    return "multiclass_classification"


def _choose_positive_label(y: pd.Series):
    """
    Choose a positive label for binary classification.
    Preference: sorted order if possible, otherwise appearance order.
    """
    labels = list(pd.unique(y.dropna()))
    try:
        labels = sorted(labels)
    except Exception:
        pass
    return labels[-1]


def _safe_jsd(p, q):
    """
    Jensen-Shannon distance between two probability vectors.
    """
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    if p.sum() == 0 or q.sum() == 0:
        return np.nan

    p = p / p.sum()
    q = q / q.sum()
    return float(jensenshannon(p, q, base=2.0))


def _cohens_d(a, b):
    """
    Cohen's d effect size.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if len(a) < 2 or len(b) < 2:
        return np.nan

    va = np.var(a, ddof=1)
    vb = np.var(b, ddof=1)

    pooled = np.sqrt(((len(a) - 1) * va + (len(b) - 1) * vb) / (len(a) + len(b) - 2))
    if pooled == 0 or np.isnan(pooled):
        return 0.0

    return (np.mean(a) - np.mean(b)) / pooled


def _safe_max(values):
    vals = [v for v in values if pd.notna(v)]
    return max(vals) if vals else np.nan


def missing_signal_report(
    df,
    target_col,
    target_type="auto",
    positive_label=None,
    alpha=0.05,
    effect_threshold=0.10,
    use_mutual_info=False,
    sort_by="signal_score",
):
    """
    Report whether missingness in each feature is informative with respect to the target.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    target_col : str
        Target column name.
    target_type : str, default "auto"
        One of:
          - "auto"
          - "binary_classification"
          - "multiclass_classification"
          - "regression"
    positive_label : optional
        Positive class label for binary classification.
    alpha : float, default 0.05
        Significance level.
    effect_threshold : float, default 0.10
        Minimum effect size / signal score to recommend a missing indicator.
    use_mutual_info : bool, default False
        If True and scikit-learn is installed, compute mutual information.
    sort_by : str, default "signal_score"
        Column to sort by.

    Returns
    -------
    pd.DataFrame
        Missingness signal report.
    """

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe.")

    # Keep only rows with observed target
    analysis_df = df.loc[df[target_col].notna()].copy()

    if analysis_df.empty:
        raise ValueError("No valid rows left after dropping missing target values.")

    y_raw = analysis_df[target_col]

    if target_type == "auto":
        target_kind = _infer_target_type(y_raw)
    else:
        target_kind = target_type

    if target_kind not in {
        "binary_classification",
        "multiclass_classification",
        "regression",
    }:
        raise ValueError(
            "target_type must be one of: "
            "'auto', 'binary_classification', 'multiclass_classification', 'regression'"
        )

    # Validate / prepare target
    if target_kind == "regression":
        y = pd.to_numeric(y_raw, errors="coerce")
        if y.isna().any():
            raise ValueError(
                "Regression target must be numeric. "
                "If this is a categorical target, set target_type='binary_classification' "
                "or 'multiclass_classification'."
            )
    elif target_kind == "binary_classification":
        if positive_label is None:
            positive_label = _choose_positive_label(y_raw)

        if positive_label not in set(pd.unique(y_raw)):
            raise ValueError(f"positive_label='{positive_label}' not found in target values.")

        y = (y_raw == positive_label).astype(int)
    else:
        y = y_raw.copy()

    rows = []
    feature_cols = [c for c in analysis_df.columns if c != target_col]

    for col in feature_cols:
        x = analysis_df[col]
        miss = x.isna()
        miss_count = int(miss.sum())
        non_missing_count = int((~miss).sum())
        missing_pct = (miss_count / len(analysis_df) * 100) if len(analysis_df) else np.nan

        base_row = {
            "column": col,
            "dtype": str(x.dtype),
            "missing_count": miss_count,
            "missing_percentage": round(missing_pct, 2) if pd.notna(missing_pct) else np.nan,
            "non_missing_count": non_missing_count,
            "target_type": target_kind,
            "test_used": None,
            "effect_size": np.nan,
            "p_value": np.nan,
            "signal_score": np.nan,
            "missing_indicator_recommended": False,
            "missing_pattern_guess": None,
        }

        # No comparison possible
        if miss_count == 0:
            base_row["missing_pattern_guess"] = "no_missing"
            base_row["signal_score"] = 0.0
            rows.append(base_row)
            continue

        if non_missing_count == 0:
            base_row["missing_pattern_guess"] = "all_missing"
            base_row["signal_score"] = 0.0
            rows.append(base_row)
            continue

        # ---------------------------------------------------------
        # BINARY CLASSIFICATION
        # ---------------------------------------------------------
        if target_kind == "binary_classification":
            y_bin = y.astype(int)

            table = pd.crosstab(miss, y_bin).reindex(index=[False, True], columns=[0, 1], fill_value=0)
            table_np = table.to_numpy()

            # Fisher exact test is exact and robust for 2x2
            odds_ratio, p_value = stats.fisher_exact(table_np)

            # Effect size: phi coefficient (same as sqrt(chi2 / n) for 2x2)
            chi2 = stats.chi2_contingency(table_np, correction=False)[0]
            n = table_np.sum()
            phi = np.sqrt(chi2 / n) if n > 0 else np.nan

            # Rates
            pos_rate_missing = y_bin[miss].mean()
            pos_rate_non_missing = y_bin[~miss].mean()
            rate_gap_abs = abs(pos_rate_missing - pos_rate_non_missing)

            # Distribution shift
            jsd = _safe_jsd(
                [pos_rate_missing, 1 - pos_rate_missing],
                [pos_rate_non_missing, 1 - pos_rate_non_missing],
            )

            # Optional mutual information
            mi = np.nan
            if use_mutual_info and SKLEARN_AVAILABLE:
                X = miss.astype(np.int8).to_numpy().reshape(-1, 1)
                mi = float(mutual_info_classif(X, y_bin.to_numpy(), discrete_features=True, random_state=42)[0])

            signal_score = _safe_max([abs(phi), jsd, rate_gap_abs])

            recommended = (p_value < alpha) and (signal_score >= effect_threshold)
            direction = (
                "higher_positive_rate_when_missing"
                if pos_rate_missing > pos_rate_non_missing
                else "lower_positive_rate_when_missing"
            )

            base_row.update({
                "test_used": "fisher_exact",
                "effect_size": round(float(phi), 6) if pd.notna(phi) else np.nan,
                "p_value": round(float(p_value), 6) if pd.notna(p_value) else np.nan,
                "signal_score": round(float(signal_score), 6) if pd.notna(signal_score) else np.nan,
                "missing_indicator_recommended": bool(recommended),
                "missing_pattern_guess": (
                    "likely_informative_missingness"
                    if recommended
                    else "likely_random_or_weak_signal"
                ),
                "positive_label": positive_label,
                "target_positive_rate_missing": round(float(pos_rate_missing), 6) if pd.notna(pos_rate_missing) else np.nan,
                "target_positive_rate_non_missing": round(float(pos_rate_non_missing), 6) if pd.notna(pos_rate_non_missing) else np.nan,
                "target_rate_gap_abs": round(float(rate_gap_abs), 6) if pd.notna(rate_gap_abs) else np.nan,
                "odds_ratio": round(float(odds_ratio), 6) if pd.notna(odds_ratio) else np.nan,
                "distribution_shift_jsd": round(float(jsd), 6) if pd.notna(jsd) else np.nan,
                "direction": direction,
                "mutual_info": round(float(mi), 6) if pd.notna(mi) else np.nan,
            })

        # ---------------------------------------------------------
        # MULTICLASS CLASSIFICATION
        # ---------------------------------------------------------
        elif target_kind == "multiclass_classification":
            table = pd.crosstab(miss, y).reindex(index=[False, True], fill_value=0)
            table_np = table.to_numpy()

            chi2, p_value, dof, expected = stats.chi2_contingency(table_np)
            r, c = table_np.shape
            denom = min(r - 1, c - 1)
            cramers_v = np.sqrt(chi2 / (table_np.sum() * denom)) if denom > 0 else np.nan

            missing_dist = table.loc[True].to_numpy(dtype=float)
            non_missing_dist = table.loc[False].to_numpy(dtype=float)

            missing_dist = missing_dist / missing_dist.sum()
            non_missing_dist = non_missing_dist / non_missing_dist.sum()

            jsd = _safe_jsd(missing_dist, non_missing_dist)

            abs_diff = np.abs(missing_dist - non_missing_dist)
            max_class_gap = float(np.max(abs_diff))
            largest_shift_class = table.columns[int(np.argmax(abs_diff))]

            # Optional mutual information
            mi = np.nan
            if use_mutual_info and SKLEARN_AVAILABLE:
                X = miss.astype(np.int8).to_numpy().reshape(-1, 1)
                y_codes = pd.factorize(y, sort=True)[0]
                mi = float(mutual_info_classif(X, y_codes, discrete_features=True, random_state=42)[0])

            signal_score = _safe_max([abs(cramers_v), jsd, max_class_gap])
            recommended = (p_value < alpha) and (signal_score >= effect_threshold)

            base_row.update({
                "test_used": "chi_square",
                "effect_size": round(float(cramers_v), 6) if pd.notna(cramers_v) else np.nan,
                "p_value": round(float(p_value), 6) if pd.notna(p_value) else np.nan,
                "signal_score": round(float(signal_score), 6) if pd.notna(signal_score) else np.nan,
                "missing_indicator_recommended": bool(recommended),
                "missing_pattern_guess": (
                    "likely_informative_missingness"
                    if recommended
                    else "likely_random_or_weak_signal"
                ),
                "distribution_shift_jsd": round(float(jsd), 6) if pd.notna(jsd) else np.nan,
                "max_class_gap": round(float(max_class_gap), 6),
                "largest_shift_class": largest_shift_class,
                "mutual_info": round(float(mi), 6) if pd.notna(mi) else np.nan,
            })

        # ---------------------------------------------------------
        # REGRESSION
        # ---------------------------------------------------------
        else:
            y_num = y.astype(float)

            y_missing = y_num[miss]
            y_non_missing = y_num[~miss]

            mean_missing = y_missing.mean()
            mean_non_missing = y_non_missing.mean()
            median_missing = y_missing.median()
            median_non_missing = y_non_missing.median()

            mean_gap_abs = abs(mean_missing - mean_non_missing)
            median_gap_abs = abs(median_missing - median_non_missing)

            # Primary p-value: Welch's t-test
            t_stat, p_value = stats.ttest_ind(y_missing, y_non_missing, equal_var=False, nan_policy="omit")

            # Effect size: point-biserial correlation
            r_pb, p_pb = stats.pointbiserialr(miss.astype(int), y_num)

            # Distribution shift and robust non-parametric test
            ks_stat, ks_p = stats.ks_2samp(y_missing, y_non_missing, alternative="two-sided", mode="auto")

            cohen_d = _cohens_d(y_missing, y_non_missing)

            # Optional mutual information
            mi = np.nan
            if use_mutual_info and SKLEARN_AVAILABLE:
                X = miss.astype(np.int8).to_numpy().reshape(-1, 1)
                mi = float(mutual_info_regression(X, y_num.to_numpy(), discrete_features=True, random_state=42)[0])

            signal_score = _safe_max([
                abs(r_pb),
                ks_stat,
                min(abs(cohen_d) / 3.0, 1.0) if pd.notna(cohen_d) else np.nan,
            ])

            recommended = (p_value < alpha) and (signal_score >= effect_threshold)

            direction = (
                "higher_target_when_missing"
                if mean_missing > mean_non_missing
                else "lower_target_when_missing"
            )

            base_row.update({
                "test_used": "welch_t_test",
                "effect_size": round(float(r_pb), 6) if pd.notna(r_pb) else np.nan,
                "p_value": round(float(p_value), 6) if pd.notna(p_value) else np.nan,
                "signal_score": round(float(signal_score), 6) if pd.notna(signal_score) else np.nan,
                "missing_indicator_recommended": bool(recommended),
                "missing_pattern_guess": (
                    "likely_informative_missingness"
                    if recommended
                    else "likely_random_or_weak_signal"
                ),
                "target_missing_mean": round(float(mean_missing), 6) if pd.notna(mean_missing) else np.nan,
                "target_non_missing_mean": round(float(mean_non_missing), 6) if pd.notna(mean_non_missing) else np.nan,
                "target_mean_gap_abs": round(float(mean_gap_abs), 6) if pd.notna(mean_gap_abs) else np.nan,
                "target_missing_median": round(float(median_missing), 6) if pd.notna(median_missing) else np.nan,
                "target_non_missing_median": round(float(median_non_missing), 6) if pd.notna(median_non_missing) else np.nan,
                "target_median_gap_abs": round(float(median_gap_abs), 6) if pd.notna(median_gap_abs) else np.nan,
                "point_biserial_r": round(float(r_pb), 6) if pd.notna(r_pb) else np.nan,
                "cohens_d": round(float(cohen_d), 6) if pd.notna(cohen_d) else np.nan,
                "ks_stat": round(float(ks_stat), 6) if pd.notna(ks_stat) else np.nan,
                "ks_p_value": round(float(ks_p), 6) if pd.notna(ks_p) else np.nan,
                "direction": direction,
                "mutual_info": round(float(mi), 6) if pd.notna(mi) else np.nan,
            })

        rows.append(base_row)

    report = pd.DataFrame(rows)

    # Sort by the strongest signal first
    if sort_by in report.columns:
        report = report.sort_values(
            by=[sort_by, "missing_percentage"],
            ascending=[False, False]
        ).reset_index(drop=True)

    # Useful metadata
    report.attrs["target_type"] = target_kind
    report.attrs["target_col"] = target_col
    report.attrs["rows_used"] = int(len(analysis_df))
    report.attrs["rows_dropped_target_missing"] = int(df[target_col].isna().sum())

    return report

In [24]:
train_missing_signal_report = missing_signal_report(
    train_df,
    target_col="TARGET",
    target_type="auto",       # or "regression", "binary_classification", "multiclass_classification"
    use_mutual_info=True
)

print(train_missing_signal_report.head(10))
print(train_missing_signal_report.attrs)

                     column    dtype  missing_count  missing_percentage  \
0               AMT_ANNUITY  float64             12                0.00   
1  APP_ANNUITY_INCOME_RATIO  float64             12                0.00   
2  APP_CREDIT_ANNUITY_RATIO  float64             12                0.00   
3           CNT_FAM_MEMBERS  float64              2                0.00   
4     APP_INCOME_PER_PERSON  float64              2                0.00   
5    DAYS_LAST_PHONE_CHANGE  float64              1                0.00   
6     APP_PHONE_BIRTH_RATIO  float64              1                0.00   
7  OBS_30_CNT_SOCIAL_CIRCLE  float64           1021                0.33   
8  DEF_30_CNT_SOCIAL_CIRCLE  float64           1021                0.33   
9  OBS_60_CNT_SOCIAL_CIRCLE  float64           1021                0.33   

   non_missing_count            target_type     test_used  effect_size  \
0             307499  binary_classification  fisher_exact     0.001851   
1             307499  bina

In [25]:
train_missing_signal_report.head(10)

,column,dtype,missing_count,missing_percentage,non_missing_count,target_type,test_used,effect_size,p_value,signal_score,missing_indicator_recommended,missing_pattern_guess,positive_label,target_positive_rate_missing,target_positive_rate_non_missing,target_rate_gap_abs,odds_ratio,distribution_shift_jsd,direction,mutual_info
0,AMT_ANNUITY,float64,12,0.00,307499,binary_classification,fisher_exact,0.001851,0.616206,0.203939,False,likely_random_or_weak_signal,1.0,0.00000,0.080732,0.080732,0.000000,0.203939,lower_positive_rate_when_missing,0.000003
1,APP_ANNUITY_INCOME_RATIO,float64,12,0.00,307499,binary_classification,fisher_exact,0.001851,0.616206,0.203939,False,likely_random_or_weak_signal,1.0,0.00000,0.080732,0.080732,0.000000,0.203939,lower_positive_rate_when_missing,0.000003
2,APP_CREDIT_ANNUITY_RATIO,float64,12,0.00,307499,binary_classification,fisher_exact,0.001851,0.616206,0.203939,False,likely_random_or_weak_signal,1.0,0.00000,0.080732,0.080732,0.000000,0.203939,lower_positive_rate_when_missing,0.000003
3,CNT_FAM_MEMBERS,float64,2,0.00,307509,binary_classification,fisher_exact,0.000756,1.000000,0.203936,False,likely_random_or_weak_signal,1.0,0.00000,0.080729,0.080729,0.000000,0.203936,lower_positive_rate_when_missing,0.000001
4,APP_INCOME_PER_PERSON,float64,2,0.00,307509,binary_classification,fisher_exact,0.000756,1.000000,0.203936,False,likely_random_or_weak_signal,1.0,0.00000,0.080729,0.080729,0.000000,0.203936,lower_positive_rate_when_missing,0.000001
5,DAYS_LAST_PHONE_CHANGE,float64,1,0.00,307510,binary_classification,fisher_exact,0.000534,1.000000,0.203935,False,likely_random_or_weak_signal,1.0,0.00000,0.080729,0.080729,0.000000,0.203935,lower_positive_rate_when_missing,0.000000
6,APP_PHONE_BIRTH_RATIO,float64,1,0.00,307510,binary_classification,fisher_exact,0.000534,1.000000,0.203935,False,likely_random_or_weak_signal,1.0,0.00000,0.080729,0.080729,0.000000,0.203935,lower_positive_rate_when_missing,0.000000
7,OBS_30_CNT_SOCIAL_CIRCLE,float64,1021,0.33,306490,binary_classification,fisher_exact,0.009634,0.000000,0.083900,False,likely_random_or_weak_signal,1.0,0.03526,0.080880,0.045621,0.415332,0.083900,lower_positive_rate_when_missing,0.000058
8,DEF_30_CNT_SOCIAL_CIRCLE,float64,1021,0.33,306490,binary_classification,fisher_exact,0.009634,0.000000,0.083900,False,likely_random_or_weak_signal,1.0,0.03526,0.080880,0.045621,0.415332,0.083900,lower_positive_rate_when_missing,0.000058
9,OBS_60_CNT_SOCIAL_CIRCLE,float64,1021,0.33,306490,binary_classification,fisher_exact,0.009634,0.000000,0.083900,False,likely_random_or_weak_signal,1.0,0.03526,0.080880,0.045621,0.415332,0.083900,lower_positive_rate_when_missing,0.000058


### 2.3 missing_dependency_report
* co_missing_columns_count
* null_placeholder_found

In [26]:
import pandas as pd
import numpy as np
from typing import List, Optional, Union

def missing_dependency_report(
    df: pd.DataFrame,
    co_missing_overlap_threshold: float = 0.70,
    str_placeholders: Optional[List[str]] = None,
    num_placeholders: Optional[List[Union[int, float]]] = None
) -> pd.DataFrame:
    """
    Build a comprehensive missing dependency and hidden null report.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    co_missing_overlap_threshold : float, default=0.70
        If (A ∩ B) / A >= threshold, column B is considered co-missing with column A.
    str_placeholders : list, optional
        String tokens to treat as fake null markers.
    num_placeholders : list, optional
        Numeric values often used as fake null markers (e.g., -999).
        
    Returns
    -------
    pd.DataFrame
        Report containing missingness counts, dependent columns, and detected placeholders.
    """
    if df.empty:
        return pd.DataFrame()

    # Default robust placeholders
    if str_placeholders is None:
        str_placeholders = {
            "na", "n/a", "null", "none", "unknown", 
            "missing", "?", "-", "--", "nan", "nil", " "
        }
    else:
        str_placeholders = set(s.lower().strip() for s in str_placeholders)

    if num_placeholders is None:
        num_placeholders = {-999, -9999, -99999, 9999, 99999, 999999}
    else:
        num_placeholders = set(num_placeholders)

    cols = df.columns
    
    # =========================================================
    # 1. VECTORIZED CO-MISSINGNESS (Linear Algebra approach)
    # =========================================================
    # Convert missing mask to integer matrix (Rows x Cols)
    # M.T @ M gives the intersection (co-missing count) of every column pair!
    M = df.isna().astype(int)
    intersection_matrix = M.T.dot(M).values
    
    # Number of missing values per column
    missing_counts = M.sum(axis=0).values
    
    # Avoid division by zero by replacing 0s with 1s in the denominator
    denom = np.where(missing_counts == 0, 1, missing_counts)
    
    # Calculate directional overlap ratio: (A ∩ B) / A
    # meaning: "When A is missing, how often is B also missing?"
    overlap_ratios = intersection_matrix / denom[:, None]
    
    # Remove self-correlation (diagonal)
    np.fill_diagonal(overlap_ratios, 0)

    # =========================================================
    # 2. OPTIMIZED PLACEHOLDER DETECTION & REPORT BUILDING
    # =========================================================
    rows = []
    
    for i, col in enumerate(cols):
        # Find co-missing columns for this specific feature
        co_missing_mask = overlap_ratios[i] >= co_missing_overlap_threshold
        co_missing_cols = cols[co_missing_mask].tolist()
        
        # Determine unique non-null values for fast placeholder detection
        s = df[col]
        uniques = s.dropna().unique()
        found_placeholders = []
        
        # Check string placeholders if column is object/string/category
        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
            # Check only the unique values (MASSIVE speedup)
            for val in uniques:
                val_str = str(val).strip().lower()
                if val_str in str_placeholders:
                    found_placeholders.append(val)
                    
        # Check numeric placeholders if column is numeric
        elif pd.api.types.is_numeric_dtype(s):
            for val in uniques:
                if val in num_placeholders:
                    found_placeholders.append(val)
        
        # Build the final row
        rows.append({
            "column": col,
            "missing_count": missing_counts[i],
            "co_missing_columns_count": len(co_missing_cols),
            "co_missing_columns": co_missing_cols,  # Actionable insight
            "null_placeholder_found": len(found_placeholders) > 0,
            "detected_placeholders": found_placeholders # Actionable insight
        })

    # Sort report by most issues to least
    report_df = pd.DataFrame(rows).sort_values(
        by=["missing_count", "co_missing_columns_count"], 
        ascending=[False, False]
    ).reset_index(drop=True)

    return report_df

In [27]:
train_missing_dependency_report = missing_dependency_report(train_df)
train_missing_dependency_report

,column,missing_count,co_missing_columns_count,co_missing_columns,null_placeholder_found,detected_placeholders
0,CC_PAYMENT_MIN_RATIO_MAX,248242,99,"[COMMONAREA_AVG, COMMONAREA_MODE, COMMONAREA_M...",False,[]
1,CC_PAYMENT_MIN_RATIO_MEAN,248242,99,"[COMMONAREA_AVG, COMMONAREA_MODE, COMMONAREA_M...",False,[]
2,CC_PAYMENT_TOTAL_RATIO_MAX,247736,99,"[COMMONAREA_AVG, COMMONAREA_MODE, COMMONAREA_M...",False,[]
3,CC_PAYMENT_TOTAL_RATIO_MEAN,247736,99,"[COMMONAREA_AVG, COMMONAREA_MODE, COMMONAREA_M...",False,[]
4,CC_AMT_PAYMENT_CURRENT_MAX,246451,99,"[COMMONAREA_AVG, COMMONAREA_MODE, COMMONAREA_M...",False,[]
...,...,...,...,...,...,...
579,APP_DOCUMENT_COUNT,0,0,[],False,[]
580,APP_CONTACT_COUNT,0,0,[],False,[]
581,APP_ADDRESS_MISMATCH_COUNT,0,0,[],False,[]
582,APP_MISSING_COUNT,0,0,[],False,[]


In [28]:
import pandas as pd
import numpy as np
from typing import Optional


def build_missingness_report(
    *report_dfs: pd.DataFrame,
    join_column: str = "column",
    how: str = "outer",
    validate_overlap: bool = True,
    add_overall_score: bool = True,
    sort_by: Optional[str] = None,
    prefix_duplicates: bool = True,
) -> pd.DataFrame:
    """
    Merge multiple missingness report DataFrames into a single unified report.

    Handles:
      - Arbitrary number of input reports (not just 3)
      - Duplicate column-name conflicts across reports
      - Optional overall missingness severity score
      - Input validation and informative errors

    Parameters
    ----------
    *report_dfs : pd.DataFrame
        Variable number of report DataFrames to merge.
        Each must contain `join_column`.
    join_column : str, default "column"
        Column name to join on.
    how : str, default "outer"
        Merge strategy ("outer", "inner", "left", "right").
    validate_overlap : bool, default True
        If True, warn when reports have non-overlapping column sets.
    add_overall_score : bool, default True
        If True, compute a composite missingness severity score.
    sort_by : str or None, default None
        Column name to sort the final report by (descending).
        If None and add_overall_score is True, sorts by the score.
    prefix_duplicates : bool, default True
        If True, automatically resolve duplicate column names
        with suffixes based on the report index.

    Returns
    -------
    pd.DataFrame
        Unified missingness report.

    Examples
    --------
    >>> report = build_missingness_report(
    ...     summary_report,
    ...     signal_report,
    ...     dependency_report,
    ...     sort_by="overall_missingness_score"
    ... )
    """

    # ── input validation ──────────────────────────────────────────
    if len(report_dfs) == 0:
        raise ValueError("At least one report DataFrame is required.")

    valid_reports: list[pd.DataFrame] = []
    for i, rdf in enumerate(report_dfs):
        if rdf is None:
            continue
        if not isinstance(rdf, pd.DataFrame):
            raise TypeError(f"Report at index {i} is not a DataFrame (got {type(rdf).__name__}).")
        if rdf.empty:
            continue
        if join_column not in rdf.columns:
            raise KeyError(
                f"Report at index {i} is missing the join column '{join_column}'. "
                f"Available columns: {list(rdf.columns)}"
            )
        valid_reports.append(rdf)

    if len(valid_reports) == 0:
        return pd.DataFrame(columns=[join_column])

    if len(valid_reports) == 1:
        return valid_reports[0].copy()

    # ── optional: check overlap ───────────────────────────────────
    if validate_overlap and len(valid_reports) > 1:
        col_sets = [set(r[join_column].dropna().unique()) for r in valid_reports]
        common = col_sets[0]
        for cs in col_sets[1:]:
            common = common & cs
        all_cols = set().union(*col_sets)
        if len(common) < len(all_cols):
            only_in_some = all_cols - common
            import warnings
            warnings.warn(
                f"{len(only_in_some)} column(s) appear in some reports but not all: "
                f"{sorted(only_in_some)[:10]}{'...' if len(only_in_some) > 10 else ''}. "
                f"Using how='{how}' merge; missing values will be NaN.",
                UserWarning,
                stacklevel=2,
            )

    # ── resolve duplicate field names across reports ──────────────
    if prefix_duplicates:
        report_labels = [f"_r{i}" for i in range(len(valid_reports))]
        seen_cols: dict[str, int] = {}
        renamed_reports: list[pd.DataFrame] = []

        for idx, rdf in enumerate(valid_reports):
            rename_map = {}
            for c in rdf.columns:
                if c == join_column:
                    continue
                if c in seen_cols:
                    # Conflict: rename both the first occurrence (if not yet) and this one
                    rename_map[c] = f"{c}{report_labels[idx]}"
                seen_cols[c] = idx
            renamed_reports.append(rdf.rename(columns=rename_map))
        valid_reports = renamed_reports

    # ── sequential merge ──────────────────────────────────────────
    merged = valid_reports[0]
    for rdf in valid_reports[1:]:
        merged = pd.merge(merged, rdf, on=join_column, how=how)

    # ── optional composite score ──────────────────────────────────
    if add_overall_score:
        merged = _add_overall_score(merged, join_column)

    # ── sort ──────────────────────────────────────────────────────
    if sort_by is None and add_overall_score and "overall_missingness_score" in merged.columns:
        sort_by = "overall_missingness_score"

    if sort_by and sort_by in merged.columns:
        merged = merged.sort_values(sort_by, ascending=False, na_position="last").reset_index(drop=True)

    return merged


def _add_overall_score(df: pd.DataFrame, join_column: str) -> pd.DataFrame:
    """
    Compute a composite 0–100 missingness severity score.

    Heuristic components (each normalized to 0–1, then averaged and scaled to 0–100):
      1. missing_percentage           → direct 0–100 scaled to 0–1
      2. gap_abs (signal report)      → clipped to [0, 1]
      3. co_missing_columns_count     → ratio to max observed
      4. missing_indicator_recommended→ 1 if True else 0
      5. null_placeholder_found       → 1 if True else 0

    Only components that exist in the DataFrame are used.
    """
    scores: list[pd.Series] = []

    # 1) Missing percentage (0–100 → 0–1)
    for candidate in ("missing_percentage", "missing_pct"):
        if candidate in df.columns:
            scores.append(df[candidate].fillna(0).clip(0, 100) / 100.0)
            break

    # 2) Target-rate gap (signal strength)
    if "gap_abs" in df.columns:
        scores.append(df["gap_abs"].fillna(0).clip(0, 1))
    elif "missing_vs_target_rate_gap_abs" in df.columns:
        scores.append(df["missing_vs_target_rate_gap_abs"].fillna(0).clip(0, 1))

    # 3) Co-missing breadth
    if "co_missing_columns_count" in df.columns:
        max_co = df["co_missing_columns_count"].max()
        if max_co and max_co > 0:
            scores.append(df["co_missing_columns_count"].fillna(0) / max_co)

    # 4) Indicator recommended (boolean → 0/1)
    if "missing_indicator_recommended" in df.columns:
        scores.append(df["missing_indicator_recommended"].astype(float).fillna(0))

    # 5) Placeholder found (boolean → 0/1)
    if "null_placeholder_found" in df.columns:
        scores.append(df["null_placeholder_found"].astype(float).fillna(0))

    if scores:
        combined = pd.concat(scores, axis=1)
        df = df.copy()
        df["overall_missingness_score"] = (combined.mean(axis=1) * 100).round(2)
    return df


# ──────────────────────────────────────────────────────────────────
# Usage
# ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Example with your three existing reports
    # (assuming they are already computed as DataFrames)

    # from your_module import missing_summary_report, missing_signal_report, missing_dependency_report

    # train_missing_summary_report   = missing_summary_report(train_df)
    # train_missing_signal_report    = missing_signal_report(train_df, target_col="target")
    # train_missing_dependency_report = missing_dependency_report(train_df)

    missingness_report = build_missingness_report(
        train_missing_summary_report,
        train_missing_signal_report,
        train_missing_dependency_report,
        sort_by="overall_missingness_score",
    )

missingness_report.head(10)

,column,dtype,missing_count,missing_percentage,missing_group,imputation_strategy,recommendation,unique_values,non_missing_count,completeness,...,odds_ratio,distribution_shift_jsd,direction,mutual_info,missing_count_r2,co_missing_columns_count,co_missing_columns,null_placeholder_found,detected_placeholders,overall_missingness_score
0,DAYS_LAST_PHONE_CHANGE,float64,1,0.00,low_missing,mean/median,Mean (if normal) or Median (if skewed),3773,307510,100.00,...,0.000000,0.203935,lower_positive_rate_when_missing,0.000000,1,478,"[OCCUPATION_TYPE, EXT_SOURCE_1, EXT_SOURCE_2, ...",True,[-999.0],50.00
1,BUREAU_AMT_ANNUITY_MEAN,float64,227502,73.98,very_high_missing,review,Review column necessity,40563,80009,26.02,...,0.968398,0.003727,lower_positive_rate_when_missing,0.000007,227502,92,"[COMMONAREA_AVG, COMMONAREA_MODE, COMMONAREA_M...",True,[9999.0],48.31
2,BUREAU_AMT_ANNUITY_MAX,float64,227502,73.98,very_high_missing,review,Review column necessity,23582,80009,26.02,...,0.968398,0.003727,lower_positive_rate_when_missing,0.000007,227502,92,"[COMMONAREA_AVG, COMMONAREA_MODE, COMMONAREA_M...",True,[9999.0],48.31
3,REFUSED_DAYS_DECISION_MEAN,float64,207217,67.39,very_high_missing,review,Review column necessity,17440,100294,32.61,...,0.652403,0.050552,lower_positive_rate_when_missing,0.001590,207217,96,"[BUREAU_AMT_ANNUITY_MAX, BUREAU_AMT_ANNUITY_ME...",True,[-999.0],46.87
4,REFUSED_AMT_APPLICATION_MEAN,float64,207217,67.39,very_high_missing,review,Review column necessity,32297,100294,32.61,...,0.652403,0.050552,lower_positive_rate_when_missing,0.001590,207217,96,"[BUREAU_AMT_ANNUITY_MAX, BUREAU_AMT_ANNUITY_ME...",True,[99999.0],46.87
5,REFUSED_AMT_APPLICATION_MAX,float64,207217,67.39,very_high_missing,review,Review column necessity,17300,100294,32.61,...,0.652403,0.050552,lower_positive_rate_when_missing,0.001590,207217,96,"[BUREAU_AMT_ANNUITY_MAX, BUREAU_AMT_ANNUITY_ME...",True,[99999.0],46.87
6,REFUSED_DAYS_DECISION_MIN,float64,207217,67.39,very_high_missing,review,Review column necessity,2921,100294,32.61,...,0.652403,0.050552,lower_positive_rate_when_missing,0.001590,207217,96,"[BUREAU_AMT_ANNUITY_MAX, BUREAU_AMT_ANNUITY_ME...",True,[-999.0],46.87
7,APPROVED_AMT_APPLICATION_MAX,float64,17446,5.67,moderate_missing,model-based,Consider KNN or IterativeImputer,50746,290065,94.33,...,0.729451,0.034347,lower_positive_rate_when_missing,0.000168,17446,366,"[BUREAU_AMT_ANNUITY_MAX, BUREAU_AMT_ANNUITY_ME...",True,[99999.0],45.56
8,APPROVED_AMT_CREDIT_MEAN,float64,17446,5.67,moderate_missing,model-based,Consider KNN or IterativeImputer,185460,290065,94.33,...,0.729451,0.034347,lower_positive_rate_when_missing,0.000168,17446,366,"[BUREAU_AMT_ANNUITY_MAX, BUREAU_AMT_ANNUITY_ME...",True,[99999.0],45.56
9,APPROVED_DAYS_DECISION_MIN,float64,17446,5.67,moderate_missing,model-based,Consider KNN or IterativeImputer,2921,290065,94.33,...,0.729451,0.034347,lower_positive_rate_when_missing,0.000168,17446,366,"[BUREAU_AMT_ANNUITY_MAX, BUREAU_AMT_ANNUITY_ME...",True,[-999.0],45.56


In [29]:
missingness_report.to_csv(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\missingness_report.csv")

# Step 3: Uniqueness Summary 

### DataFrame 1: uniqueness_summary_report
* unique_count
* unique_percentage
* duplicate_count
* duplicate_value_ratio
* constant_flag
* quasi_constant_flag

In [30]:
import pandas as pd
import numpy as np
from typing import Optional


def uniqueness_summary_report(
    df: pd.DataFrame,
    quasi_constant_threshold: float = 95.0,
    include_top_value: bool = True,
    top_n_examples: int = 3,
    normalize_strings: bool = False,
) -> pd.DataFrame:
    """
    Build a uniqueness / duplication / (quasi-)constant summary report per column.

    Improvements over the original
    ------------------------------
    1) Vectorized counts for performance (no heavy per-column operations except top value calc).
    2) Optional string normalization to catch "hidden" quasi-constants like "NA", " na ", "Na".
    3) Adds actionable context: most frequent value and its share, plus a few top values.
    4) Robust handling for empty df, all-null cols, and mixed dtypes.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    quasi_constant_threshold : float, default 95.0
        If the most frequent (non-null) value share >= this %, quasi_constant_flag = True.
    include_top_value : bool, default True
        If True, include top_value, top_value_count, top_value_percentage, and top_values_preview.
    top_n_examples : int, default 3
        Number of top frequent values to include in preview.
    normalize_strings : bool, default False
        If True, for string/object columns compute top frequencies on a normalized view:
        strip + lower (does not mutate the original df). Helps detect hidden quasi-constants.

    Returns
    -------
    pd.DataFrame
        Columns:
          - column
          - dtype
          - non_null_count
          - unique_count
          - unique_percentage
          - duplicate_count
          - duplicate_value_ratio
          - constant_flag
          - quasi_constant_flag
          - (optional) top_value, top_value_count, top_value_percentage, top_values_preview
    """
    if df is None or df.shape[1] == 0:
        return pd.DataFrame(
            columns=[
                "column", "dtype", "non_null_count", "unique_count", "unique_percentage",
                "duplicate_count", "duplicate_value_ratio", "constant_flag", "quasi_constant_flag"
            ]
        )

    n_rows = len(df)

    # Vectorized basic counts
    non_null_count = df.notna().sum(axis=0)
    unique_count = df.nunique(dropna=True)

    # Avoid division by zero
    denom = non_null_count.replace(0, np.nan)

    unique_percentage = (unique_count / denom) * 100.0
    duplicate_count = non_null_count - unique_count
    duplicate_value_ratio = (duplicate_count / denom) * 100.0

    constant_flag = unique_count.eq(1) & non_null_count.gt(0)

    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null_count": non_null_count.values,
        "unique_count": unique_count.values,
        "unique_percentage": unique_percentage.round(2).values,
        "duplicate_count": duplicate_count.values,
        "duplicate_value_ratio": duplicate_value_ratio.round(2).values,
        "constant_flag": constant_flag.values,
    })

    # Compute quasi-constant using top-value share
    # We do this per-column because the "top value" is inherently column-wise.
    top_value_pct = []
    top_value = []
    top_value_count = []
    top_values_preview = []

    for col in df.columns:
        s = df[col].dropna()
        if s.empty:
            top_value_pct.append(np.nan)
            top_value.append(np.nan)
            top_value_count.append(0)
            top_values_preview.append([])
            continue

        # Optional normalization for string-like columns
        s_for_freq = s
        if normalize_strings and (
            pd.api.types.is_object_dtype(df[col])
            or pd.api.types.is_string_dtype(df[col])
            or pd.api.types.is_categorical_dtype(df[col])
        ):
            # Use pandas "string" operations safely
            s_for_freq = s.astype("string").str.strip().str.lower()

        vc = s_for_freq.value_counts(dropna=True)

        tv = vc.index[0]
        tv_count = int(vc.iloc[0])
        tv_pct = float((tv_count / len(s_for_freq)) * 100.0)

        top_value_pct.append(tv_pct)
        top_value.append(tv)
        top_value_count.append(tv_count)

        if include_top_value:
            preview = []
            for k in range(min(top_n_examples, len(vc))):
                preview.append((vc.index[k], int(vc.iloc[k])))
            top_values_preview.append(preview)
        else:
            top_values_preview.append([])

    report["top_value_percentage"] = pd.Series(top_value_pct).round(2).values
    report["quasi_constant_flag"] = (
        report["top_value_percentage"].ge(quasi_constant_threshold) & report["non_null_count"].gt(0)
    ).values

    if include_top_value:
        report["top_value"] = top_value
        report["top_value_count"] = top_value_count
        report["top_values_preview"] = top_values_preview

    # Helpful sort: most problematic first (constants/quasi-constants, then low uniqueness)
    report = report.sort_values(
        by=["constant_flag", "quasi_constant_flag", "unique_percentage", "non_null_count"],
        ascending=[False, False, True, False],
        na_position="last",
    ).reset_index(drop=True)

    return report

In [31]:
train_uniqueness_summary_report = uniqueness_summary_report(train_df)
train_uniqueness_summary_report.head()

,column,dtype,non_null_count,unique_count,unique_percentage,duplicate_count,duplicate_value_ratio,constant_flag,top_value_percentage,quasi_constant_flag,top_value,top_value_count,top_values_preview
0,PREV_NAME_CONTRACT_TYPE_nan_MEAN,float64,291057,1,0.0,291056,100.0,True,100.0,True,0.0,291057,"[(0.0, 291057)]"
1,PREV_WEEKDAY_APPR_PROCESS_START_nan_MEAN,float64,291057,1,0.0,291056,100.0,True,100.0,True,0.0,291057,"[(0.0, 291057)]"
2,PREV_FLAG_LAST_APPL_PER_CONTRACT_nan_MEAN,float64,291057,1,0.0,291056,100.0,True,100.0,True,0.0,291057,"[(0.0, 291057)]"
3,PREV_NAME_CASH_LOAN_PURPOSE_nan_MEAN,float64,291057,1,0.0,291056,100.0,True,100.0,True,0.0,291057,"[(0.0, 291057)]"
4,PREV_NAME_CONTRACT_STATUS_nan_MEAN,float64,291057,1,0.0,291056,100.0,True,100.0,True,0.0,291057,"[(0.0, 291057)]"


### 3.2 value_dominance_report
* dominant_value
* dominant_value_percentage
* second_dominant_value 
* suspicious_uniformity_flag

In [32]:
import pandas as pd
import numpy as np
from typing import Optional, Any


def value_dominance_report(
    df: pd.DataFrame,
    suspicious_uniformity_threshold: float = 98.0,
    min_non_null_count: int = 10,
    top_n_values: int = 3,
    truncate_value_length: int = 60,
) -> pd.DataFrame:
    """
    Build a value-dominance report for each column, highlighting overly dominant distributions
    and potential “uniform” or quasi-constant issues.

    Signal guidance: >= threshold dominance is flagged as suspicious. For very small samples,
    dominance stats are included but the uniformity flag is conservative unless the column is
    truly constant.

    Output columns:
    - column
    - non_null_count
    - dominant_value
    - dominant_value_pct
    - second_dominant_value
    - second_dominant_pct
    - top_values (list of tuples: value, pct rounded)
    - suspicious_uniformity_flag
    - dominance_note (short diagnostic)
    """
    if df.empty:
        return pd.DataFrame(
            columns=[
                "column", "non_null_count",
                "dominant_value", "dominant_value_pct",
                "second_dominant_value", "second_dominant_pct",
                "top_values", "suspicious_uniformity_flag", "dominance_note"
            ]
        )

    rows = []
    n_rows = len(df)

    # Vectorized non-null counts to avoid repeated dropna calls
    non_null_counts = df.notna().sum()

    for col in df.columns:
        nn_count = int(non_null_counts[col])

        if nn_count == 0:
            rows.append({
                "column": col,
                "non_null_count": 0,
                "dominant_value": np.nan,
                "dominant_value_pct": np.nan,
                "second_dominant_value": np.nan,
                "second_dominant_pct": np.nan,
                "top_values": [],
                "suspicious_uniformity_flag": False,
                "dominance_note": "All values NULL",
            })
            continue

        non_nulls = df[col].dropna()

        if nn_count < min_non_null_count:
            # Warn that dominance/“uniform” judgment may be unstable for thin samples
            vc = non_nulls.value_counts(dropna=True)
            top_vals = vc.iloc[:top_n_values] if len(vc) > 0 else pd.Series()
            top_list = [(_truncate_value(v, truncate_value_length), round(100 * p / nn_count, 2))
                        for v, p in top_vals.items()]
            dom_note = f"Low non-null count: {nn_count} (threshold behavior limited)"
            suspicious = len(vc) == 1  # only flag as truly suspicious if constant-like
        else:
            vc = non_nulls.value_counts(dropna=True)
            if len(vc) == 0:
                dom_note = "No non-null values after dropna (rare edge case)"
                top_list, suspicious = [], False
                dom_pct, sec_pct = np.nan, np.nan
                dom_val, sec_val = np.nan, np.nan
            else:
                top_vals = vc.iloc[:top_n_values]
                top_list = [(_truncate_value(v, truncate_value_length), round(100 * p / nn_count, 2))
                            for v, p in top_vals.items()]

                dom_val = top_vals.index[0]
                dom_pct = (top_vals.iloc[0] / nn_count) * 100
                sec_val = top_vals.index[1] if len(top_vals) > 1 else np.nan
                sec_pct = (top_vals.iloc[1] / nn_count) * 100 if len(top_vals) > 1 else np.nan

                suspicious = dom_pct >= suspicious_uniformity_threshold
                dom_note = (
                    "Appears uniformly dominated" if suspicious
                    else f"Most frequent dominates with {round(dom_pct,2)}%"
                )
        rows.append({
            "column": col,
            "non_null_count": nn_count,
            "dominant_value": _truncate_value(dom_val, truncate_value_length) if pd.notna(dom_val) else np.nan,
            "dominant_value_pct": round(dom_pct, 2) if pd.notna(dom_pct) else np.nan,
            "second_dominant_value": _truncate_value(sec_val, truncate_value_length) if pd.notna(sec_val) else np.nan,
            "second_dominant_pct": round(sec_pct, 2) if pd.notna(sec_pct) else np.nan,
            "top_values": top_list,
            "suspicious_uniformity_flag": suspicious,
            "dominance_note": dom_note,
        })

    out = pd.DataFrame(rows).sort_values(
        ["suspicious_uniformity_flag", "dominant_value_pct"],
        ascending=[False, False]
    ).reset_index(drop=True)

    if (out["non_null_count"] > n_rows).any():
        raise RuntimeError("non_null_count exceeds row count; filtering logic error detected.")

    return out


def _truncate_value(val: Any, max_len: int) -> str:
    """Safe string representation with trimming (handles floats/timestamps/strings cleanly)."""
    s = str(val)
    if len(s) > max_len:
        return s[:max_len] + "…"
    return s

In [33]:
train_value_dominance_report = value_dominance_report(train_df)
train_value_dominance_report

,column,non_null_count,dominant_value,dominant_value_pct,second_dominant_value,second_dominant_pct,top_values,suspicious_uniformity_flag,dominance_note
0,FLAG_MOBIL,307511,1,100.00,0,0.00,"[(1, 100.0), (0, 0.0)]",True,Appears uniformly dominated
1,FLAG_DOCUMENT_2,307511,0,100.00,1,0.00,"[(0, 100.0), (1, 0.0)]",True,Appears uniformly dominated
2,FLAG_DOCUMENT_10,307511,0,100.00,1,0.00,"[(0, 100.0), (1, 0.0)]",True,Appears uniformly dominated
3,FLAG_DOCUMENT_12,307511,0,100.00,1,0.00,"[(0, 100.0), (1, 0.0)]",True,Appears uniformly dominated
4,BUREAU_BB_STATUS_nan_MEAN_MEAN,92231,0.0,100.00,NaN,NaN,"[(0.0, 100.0)]",True,Appears uniformly dominated
...,...,...,...,...,...,...,...,...,...
579,INS_AMT_PAYMENT_SUM,291643,204935.58,0.01,56555.01,0.01,"[(204935.58, 0.01), (56555.01, 0.01), (127421....",False,Most frequent dominates with 0.01%
580,SK_ID_CURR,307511,100002,0.00,100003,0.00,"[(100002, 0.0), (100003, 0.0), (100004, 0.0)]",False,Most frequent dominates with 0.0%
581,EXT_SOURCE_1,134133,0.6227066347478732,0.00,0.528197430013715,0.00,"[(0.6227066347478732, 0.0), (0.528197430013715...",False,Most frequent dominates with 0.0%
582,APP_EMPLOYED_BIRTH_RATIO,252137,0.0833333333333333,0.00,0.0909090909090909,0.00,"[(0.0833333333333333, 0.0), (0.090909090909090...",False,Most frequent dominates with 0.0%


In [34]:
import pandas as pd

uniqueness_report = pd.merge(
    train_uniqueness_summary_report,
    train_value_dominance_report,
    on="column",
    how="outer"
)

uniqueness_report

,column,dtype,non_null_count_x,unique_count,unique_percentage,duplicate_count,duplicate_value_ratio,constant_flag,top_value_percentage,quasi_constant_flag,...,top_value_count,top_values_preview,non_null_count_y,dominant_value,dominant_value_pct,second_dominant_value,second_dominant_pct,top_values,suspicious_uniformity_flag,dominance_note
0,AMT_ANNUITY,float64,307499,13672,4.45,293827,95.55,False,2.08,False,...,6385,"[(9000.0, 6385), (13500.0, 5514), (6750.0, 2279)]",307499,9000.0,2.08,13500.0,1.79,"[(9000.0, 2.08), (13500.0, 1.79), (6750.0, 0.74)]",False,Most frequent dominates with 2.08%
1,AMT_CREDIT,float64,307511,5603,1.82,301908,98.18,False,3.16,False,...,9709,"[(450000.0, 9709), (675000.0, 8877), (225000.0...",307511,450000.0,3.16,675000.0,2.89,"[(450000.0, 3.16), (675000.0, 2.89), (225000.0...",False,Most frequent dominates with 3.16%
2,AMT_GOODS_PRICE,float64,307233,1002,0.33,306231,99.67,False,8.47,False,...,26022,"[(450000.0, 26022), (225000.0, 25282), (675000...",307233,450000.0,8.47,225000.0,8.23,"[(450000.0, 8.47), (225000.0, 8.23), (675000.0...",False,Most frequent dominates with 8.47%
3,AMT_INCOME_TOTAL,float64,307511,2548,0.83,304963,99.17,False,11.63,False,...,35750,"[(135000.0, 35750), (112500.0, 31019), (157500...",307511,135000.0,11.63,112500.0,10.09,"[(135000.0, 11.63), (112500.0, 10.09), (157500...",False,Most frequent dominates with 11.63%
4,AMT_REQ_CREDIT_BUREAU_DAY,float64,265992,9,0.00,265983,100.00,False,99.44,True,...,264503,"[(0.0, 264503), (1.0, 1292), (2.0, 106)]",265992,0.0,99.44,1.0,0.49,"[(0.0, 99.44), (1.0, 0.49), (2.0, 0.04)]",True,Appears uniformly dominated
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,YEARS_BEGINEXPLUATATION_MEDI,float64,157504,245,0.16,157259,99.84,False,2.74,False,...,4314,"[(0.9871, 4314), (0.9861, 4247), (0.9856, 4199)]",157504,0.9871,2.74,0.9861,2.70,"[(0.9871, 2.74), (0.9861, 2.7), (0.9856, 2.67)]",False,Most frequent dominates with 2.74%
580,YEARS_BEGINEXPLUATATION_MODE,float64,157504,221,0.14,157283,99.86,False,2.72,False,...,4291,"[(0.9871, 4291), (0.9866, 4173), (0.9861, 4167)]",157504,0.9871,2.72,0.9866,2.65,"[(0.9871, 2.72), (0.9866, 2.65), (0.9861, 2.65)]",False,Most frequent dominates with 2.72%
581,YEARS_BUILD_AVG,float64,103023,149,0.14,102874,99.86,False,2.91,False,...,2999,"[(0.8232, 2999), (0.8164, 2864), (0.8028, 2848)]",103023,0.8232,2.91,0.8164,2.78,"[(0.8232, 2.91), (0.8164, 2.78), (0.8028, 2.76)]",False,Most frequent dominates with 2.91%
582,YEARS_BUILD_MEDI,float64,103023,151,0.15,102872,99.85,False,2.91,False,...,2994,"[(0.8256, 2994), (0.8189, 2883), (0.8054, 2842)]",103023,0.8256,2.91,0.8189,2.80,"[(0.8256, 2.91), (0.8189, 2.8), (0.8054, 2.76)]",False,Most frequent dominates with 2.91%


In [35]:
uniqueness_report.to_csv(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\3.uniqueness_report.csv")

# Step 4. Categorical Feature Profile 10

### 4.1 category_cardinality_report
* cardinality_count
* cardinality_group
* high_cardinality_flag

In [36]:
import pandas as pd
import numpy as np
from typing import Optional, List, Dict, Any


def category_cardinality_report(
    df: pd.DataFrame,
    high_cardinality_threshold: int = 50,
    very_high_cardinality_threshold: int = 200,
    id_cardinality_ratio: float = 0.95,
    include_samples: bool = True,
    sample_size: int = 5,
    detect_potential_ids: bool = True,
) -> pd.DataFrame:
    """
    Build a comprehensive categorical cardinality report for each column.

    Improvements over original
    -------------------------
    1. Vectorized nunique() call for speed.
    2. Detects potential ID columns (cardinality ≈ row count).
    3. Recommends encoding strategies based on cardinality.
    4. Includes sample values for verification.
    5. Handles categorical dtype explicitly.
    6. Adds dtype and missing context.
    7. More refined cardinality groupings.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    high_cardinality_threshold : int, default 50
        Threshold above which cardinality is considered "high".
    very_high_cardinality_threshold : int, default 200
        Threshold above which cardinality is considered "very high".
    id_cardinality_ratio : float, default 0.95
        If cardinality / n_rows >= this ratio, flag as potential ID.
    include_samples : bool, default True
        Include sample values from the column.
    sample_size : int, default 5
        Number of sample values to include.
    detect_potential_ids : bool, default True
        Flag columns that might be unique identifiers.

    Returns
    -------
    pd.DataFrame
        Report with cardinality statistics, flags, and recommendations.
    """
    if df.empty:
        return pd.DataFrame(columns=_get_output_columns())

    n_rows = len(df)

    # Vectorized: non-null counts and unique counts
    non_null_counts = df.notna().sum()
    unique_counts = df.nunique(dropna=True)

    # Determine dtypes efficiently
    dtypes = df.dtypes.astype(str)

    # Build report rows
    rows = []

    for col in df.columns:
        nn_count = int(non_null_counts[col])
        n_unique = int(unique_counts[col])
        dtype = dtypes[col]

        # Determine cardinality group
        cardinality_group = _assign_cardinality_group(
            n_unique, high_cardinality_threshold, very_high_cardinality_threshold
        )

        # Flags
        high_cardinality_flag = n_unique > high_cardinality_threshold
        very_high_cardinality_flag = n_unique > very_high_cardinality_threshold

        # Potential ID detection
        is_potential_id = False
        if detect_potential_ids and nn_count > 0:
            uniqueness_ratio = n_unique / nn_count
            is_potential_id = (
                uniqueness_ratio >= id_cardinality_ratio
                and n_unique > high_cardinality_threshold
                and _looks_like_id(col, dtype)
            )

        # Encoding recommendation
        encoding_rec = _recommend_encoding(
            n_unique, nn_count, n_rows, high_cardinality_threshold, very_high_cardinality_threshold
        )

        # Sample values
        samples = []
        if include_samples and nn_count > 0:
            samples = _get_samples(df[col], sample_size)

        # Detect if numeric column is actually categorical
        is_numeric_categorical = (
            pd.api.types.is_numeric_dtype(df[col])
            and n_unique <= very_high_cardinality_threshold
            and n_unique > 2
        )

        rows.append({
            "column": col,
            "dtype": dtype,
            "non_null_count": nn_count,
            "missing_count": n_rows - nn_count,
            "missing_pct": round((n_rows - nn_count) / n_rows * 100, 2),
            "cardinality_count": n_unique,
            "cardinality_group": cardinality_group,
            "high_cardinality_flag": high_cardinality_flag,
            "very_high_cardinality_flag": very_high_cardinality_flag,
            "potential_id_flag": is_potential_id,
            "numeric_categorical_flag": is_numeric_categorical,
            "encoding_recommendation": encoding_rec,
            "sample_values": samples,
        })

    # Create DataFrame
    report = pd.DataFrame(rows)

    # Smart sorting: most interesting/useful first
    # Priority: potential IDs > very high > high > medium > low > binary > single
    cardinality_priority = {
        "potential_id": 0,
        "very_high_cardinality": 1,
        "high_cardinality": 2,
        "medium_cardinality": 3,
        "low_cardinality": 4,
        "binary": 5,
        "single_category": 6,
        "no_category": 7,
    }

    # Map groups to priority
    def get_priority(group):
        if group == "very_high_cardinality":
            return 1
        elif group == "high_cardinality":
            return 2
        elif group == "medium_cardinality":
            return 3
        elif group == "low_cardinality":
            return 4
        elif group == "binary":
            return 5
        elif group == "single_category":
            return 6
        elif group == "no_category":
            return 7
        return 99

    report["_sort_priority"] = report["cardinality_group"].apply(get_priority)
    report["_sort_priority"] = report.apply(
        lambda r: 0 if r["potential_id_flag"] else r["_sort_priority"], axis=1
    )

    report = report.sort_values(
        by=["_sort_priority", "cardinality_count"],
        ascending=[True, False],
        na_position="last",
    ).drop(columns=["_sort_priority"]).reset_index(drop=True)

    return report


def _get_output_columns() -> List[str]:
    return [
        "column", "dtype", "non_null_count", "missing_count", "missing_pct",
        "cardinality_count", "cardinality_group", "high_cardinality_flag",
        "very_high_cardinality_flag", "potential_id_flag", "numeric_categorical_flag",
        "encoding_recommendation", "sample_values"
    ]


def _assign_cardinality_group(
    n_unique: int,
    high_threshold: int,
    very_high_threshold: int
) -> str:
    """Assign cardinality group based on unique value count."""
    if n_unique == 0:
        return "no_category"
    elif n_unique == 1:
        return "single_category"
    elif n_unique == 2:
        return "binary"
    elif n_unique <= 10:
        return "low_cardinality"
    elif n_unique <= 50:
        return "medium_cardinality"
    elif n_unique <= very_high_threshold:
        return "high_cardinality"
    else:
        return "very_high_cardinality"


def _looks_like_id(col_name: str, dtype: str) -> bool:
    """Heuristic: column name suggests an identifier."""
    col_lower = col_name.lower().strip()
    id_indicators = {
        "id", "key", "code", "number", "no", "num", "ref", "reference",
        "uuid", "guid", "token", "identifier", "account", "user_id",
        "customer_id", "product_id", "order_id", "transaction_id"
    }
    return any(indicator in col_lower for indicator in id_indicators)


def _recommend_encoding(
    n_unique: int,
    nn_count: int,
    n_rows: int,
    high_threshold: int,
    very_high_threshold: int
) -> str:
    """Recommend encoding strategy based on cardinality."""
    if n_unique == 0:
        return "drop_column"
    elif n_unique == 1:
        return "drop_column (constant)"
    elif n_unique == 2:
        return "binary_encoding or label_encoding"
    elif n_unique <= 10:
        return "one_hot_encoding or label_encoding"
    elif n_unique <= high_threshold:
        return "target_encoding, leave_as_integer, or one_hot_if_dense"
    elif n_unique <= very_high_threshold:
        return "target_encoding, frequency_encoding, or hash_encoding"
    else:
        return "target_encoding, hash_encoding, or embedding_layer"


def _get_samples(col: pd.Series, n: int) -> List[Any]:
    """Get representative sample values from column."""
    try:
        unique_vals = col.dropna().unique()
        if len(unique_vals) == 0:
            return []
        
        # Get evenly spaced samples across unique values
        step = max(1, len(unique_vals) // n)
        samples = unique_vals[::step][:n]
        
        # Convert to clean strings, truncate long ones
        result = []
        for v in samples:
            s = str(v)
            if len(s) > 40:
                s = s[:40] + "…"
            result.append(s)
        return result
    except Exception:
        return []


# ─────────────────────────────────────────────────────────────────
# Optional: Fast summary view (if you only want the basics)
# ─────────────────────────────────────────────────────────────────

def cardinality_summary(
    df: pd.DataFrame,
    high_threshold: int = 50
) -> pd.DataFrame:
    """
    Lightweight cardinality summary - just the essentials.
    Useful for a quick overview before deeper analysis.
    """
    unique_counts = df.nunique(dropna=True)
    non_null_counts = df.notna().sum()

    summary = pd.DataFrame({
        "column": df.columns,
        "unique_count": unique_counts.values,
        "non_null_count": non_null_counts.values,
        "cardinality": pd.cut(
            unique_counts,
            bins=[-1, 1, 2, 10, 50, 200, float("inf")],
            labels=["constant", "binary", "low", "medium", "high", "very_high"]
        )
    })

    summary["action"] = summary["unique_count"].apply(
        lambda x: "drop" if x <= 1 else ("one_hot" if x <= 10 else "encode")
    )

    return summary.sort_values("unique_count", ascending=False).reset_index(drop=True)

In [37]:
train_category_cardinality_report = category_cardinality_report(train_df)
train_category_cardinality_report

,column,dtype,non_null_count,missing_count,missing_pct,cardinality_count,cardinality_group,high_cardinality_flag,very_high_cardinality_flag,potential_id_flag,numeric_categorical_flag,encoding_recommendation,sample_values
0,SK_ID_CURR,int64,307511,0,0.00,307511,very_high_cardinality,True,True,True,False,"target_encoding, hash_encoding, or embedding_l...","[100002, 171327, 242626, 313865, 384696]"
1,APP_EXT_SOURCE_MEAN,float64,307339,172,0.06,300347,very_high_cardinality,True,True,False,False,"target_encoding, hash_encoding, or embedding_l...","[0.1617871134127632, 0.560550068258009, 0.5274..."
2,INS_AMT_PAYMENT_MEAN,float64,291635,15876,5.16,288396,very_high_cardinality,True,True,False,False,"target_encoding, hash_encoding, or embedding_l...","[11559.24710526316, 38248.4594117647, 7334.955..."
3,INS_AMT_INSTALMENT_MEAN,float64,291643,15868,5.16,288020,very_high_cardinality,True,True,False,False,"target_encoding, hash_encoding, or embedding_l...","[11559.24710526316, 10210.03415492958, 10638.4..."
4,INS_AMT_INSTALMENT_SUM,float64,291643,15868,5.16,287801,very_high_cardinality,True,True,False,False,"target_encoding, hash_encoding, or embedding_l...","[219625.695, 200023.38, 151784.505, 24103.08, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,PREV_CHANNEL_TYPE_nan_MEAN,float64,291057,16454,5.35,1,single_category,False,False,False,False,drop_column (constant),[0.0]
580,PREV_NAME_SELLER_INDUSTRY_nan_MEAN,float64,291057,16454,5.35,1,single_category,False,False,False,False,drop_column (constant),[0.0]
581,PREV_NAME_YIELD_GROUP_nan_MEAN,float64,291057,16454,5.35,1,single_category,False,False,False,False,drop_column (constant),[0.0]
582,POS_NAME_CONTRACT_STATUS_nan_MEAN,float64,289444,18067,5.88,1,single_category,False,False,False,False,drop_column (constant),[0.0]


# 4.2 category_frequency_report

* top_category
* top_category_percentage
* rare_category_count
* rare_category_percentage
* rare_grouping_recommended
* category_distribution_balance

In [38]:
import pandas as pd
import numpy as np
from scipy.stats import entropy as scipy_entropy


_OUTPUT_COLUMNS = [
    "column",
    "unique_count",
    "missing_count",
    "missing_percentage",
    "top_category",
    "top_category_percentage",
    "rare_category_count",
    "rare_category_percentage",
    "rare_grouping_recommended",
    "category_distribution_balance",
    "normalized_entropy",
    "high_cardinality",
]


def category_frequency_report(
    df: pd.DataFrame,
    columns: list[str] | None = None,
    include_numeric: bool = False,
    rare_threshold: float = 0.01,
    rare_grouping_threshold: float = 0.20,
    highly_dominant_threshold: float = 80.0,
    moderately_skewed_threshold: float = 50.0,
    high_cardinality_threshold: int = 100,
) -> pd.DataFrame:
    """
    Build a category-frequency report for every selected column.

    Output columns
    --------------
    column, unique_count, missing_count, missing_percentage,
    top_category, top_category_percentage,
    rare_category_count, rare_category_percentage, rare_grouping_recommended,
    category_distribution_balance, normalized_entropy, high_cardinality

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    columns : list[str] | None
        Explicit column list.  When *None* the function auto-selects
        categorical / object / bool columns (unless *include_numeric* is True).
    include_numeric : bool, default False
        When *columns* is None, also analyse numeric columns.
    rare_threshold : float, default 0.01
        Categories whose frequency proportion ≤ this value are "rare".
    rare_grouping_threshold : float, default 0.20
        If the fraction of rare categories ≥ this value, grouping is
        recommended.
    highly_dominant_threshold : float, default 80.0
        Top-category % above which the column is labelled *highly_dominant*.
    moderately_skewed_threshold : float, default 50.0
        Top-category % above which the column is labelled *moderately_skewed*.
    high_cardinality_threshold : int, default 100
        Unique-count ≥ this value flags the column as high-cardinality.

    Returns
    -------
    pd.DataFrame
        One row per analysed column.
    """
    # ── validation ──────────────────────────────────────────────────────
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected pd.DataFrame, got {type(df).__name__}")

    if df.empty:
        return pd.DataFrame(columns=_OUTPUT_COLUMNS)

    # ── column selection ────────────────────────────────────────────────
    target_cols = _resolve_columns(df, columns, include_numeric)

    # ── build report ────────────────────────────────────────────────────
    rows: list[dict] = []
    for col in target_cols:
        rows.append(
            _analyse_column(
                df[col],
                col,
                rare_threshold,
                rare_grouping_threshold,
                highly_dominant_threshold,
                moderately_skewed_threshold,
                high_cardinality_threshold,
            )
        )

    return pd.DataFrame(rows, columns=_OUTPUT_COLUMNS)


# ── helpers ─────────────────────────────────────────────────────────────


def _resolve_columns(
    df: pd.DataFrame,
    columns: list[str] | None,
    include_numeric: bool,
) -> list[str]:
    """Return the list of columns to analyse."""
    if columns is not None:
        missing = set(columns) - set(df.columns)
        if missing:
            raise ValueError(f"Columns not found in DataFrame: {missing}")
        return list(columns)

    if include_numeric:
        return df.columns.tolist()

    return df.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()


def _normalised_entropy(proportions: np.ndarray, n_unique: int) -> float:
    """Return Shannon entropy normalised to [0, 1].

    0 → single category dominates;  1 → perfectly uniform.
    Uses ``scipy.stats.entropy`` (base-2) for numerical stability.
    """
    if n_unique <= 1:
        return 0.0
    ent = scipy_entropy(proportions, base=2)
    max_ent = np.log2(n_unique)
    return float(ent / max_ent) if max_ent > 0 else 0.0


def _classify_balance(
    n_unique: int,
    top_pct: float,
    norm_entropy: float,
    highly_dominant_threshold: float,
    moderately_skewed_threshold: float,
) -> str:
    """Classify distribution balance using *both* top-% and entropy."""
    if n_unique <= 1:
        return "single_category"
    if top_pct >= highly_dominant_threshold:
        return "highly_dominant"
    if top_pct >= moderately_skewed_threshold:
        return "moderately_skewed"
    # entropy refines the "balanced" bucket
    if norm_entropy >= 0.90:
        return "highly_balanced"
    return "relatively_balanced"


def _analyse_column(
    series: pd.Series,
    col_name: str,
    rare_threshold: float,
    rare_grouping_threshold: float,
    highly_dominant_threshold: float,
    moderately_skewed_threshold: float,
    high_cardinality_threshold: int,
) -> dict:
    """Return the metrics dict for a single column."""
    n_total = len(series)
    n_missing = int(series.isna().sum())
    missing_pct = round(n_missing / n_total * 100, 2) if n_total > 0 else np.nan

    s = series.dropna()
    n_valid = len(s)

    # ── empty-after-dropna shortcut ─────────────────────────────────
    if n_valid == 0:
        return {
            "column": col_name,
            "unique_count": 0,
            "missing_count": n_missing,
            "missing_percentage": missing_pct,
            "top_category": np.nan,
            "top_category_percentage": np.nan,
            "rare_category_count": 0,
            "rare_category_percentage": np.nan,
            "rare_grouping_recommended": False,
            "category_distribution_balance": "no_data",
            "normalized_entropy": np.nan,
            "high_cardinality": False,
        }

    # ── single value_counts → derive proportions in-place ───────────
    value_counts = s.value_counts(dropna=True)
    n_unique = len(value_counts)
    proportions = value_counts.values / n_valid          # numpy – fast

    top_category = value_counts.index[0]
    top_pct = round(proportions[0] * 100, 2)

    # ── rare categories ─────────────────────────────────────────────
    rare_count = int((proportions <= rare_threshold).sum())
    rare_pct = round(rare_count / n_unique * 100, 2)

    rare_grouping = (
        n_unique > 1
        and (rare_count / n_unique) >= rare_grouping_threshold
    )

    # ── entropy + balance ───────────────────────────────────────────
    norm_ent = _normalised_entropy(proportions, n_unique)
    balance = _classify_balance(
        n_unique,
        top_pct,
        norm_ent,
        highly_dominant_threshold,
        moderately_skewed_threshold,
    )

    return {
        "column": col_name,
        "unique_count": n_unique,
        "missing_count": n_missing,
        "missing_percentage": missing_pct,
        "top_category": top_category,
        "top_category_percentage": top_pct,
        "rare_category_count": rare_count,
        "rare_category_percentage": rare_pct,
        "rare_grouping_recommended": rare_grouping,
        "category_distribution_balance": balance,
        "normalized_entropy": round(norm_ent, 4),
        "high_cardinality": n_unique >= high_cardinality_threshold,
    }

In [39]:
train_category_frequency_report = category_frequency_report(train_df)
train_category_frequency_report

,column,unique_count,missing_count,missing_percentage,top_category,top_category_percentage,rare_category_count,rare_category_percentage,rare_grouping_recommended,category_distribution_balance,normalized_entropy,high_cardinality
0,NAME_CONTRACT_TYPE,2,0,0.00,Cash loans,90.48,0,0.00,False,highly_dominant,0.4536,False
1,CODE_GENDER,3,0,0.00,F,65.83,1,33.33,True,moderately_skewed,0.5846,False
2,FLAG_OWN_CAR,2,0,0.00,N,65.99,0,0.00,False,moderately_skewed,0.9249,False
3,FLAG_OWN_REALTY,2,0,0.00,Y,69.37,0,0.00,False,moderately_skewed,0.8889,False
4,NAME_TYPE_SUITE,7,1292,0.42,Unaccompanied,81.16,3,42.86,True,highly_dominant,0.3387,False
5,NAME_INCOME_TYPE,8,0,0.00,Working,51.63,4,50.00,True,moderately_skewed,0.5666,False
6,NAME_EDUCATION_TYPE,5,0,0.00,Secondary / secondary special,71.02,1,20.00,True,moderately_skewed,0.4716,False
7,NAME_FAMILY_STATUS,6,0,0.00,Married,63.88,1,16.67,False,moderately_skewed,0.6283,False
8,NAME_HOUSING_TYPE,6,0,0.00,House / apartment,88.73,2,33.33,True,highly_dominant,0.2789,False
9,OCCUPATION_TYPE,18,96391,31.35,Laborers,26.14,6,33.33,True,relatively_balanced,0.7917,False


In [40]:
import pandas as pd
import numpy as np
from typing import Tuple, Dict, List
from dataclasses import dataclass
from collections import Counter
import warnings
from sklearn.feature_extraction.text import CountVectorizer
from thefuzz import fuzz  # fuzzy string matching

@dataclass
class CategoryAnalysis:
    """Data class to store category analysis results"""
    column: str
    unseen_categories: set
    unseen_category_risk: bool
    unseen_category_count: int
    unseen_category_ratio: float
    potential_typos: List[Tuple[str, str, int]]  # (test_val, train_val, similarity_score)
    train_category_dist: Dict[str, int]
    test_category_dist: Dict[str, int]

def _is_categorical_like(series: pd.Series) -> bool:
    """Improved categorical detection with more comprehensive checks"""
    if pd.api.types.is_categorical_dtype(series):
        return True

    # Check for object/string types
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        # Additional check for low cardinality (typical of categorical data)
        unique_ratio = series.nunique() / len(series)
        return unique_ratio < 0.5  # If <50% unique values, likely categorical

    return False

def _normalize_categories(series: pd.Series) -> pd.Series:
    """Normalize categories with consistent handling"""
    if series.empty:
        return series

    # Convert to string and handle NaN
    series = series.astype(str).str.strip().str.lower()

    # Replace common problematic values
    series = series.replace({
        'nan': np.nan,
        'none': np.nan,
        'null': np.nan,
        '': np.nan
    })

    return series

def _detect_potential_typos(test_val: str, train_categories: set, threshold: int = 80) -> List[Tuple[str, int]]:
    """Detect potential typos in test categories using fuzzy matching"""
    if not test_val or pd.isna(test_val):
        return []

    matches = []
    for train_val in train_categories:
        if pd.isna(train_val):
            continue
        similarity = fuzz.ratio(test_val, train_val)
        if similarity >= threshold:
            matches.append((train_val, similarity))

    # Sort by similarity score descending
    matches.sort(key=lambda x: x[1], reverse=True)
    return matches

def _calculate_category_distribution(series: pd.Series) -> Dict[str, int]:
    """Calculate category distribution with proper NaN handling"""
    if series.empty:
        return {}

    # Count non-null values
    counts = Counter(series.dropna())
    return dict(counts)

def category_generalization_report(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    typo_threshold: int = 80,
    min_category_count: int = 5
) -> pd.DataFrame:
    """
    Enhanced categorical generalization report comparing train vs test datasets.

    Parameters:
    -----------
    train_df : pd.DataFrame
        Training dataset
    test_df : pd.DataFrame
        Test dataset
    typo_threshold : int, default=80
        Minimum similarity score (0-100) to consider a test value as a potential typo
    min_category_count : int, default=5
        Minimum count of a category in train to be considered for typo detection

    Returns:
    --------
    pd.DataFrame
        Report with detailed category analysis for each column

    Features:
    ---------
    - More accurate categorical detection
    - Fuzzy matching for potential typos
    - Category distribution analysis
    - Better handling of missing values
    - More comprehensive metrics
    - Type hints for better code clarity
    """

    # Input validation
    if not isinstance(train_df, pd.DataFrame) or not isinstance(test_df, pd.DataFrame):
        raise ValueError("Both train_df and test_df must be pandas DataFrames")

    common_columns = sorted(set(train_df.columns).intersection(set(test_df.columns)))
    analysis_results = []

    for col in common_columns:
        try:
            train_s = train_df[col]
            test_s = test_df[col]

            # Skip if column is empty in both datasets
            if train_s.empty and test_s.empty:
                warnings.warn(f"Column '{col}' is empty in both datasets - skipping")
                continue

            # Check if column is categorical-like
            if not _is_categorical_like(train_s) and not _is_categorical_like(test_s):
                analysis_results.append(CategoryAnalysis(
                    column=col,
                    unseen_categories=set(),
                    unseen_category_risk=False,
                    unseen_category_count=0,
                    unseen_category_ratio=0.0,
                    potential_typos=[],
                    train_category_dist={},
                    test_category_dist={}
                ))
                continue

            # Normalize categories
            train_s_norm = _normalize_categories(train_s)
            test_s_norm = _normalize_categories(test_s)

            # Get unique categories
            train_categories = set(train_s_norm.dropna().unique())
            test_categories = set(test_s_norm.dropna().unique())

            # Calculate unseen categories
            unseen_categories = test_categories - train_categories
            unseen_category_count = len(unseen_categories)
            unseen_category_ratio = unseen_category_count / len(test_categories) if test_categories else 0.0
            unseen_category_risk = unseen_category_count > 0

            # Detect potential typos (only for columns with reasonable category counts)
            potential_typos = []
            if len(train_categories) > 0 and len(test_categories) > 0:
                # Only check for typos if we have enough data
                if len(train_s) >= min_category_count and len(test_s) >= min_category_count:
                    for test_val in unseen_categories:
                        matches = _detect_potential_typos(test_val, train_categories, typo_threshold)
                        if matches:
                            potential_typos.append((test_val, matches[0][0], matches[0][1]))

            # Calculate category distributions
            train_dist = _calculate_category_distribution(train_s_norm)
            test_dist = _calculate_category_distribution(test_s_norm)

            analysis_results.append(CategoryAnalysis(
                column=col,
                unseen_categories=unseen_categories,
                unseen_category_risk=unseen_category_risk,
                unseen_category_count=unseen_category_count,
                unseen_category_ratio=unseen_category_ratio,
                potential_typos=potential_typos,
                train_category_dist=train_dist,
                test_category_dist=test_dist
            ))

        except Exception as e:
            warnings.warn(f"Error processing column '{col}': {str(e)}")
            continue

    # Convert results to DataFrame
    report = pd.DataFrame([{
        "column": res.column,
        "unseen_category_risk": res.unseen_category_risk,
        "unseen_category_count": res.unseen_category_count,
        "unseen_category_ratio": res.unseen_category_ratio,
        "unseen_categories": ", ".join(sorted(res.unseen_categories)) if res.unseen_categories else None,
        "potential_typos": "; ".join([f"{t[0]}→{t[1]}({t[2]})" for t in res.potential_typos]) if res.potential_typos else None,
        "train_category_count": len(res.train_category_dist),
        "test_category_count": len(res.test_category_dist),
        "train_top_categories": ", ".join([f"{k}({v})" for k, v in Counter(res.train_category_dist).most_common(3)]),
        "test_top_categories": ", ".join([f"{k}({v})" for k, v in Counter(res.test_category_dist).most_common(3)])
    } for res in analysis_results])

    # Reorder columns for better readability
    column_order = [
        "column",
        "unseen_category_risk",
        "unseen_category_count",
        "unseen_category_ratio",
        "unseen_categories",
        "potential_typos",
        "train_category_count",
        "test_category_count",
        "train_top_categories",
        "test_top_categories"
    ]

    return report[column_order]

In [41]:
category_generalization_report_df = category_generalization_report(train_df, test_df)
category_generalization_report_df

,column,unseen_category_risk,unseen_category_count,unseen_category_ratio,unseen_categories,potential_typos,train_category_count,test_category_count,train_top_categories,test_top_categories
0,AMT_ANNUITY,False,0,0.0,None,None,0,0,,
1,AMT_CREDIT,False,0,0.0,None,None,0,0,,
2,AMT_GOODS_PRICE,False,0,0.0,None,None,0,0,,
3,AMT_INCOME_TOTAL,False,0,0.0,None,None,0,0,,
4,AMT_REQ_CREDIT_BUREAU_DAY,False,0,0.0,None,None,0,0,,
...,...,...,...,...,...,...,...,...,...,...
578,YEARS_BEGINEXPLUATATION_MEDI,False,0,0.0,None,None,0,0,,
579,YEARS_BEGINEXPLUATATION_MODE,False,0,0.0,None,None,0,0,,
580,YEARS_BUILD_AVG,False,0,0.0,None,None,0,0,,
581,YEARS_BUILD_MEDI,False,0,0.0,None,None,0,0,,


In [42]:
import pandas as pd
from functools import reduce

categorical_feature_profile_report = reduce(
    lambda left, right: pd.merge(left, right, on="column", how="outer"),
    [
        train_category_cardinality_report,
        train_category_frequency_report,
        category_generalization_report_df
    ]
)

categorical_feature_profile_report

,column,dtype,non_null_count,missing_count_x,missing_pct,cardinality_count,cardinality_group,high_cardinality_flag,very_high_cardinality_flag,potential_id_flag,...,high_cardinality,unseen_category_risk,unseen_category_count,unseen_category_ratio,unseen_categories,potential_typos,train_category_count,test_category_count,train_top_categories,test_top_categories
0,AMT_ANNUITY,float64,307499,12,0.00,13672,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
1,AMT_CREDIT,float64,307511,0,0.00,5603,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
2,AMT_GOODS_PRICE,float64,307233,278,0.09,1002,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
3,AMT_INCOME_TOTAL,float64,307511,0,0.00,2548,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
4,AMT_REQ_CREDIT_BUREAU_DAY,float64,265992,41519,13.50,9,low_cardinality,False,False,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,YEARS_BEGINEXPLUATATION_MEDI,float64,157504,150007,48.78,245,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
580,YEARS_BEGINEXPLUATATION_MODE,float64,157504,150007,48.78,221,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
581,YEARS_BUILD_AVG,float64,103023,204488,66.50,149,high_cardinality,True,False,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
582,YEARS_BUILD_MEDI,float64,103023,204488,66.50,151,high_cardinality,True,False,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,


In [43]:
categorical_feature_profile_report.to_csv(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\4.categorical_feature.csv")

# Step 5 Categorical Target-Aware Audit 

### 5.1 category_target_signal_report
* category_target_rate_variation
* chi_square_test_pvalue
* cramers_v
* mutual_information

In [44]:
import warnings

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, entropy as scipy_entropy
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score


# ── helpers ────────────────────────────────────────────────────────────────

def _safe_round(value, decimals):
    """Round if finite, else NaN."""
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), decimals)


def _bias_corrected_cramers_v(chi2, n, r, k):
    """
    Bias-corrected Cramér's V  (Bergsma, 2013).

    Standard Cramér's V over-estimates association in small samples
    or when the contingency table has many cells.  The correction
    shrinks phi² and the row/column counts toward their expected
    values under independence.

    Parameters
    ----------
    chi2 : float   – chi-squared statistic
    n    : int     – total observations
    r    : int     – rows in contingency table
    k    : int     – columns in contingency table
    """
    if n <= 1:
        return np.nan

    phi2      = chi2 / n
    phi2_corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr    = r - ((r - 1) ** 2) / (n - 1)
    k_corr    = k - ((k - 1) ** 2) / (n - 1)
    denom     = min(k_corr - 1, r_corr - 1)

    return np.sqrt(phi2_corr / denom) if denom > 0 else np.nan


def _theils_u(feature, target):
    """
    Theil's Uncertainty Coefficient  U(target | feature).

    Interpretation: "How much does knowing *feature* reduce our
    uncertainty about *target*?"     Range [0, 1].
    """
    target_counts = pd.Series(target).value_counts().values
    h_target = scipy_entropy(target_counts)       # natural-log; normalises counts

    if h_target == 0:
        return 0.0

    mi = mutual_info_score(target, feature)        # also natural-log ⇒ compatible
    return float(np.clip(mi / h_target, 0.0, 1.0))


def _get_categorical_columns(df, target_col):
    """Return column names whose dtype is object / string / Categorical."""
    return [
        col for col in df.columns
        if col != target_col
        and (
            pd.api.types.is_object_dtype(df[col])
            or pd.api.types.is_string_dtype(df[col])
            or isinstance(df[col].dtype, pd.CategoricalDtype)   # pandas ≥ 2.1 safe
        )
    ]


# ── main report ────────────────────────────────────────────────────────────

def category_target_signal_report(
    df: pd.DataFrame,
    target_col: str,
    min_category_count: int = 5,
    sort_by: str = "cramers_v",
    ascending: bool = False,
) -> pd.DataFrame:
    """
    Target-aware signal report for every categorical feature.

    Metrics (one row per feature)
    -----------------------------
    n_categories                     unique-value count
    category_target_rate_variation   max − min target rate (numeric targets only)
    weighted_target_rate_std         size-weighted σ of target rates
    chi_square_pvalue                χ² test of independence
    cramers_v                        bias-corrected Cramér's V  (Bergsma 2013)
    normalized_mutual_info           NMI ∈ [0, 1] – comparable across features
    theils_u                         U(target | feature) ∈ [0, 1]

    Parameters
    ----------
    df                 : input DataFrame
    target_col         : binary / discrete target column name
    min_category_count : drop categories below this count for rate stats
    sort_by            : metric to rank features by
    ascending          : False → strongest signal first

    Returns
    -------
    pd.DataFrame
    """
    # ── validation ──────────────────────────────────────────────────────
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found.")

    target_nunique = df[target_col].nunique()
    if target_nunique > 20:
        warnings.warn(
            f"Target has {target_nunique} unique values – this report is "
            "designed for binary / low-cardinality discrete targets.",
            UserWarning,
        )

    cat_cols = _get_categorical_columns(df, target_col)
    if not cat_cols:
        warnings.warn("No categorical columns detected.", UserWarning)
        return pd.DataFrame()

    is_numeric_target = pd.api.types.is_numeric_dtype(df[target_col])

    # ── per-column metrics ──────────────────────────────────────────────
    records: list[dict] = []

    for col in cat_cols:
        temp = df[[col, target_col]].dropna()

        # nothing left after dropping NaNs
        if temp.empty:
            records.append({"column": col})
            continue

        feat = temp[col].astype(str)
        tgt  = temp[target_col]
        n_categories = feat.nunique()

        # 1 ── target-rate statistics (only meaningful for numeric 0/1) ──
        rate_variation = w_std = np.nan
        if is_numeric_target:
            rate_df = pd.DataFrame({"cat": feat.values, "tgt": tgt.values})
            grp = (
                rate_df
                .groupby("cat")["tgt"]
                .agg(["mean", "count"])
            )
            grp = grp[grp["count"] >= min_category_count]

            if len(grp) >= 2:
                rate_variation = float(grp["mean"].max() - grp["mean"].min())
                w     = grp["count"] / grp["count"].sum()
                wmean = np.average(grp["mean"], weights=w)
                w_std = float(
                    np.sqrt(np.average((grp["mean"] - wmean) ** 2, weights=w))
                )

        # 2 ── chi-square + Cramér's V  (ONE chi² call) ─────────────────
        ct = pd.crosstab(feat, tgt)
        pvalue = cramers_v = np.nan

        if ct.shape[0] >= 2 and ct.shape[1] >= 2:
            try:
                chi2_stat, pvalue, _, _ = chi2_contingency(ct)
                n    = ct.values.sum()
                r, k = ct.shape
                cramers_v = _bias_corrected_cramers_v(chi2_stat, n, r, k)
            except ValueError as exc:          # e.g. zero marginal
                warnings.warn(f"χ² failed for '{col}': {exc}", RuntimeWarning)

        # 3 ── normalised mutual information ────────────────────────────
        try:
            nmi = normalized_mutual_info_score(
                feat, tgt, average_method="arithmetic",
            )
        except ValueError:
            nmi = np.nan

        # 4 ── Theil's U(target | feature) ─────────────────────────────
        try:
            tu = _theils_u(feat, tgt)
        except ValueError:
            tu = np.nan

        records.append(
            {
                "column":                          col,
                "n_categories":                    n_categories,
                "category_target_rate_variation":   _safe_round(rate_variation, 4),
                "weighted_target_rate_std":         _safe_round(w_std, 4),
                "chi_square_pvalue":               _safe_round(pvalue, 6),
                "cramers_v":                       _safe_round(cramers_v, 4),
                "normalized_mutual_info":          _safe_round(nmi, 6),
                "theils_u":                        _safe_round(tu, 4),
            }
        )

    # ── assemble & sort ─────────────────────────────────────────────────
    result = pd.DataFrame(records)

    if sort_by in result.columns and not result.empty:
        result = (
            result
            .sort_values(sort_by, ascending=ascending, na_position="last")
            .reset_index(drop=True)
        )

    return result

In [45]:
train_category_target_signal_report = category_target_signal_report(
    df=train_df,
    target_col="TARGET"
)

train_category_target_signal_report

,column,n_categories,category_target_rate_variation,weighted_target_rate_std,chi_square_pvalue,cramers_v,normalized_mutual_info,theils_u
0,OCCUPATION_TYPE,18,0.1232,0.0231,0.000000,0.0810,0.002617,0.0114
1,ORGANIZATION_TYPE,58,0.1263,0.0197,0.000000,0.0710,0.001708,0.0095
2,NAME_INCOME_TYPE,8,0.4000,0.0174,0.000000,0.0637,0.002898,0.0075
3,NAME_EDUCATION_TYPE,5,0.0910,0.0157,0.000000,0.0575,0.003456,0.0064
4,CODE_GENDER,3,0.0314,0.0149,0.000000,0.0547,0.003146,0.0052
5,NAME_FAMILY_STATUS,6,0.0412,0.0110,0.000000,0.0403,0.001150,0.0029
6,NAME_HOUSING_TYPE,6,0.0574,0.0101,0.000000,0.0368,0.001570,0.0022
7,NAME_CONTRACT_TYPE,2,0.0287,0.0084,0.000000,0.0308,0.001773,0.0019
8,WALLSMATERIAL_MODE,7,0.0498,0.0077,0.000000,0.0297,0.000625,0.0018
9,FLAG_OWN_CAR,2,0.0126,0.0060,0.000000,0.0218,0.000527,0.0009


### 5.2 category_target_rate_table_export_df
* category
* category_count
* category_percentage
* category_target_rate

In [46]:
import logging
import warnings
from typing import List, Optional

import pandas as pd
import numpy as np

logger = logging.getLogger(__name__)


def _detect_categorical_columns(
    df: pd.DataFrame,
    target_col: str,
    max_cardinality: int,
) -> List[str]:
    """
    Detect categorical columns intelligently.

    Includes:
    - object / string / categorical / boolean dtypes
    - Numeric columns with cardinality <= max_cardinality (likely encoded cats)

    Excludes:
    - The target column itself
    """
    candidates = []
    for col in df.columns:
        if col == target_col:
            continue

        dtype = df[col].dtype

        # Obvious categorical types
        if pd.api.types.is_object_dtype(dtype):
            candidates.append(col)
        elif pd.api.types.is_string_dtype(dtype):
            candidates.append(col)
        elif isinstance(dtype, pd.CategoricalDtype):
            candidates.append(col)
        elif pd.api.types.is_bool_dtype(dtype):
            candidates.append(col)
        # Low-cardinality numerics are often encoded categoricals
        elif pd.api.types.is_numeric_dtype(dtype):
            nunique = df[col].nunique(dropna=True)
            if 0 < nunique <= max_cardinality:
                candidates.append(col)
                logger.info(
                    f"Column '{col}' is numeric but has only {nunique} unique "
                    f"values — treating as categorical."
                )
    return candidates


def _validate_inputs(df: pd.DataFrame, target_col: str) -> None:
    """Validate dataframe and target column."""
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected pd.DataFrame, got {type(df).__name__}")

    if df.empty:
        raise ValueError("Input dataframe is empty.")

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe.")

    if not pd.api.types.is_numeric_dtype(df[target_col]):
        raise ValueError(
            f"Target column '{target_col}' must be numeric (got {df[target_col].dtype}). "
            f"Target rate (mean) is only meaningful for numeric/binary targets."
        )

    # Warn if target doesn't look binary
    unique_vals = df[target_col].dropna().unique()
    if not set(unique_vals).issubset({0, 1, 0.0, 1.0, True, False}):
        warnings.warn(
            f"Target column '{target_col}' has values beyond {{0, 1}}. "
            f"Target rate (mean) is most interpretable for binary targets.",
            UserWarning,
            stacklevel=3,
        )


def _build_single_column_stats(
    df: pd.DataFrame,
    col: str,
    target_col: str,
    missing_label: str,
    include_missing: bool,
) -> Optional[pd.DataFrame]:
    """Compute category-level stats for a single column."""
    temp = df[[col, target_col]].copy()

    if include_missing:
        # Fill NaN with missing label — use object dtype to safely insert string
        temp[col] = temp[col].astype("object").fillna(missing_label)
        temp = temp.dropna(subset=[target_col])
    else:
        temp = temp.dropna(subset=[col, target_col])

    if temp.empty:
        logger.warning(
            f"Column '{col}' has no valid rows after dropping nulls — skipping."
        )
        return None

    total = len(temp)

    grouped = (
        temp
        .groupby(col, observed=True)[target_col]
        .agg(category_count="count", category_target_rate="mean")
        .reset_index()
        .rename(columns={col: "category"})
    )

    grouped.insert(0, "column", col)
    grouped["category_percentage"] = (grouped["category_count"] / total) * 100

    # Round
    grouped = grouped.round({
        "category_target_rate": 4,
        "category_percentage": 2,
    })

    # Sort by count descending
    grouped = (
        grouped[["column", "category", "category_count",
                 "category_percentage", "category_target_rate"]]
        .sort_values("category_count", ascending=False)
        .reset_index(drop=True)
    )

    return grouped


# ─── Output column schema (single source of truth) ───
OUTPUT_COLUMNS = [
    "column",
    "category",
    "category_count",
    "category_percentage",
    "category_target_rate",
]


def category_target_rate_table_export_df(
    df: pd.DataFrame,
    target_col: str,
    include_missing_as_category: bool = False,
    missing_label: str = "__MISSING__",
    max_cardinality: int = 50,
) -> pd.DataFrame:
    """
    Build a long-format category-wise target rate export table.

    Output columns
    ──────────────
    column                : Name of the categorical feature
    category              : Category value
    category_count        : Number of rows with this category
    category_percentage   : Percentage of rows (within the column)
    category_target_rate  : Mean of target for this category

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    target_col : str
        Name of the binary/numeric target column.
    include_missing_as_category : bool, default False
        If True, NaN values are treated as a separate category.
    missing_label : str, default '__MISSING__'
        Label used for missing values when ``include_missing_as_category=True``.
    max_cardinality : int, default 50
        Numeric columns with at most this many unique values are treated
        as categorical.  Set to 0 to disable this heuristic.

    Returns
    -------
    pd.DataFrame
        Long-format table with one row per (column, category) pair.

    Raises
    ------
    TypeError
        If ``df`` is not a DataFrame.
    ValueError
        If ``df`` is empty or ``target_col`` is missing / non-numeric.

    Examples
    --------
    >>> import pandas as pd
    >>> data = pd.DataFrame({
    ...     "color": ["red", "blue", "red", "green", "blue", "red"],
    ...     "size":  ["S", "M", "L", "M", "S", "L"],
    ...     "target": [1, 0, 1, 0, 1, 0],
    ... })
    >>> result = category_target_rate_table_export_df(data, "target")
    >>> result.columns.tolist()
    ['column', 'category', 'category_count', 'category_percentage', 'category_target_rate']
    """

    # ── Validate ──────────────────────────────────────────────────────────
    _validate_inputs(df, target_col)

    # Check collision between missing_label and real data
    if include_missing_as_category:
        for col in df.columns:
            if col == target_col:
                continue
            if df[col].astype(str).eq(missing_label).any():
                warnings.warn(
                    f"Column '{col}' already contains the value '{missing_label}'. "
                    f"Consider using a different `missing_label` to avoid ambiguity.",
                    UserWarning,
                    stacklevel=2,
                )
                break  # warn once is enough

    # ── Detect categorical columns ────────────────────────────────────────
    categorical_cols = _detect_categorical_columns(df, target_col, max_cardinality)

    if not categorical_cols:
        logger.warning("No categorical columns detected — returning empty DataFrame.")
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    logger.info(f"Processing {len(categorical_cols)} categorical column(s): {categorical_cols}")

    # ── Build per-column stats ────────────────────────────────────────────
    result_tables = []
    for col in categorical_cols:
        stats = _build_single_column_stats(
            df, col, target_col, missing_label, include_missing_as_category
        )
        if stats is not None:
            result_tables.append(stats)

    # ── Combine & return ──────────────────────────────────────────────────
    if result_tables:
        return pd.concat(result_tables, ignore_index=True)

    return pd.DataFrame(columns=OUTPUT_COLUMNS)

In [47]:
train_category_target_rate_table_export_df = category_target_rate_table_export_df(
    df=train_df,
    target_col="TARGET"
)

train_category_target_rate_table_export_df

,column,category,category_count,category_percentage,category_target_rate
0,NAME_CONTRACT_TYPE,Cash loans,278232,90.48,0.0835
1,NAME_CONTRACT_TYPE,Revolving loans,29279,9.52,0.0548
2,CODE_GENDER,F,202448,65.83,0.0700
3,CODE_GENDER,M,105059,34.16,0.1014
4,CODE_GENDER,XNA,4,0.00,0.0000
...,...,...,...,...,...
2009,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,0.010101,1,0.00,0.0000
2010,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,0.010204,1,0.00,0.0000
2011,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,0.014706,1,0.00,0.0000
2012,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,0.02381,1,0.00,0.0000


In [48]:
train_category_target_rate_table_export_df[
    train_category_target_rate_table_export_df["column"] == "NAME_CONTRACT_TYPE"
]

,column,category,category_count,category_percentage,category_target_rate
0,NAME_CONTRACT_TYPE,Cash loans,278232,90.48,0.0835
1,NAME_CONTRACT_TYPE,Revolving loans,29279,9.52,0.0548


In [49]:
train_category_target_rate_table_export_df.query("column == 'NAME_CONTRACT_TYPE'")

,column,category,category_count,category_percentage,category_target_rate
0,NAME_CONTRACT_TYPE,Cash loans,278232,90.48,0.0835
1,NAME_CONTRACT_TYPE,Revolving loans,29279,9.52,0.0548


### 5.3 category_encoding_risk_report
* ordinal_relationship_possible
* encoding_recommendation
* leakage_prone_category_warning

In [50]:
import pandas as pd
import numpy as np
import re
import calendar
from pandas.api.types import is_object_dtype, is_string_dtype
from rapidfuzz import process, fuzz # PyPI package for high-accuracy string matching

def category_encoding_risk_report(
    df: pd.DataFrame,
    low_cardinality_threshold: int = 10,
    high_cardinality_threshold: int = 50
) -> pd.DataFrame:
    """
    Highly Optimized Categorical Encoding + Leakage-Risk Audit.
    """

    # --- 1. Efficient Helpers ---
    def normalize_text(text):
        """Standardizes text by removing special chars and extra spaces."""
        text = str(text).strip().lower()
        return re.sub(r'[^a-z0-9\s]', ' ', text) # Replace non-alphanumeric with space

    def is_categorical_series(s):
        """Modern Pandas check for categorical/string types."""
        return (
            is_object_dtype(s) or 
            is_string_dtype(s) or 
            isinstance(s.dtype, pd.CategoricalDtype)
        )

    # Automatically generate month and weekday sets using standard library
    MONTH_TOKENS = {m.lower() for m in calendar.month_name if m}
    WEEKDAY_TOKENS = {d.lower() for d in calendar.day_name}

    def detect_cyclical_type(unique_vals_set):
        if not unique_vals_set:
            return None
        if unique_vals_set.issubset(WEEKDAY_TOKENS):
            return "weekday"
        if unique_vals_set.issubset(MONTH_TOKENS):
            return "month"
        return None

    def detect_ordinal_possible(unique_vals_set, col_name_lower):
        if not unique_vals_set or len(unique_vals_set) < 2:
            return False

        # Use Sets for O(1) lookups
        ordered_vocab_pool = {
            "low", "medium", "high", "poor", "fair", "good", "excellent",
            "small", "large", "junior", "mid", "senior", "bronze", "silver",
            "gold", "platinum", "lower", "secondary", "higher", "academic"
        }

        # Check exact overlap first (Fast)
        overlap = unique_vals_set.intersection(ordered_vocab_pool)
        if len(overlap) >= 2:
            return True

        # PyPI Integration: RapidFuzz to catch typos (e.g., "exelent", "medum")
        # Achieves >90% accuracy on messy real-world data
        fuzzy_matches = 0
        for val in unique_vals_set:
            # Get best match from the pool
            match = process.extractOne(val, ordered_vocab_pool, scorer=fuzz.WRatio)
            if match and match[1] >= 85: # 85% similarity threshold
                fuzzy_matches += 1
                
        if fuzzy_matches >= 2:
            return True

        # Generic name-based ordinal hints
        ordinal_name_hints = [
            "education", "grade", "level", "rank", "band",
            "quality", "size", "tier", "stage", "score_group"
        ]
        return any(hint in col_name_lower for hint in ordinal_name_hints)

    def leakage_warning(col_name_lower):
        """Uses Regex Word Boundaries (\b) to prevent False Positives."""
        # E.g. prevents "post_code" from triggering on "post"
        leakage_patterns = [
            r"\btarget\b", r"\bdefault\b", r"\bapproved\b", r"\brejected\b", 
            r"\bdecision\b", r"\boutcome\b", r"\bresult\b", r"\bstatus after\b",
            r"\bpost\b", r"\bafter\b", r"\bdelinquent\b", r"\bwriteoff\b",
            r"\brecovery\b", r"\bcharge off\b"
        ]
        combined_pattern = re.compile("|".join(leakage_patterns))
        return bool(combined_pattern.search(col_name_lower))

    # --- 2. Main Execution Loop ---
    rows = []
    
    # Filter columns efficiently
    categorical_cols = [col for col in df.columns if is_categorical_series(df[col])]

    for col in categorical_cols:
        col_lower = normalize_text(col)
        
        # EXTRACT UNIQUE VALUES ONCE (Huge performance boost)
        raw_uniques = df[col].dropna().unique()
        cardinality = len(raw_uniques)
        
        # Normalize uniques once into a set
        unique_vals_set = {normalize_text(x) for x in raw_uniques}

        # Execute logic
        binary_like = (cardinality == 2) # Any 2-class column is inherently binary
        cyclical_type = detect_cyclical_type(unique_vals_set)
        ordinal_possible = detect_ordinal_possible(unique_vals_set, col_lower)
        leakage_flag = leakage_warning(col_lower)

        # Determine Recommendation
        if cardinality <= 1:
            recommendation = "drop_or_review"
        elif leakage_flag:
            recommendation = "review_before_encoding (High Leakage Risk)"
        elif binary_like:
            recommendation = "binary_mapping_or_one_hot"
        elif cyclical_type == "weekday":
            recommendation = "one_hot_or_cyclical_weekday"
        elif cyclical_type == "month":
            recommendation = "one_hot_or_cyclical_month"
        elif ordinal_possible:
            recommendation = "ordinal_encoding"
        elif cardinality <= low_cardinality_threshold:
            recommendation = "one_hot"
        elif cardinality <= high_cardinality_threshold:
            recommendation = "one_hot_or_frequency"
        else:
            recommendation = "frequency_or_target_encoding"

        rows.append({
            "column": col,
            "cardinality": cardinality,
            "ordinal_relationship_possible": ordinal_possible,
            "encoding_recommendation": recommendation,
            "leakage_prone_category_warning": leakage_flag
        })

    return pd.DataFrame(rows)

In [51]:
train_category_encoding_risk_report = category_encoding_risk_report(train_df)
train_category_encoding_risk_report

,column,cardinality,ordinal_relationship_possible,encoding_recommendation,leakage_prone_category_warning
0,NAME_CONTRACT_TYPE,2,False,binary_mapping_or_one_hot,False
1,CODE_GENDER,3,True,ordinal_encoding,False
2,FLAG_OWN_CAR,2,False,binary_mapping_or_one_hot,False
3,FLAG_OWN_REALTY,2,False,binary_mapping_or_one_hot,False
4,NAME_TYPE_SUITE,7,False,one_hot,False
5,NAME_INCOME_TYPE,8,False,one_hot,False
6,NAME_EDUCATION_TYPE,5,True,ordinal_encoding,False
7,NAME_FAMILY_STATUS,6,False,one_hot,False
8,NAME_HOUSING_TYPE,6,False,one_hot,False
9,OCCUPATION_TYPE,18,True,ordinal_encoding,False


### 5.4 category_stability_drift_report
* category_stability_drift_report
* category_drift_summary

In [52]:
import pandas as pd
import numpy as np
from typing import Dict, Any, Tuple
from dataclasses import dataclass


@dataclass
class DriftConfig:
    """Configuration for categorical drift detection."""
    stability_thresholds: Tuple[float, float, float] = (0.95, 0.85, 0.70)
    max_cardinality: int = 50          # Skip high cardinality columns
    min_samples: int = 100
    epsilon: float = 1e-8              # For PSI calculation
    include_psi: bool = True


def _is_categorical_column(series: pd.Series, max_cardinality: int = 50) -> bool:
    """Improved categorical detection with cardinality check."""
    if series.dropna().empty:
        return False
    
    # Check dtype
    if pd.api.types.is_categorical_dtype(series) or \
       pd.api.types.is_object_dtype(series) or \
       pd.api.types.is_string_dtype(series):
        n_unique = series.nunique(dropna=True)
        return n_unique <= max_cardinality
    
    return False


def _calculate_psi(train_dist: pd.Series, test_dist: pd.Series, epsilon: float = 1e-8) -> float:
    """Calculate Population Stability Index (industry standard)."""
    train_dist = train_dist + epsilon
    test_dist = test_dist + epsilon
    train_dist = train_dist / train_dist.sum()
    test_dist = test_dist / test_dist.sum()
    
    psi = (train_dist - test_dist) * np.log(train_dist / test_dist)
    return psi.sum()


def category_stability_drift_report(
    train_df: pd.DataFrame, 
    test_df: pd.DataFrame,
    config: DriftConfig = None
) -> pd.DataFrame:
    """
    Enhanced categorical stability & drift report between train and test sets.
    
    Returns richer metrics with better logic and configurability.
    """
    if config is None:
        config = DriftConfig()

    common_cols = sorted(set(train_df.columns) & set(test_df.columns))
    results = []

    for col in common_cols:
        train_s = train_df[col]
        test_s = test_df[col]

        # Skip non-categorical or high cardinality columns
        if not _is_categorical_column(train_s, config.max_cardinality) and \
           not _is_categorical_column(test_s, config.max_cardinality):
            results.append({
                "column": col,
                "is_categorical": False,
                "category_stability_score": np.nan,
                "psi": np.nan,
                "drift_summary": "not_applicable",
                "n_categories_train": train_s.nunique(dropna=True),
                "n_categories_test": test_s.nunique(dropna=True),
                "missing_rate_train": train_s.isna().mean().round(4),
                "missing_rate_test": test_s.isna().mean().round(4),
            })
            continue

        # Clean and prepare data
        train_clean = train_s.dropna().astype(str).str.strip()
        test_clean = test_s.dropna().astype(str).str.strip()

        if len(train_clean) < config.min_samples or len(test_clean) < config.min_samples:
            results.append({
                "column": col,
                "is_categorical": True,
                "category_stability_score": np.nan,
                "psi": np.nan,
                "drift_summary": "insufficient_samples",
                "n_categories_train": train_clean.nunique(),
                "n_categories_test": test_clean.nunique(),
                "missing_rate_train": train_s.isna().mean().round(4),
                "missing_rate_test": test_s.isna().mean().round(4),
            })
            continue

        # Distribution calculation
        train_dist = train_clean.value_counts(normalize=True)
        test_dist = test_clean.value_counts(normalize=True)
        all_cats = sorted(set(train_dist.index) | set(test_dist.index))

        train_aligned = train_dist.reindex(all_cats, fill_value=0.0)
        test_aligned = test_dist.reindex(all_cats, fill_value=0.0)

        # Total Variation Distance
        tvd = 0.5 * np.abs(train_aligned - test_aligned).sum()
        stability_score = round(1 - tvd, 4)

        # PSI (very useful for drift detection)
        psi = round(_calculate_psi(train_dist, test_dist, config.epsilon), 4) if config.include_psi else np.nan

        # Improved drift logic (more balanced thresholds)
        unseen_in_test = len(set(test_dist.index) - set(train_dist.index))
        
        if unseen_in_test > 0:
            drift_summary = "new_category_detected"
        elif stability_score >= config.stability_thresholds[0] and psi < 0.1:
            drift_summary = "stable"
        elif stability_score >= config.stability_thresholds[1] and psi < 0.25:
            drift_summary = "mild_drift"
        elif stability_score >= config.stability_thresholds[2] and psi < 0.5:
            drift_summary = "moderate_drift"
        else:
            drift_summary = "high_drift"

        results.append({
            "column": col,
            "is_categorical": True,
            "category_stability_score": stability_score,
            "psi": psi,
            "drift_summary": drift_summary,
            "n_categories_train": len(train_dist),
            "n_categories_test": len(test_dist),
            "new_categories": unseen_in_test,
            "missing_rate_train": round(train_s.isna().mean(), 4),
            "missing_rate_test": round(test_s.isna().mean(), 4),
        })

    return pd.DataFrame(results)


In [53]:
train_category_stability_drift_report = category_stability_drift_report(train_df, test_df)
train_category_stability_drift_report

,column,is_categorical,category_stability_score,psi,drift_summary,n_categories_train,n_categories_test,missing_rate_train,missing_rate_test,new_categories
0,AMT_ANNUITY,False,NaN,NaN,not_applicable,13672,7491,0.0000,0.0005,NaN
1,AMT_CREDIT,False,NaN,NaN,not_applicable,5603,2937,0.0000,0.0000,NaN
2,AMT_GOODS_PRICE,False,NaN,NaN,not_applicable,1002,677,0.0009,0.0000,NaN
3,AMT_INCOME_TOTAL,False,NaN,NaN,not_applicable,2548,606,0.0000,0.0000,NaN
4,AMT_REQ_CREDIT_BUREAU_DAY,False,NaN,NaN,not_applicable,9,3,0.1350,0.1241,NaN
...,...,...,...,...,...,...,...,...,...,...
578,YEARS_BEGINEXPLUATATION_MEDI,False,NaN,NaN,not_applicable,245,169,0.4878,0.4689,NaN
579,YEARS_BEGINEXPLUATATION_MODE,False,NaN,NaN,not_applicable,221,160,0.4878,0.4689,NaN
580,YEARS_BUILD_AVG,False,NaN,NaN,not_applicable,149,130,0.6650,0.6528,NaN
581,YEARS_BUILD_MEDI,False,NaN,NaN,not_applicable,151,129,0.6650,0.6528,NaN


In [54]:
import pandas as pd
from functools import reduce

def merge_reports_on_column(reports, key="column", how="outer"):
    if not reports:
        return pd.DataFrame()

    validated = []
    for i, df in enumerate(reports, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Item {i} is not a pandas DataFrame")
        if key not in df.columns:
            raise ValueError(f"DataFrame {i} is missing required key column: '{key}'")
        validated.append(df.copy())

    result = validated[0]
    for i, df in enumerate(validated[1:], start=2):
        overlapping = [c for c in result.columns if c in df.columns and c != key]
        if overlapping:
            df = df.rename(columns={c: f"{c}_report{i}" for c in overlapping})
        result = result.merge(df, on=key, how=how)

    return result

categorical_target_aware_audit_report = merge_reports_on_column([
    train_category_target_signal_report,
    train_category_encoding_risk_report,
    train_category_stability_drift_report
])

In [55]:
categorical_feature_profile_report.head()

,column,dtype,non_null_count,missing_count_x,missing_pct,cardinality_count,cardinality_group,high_cardinality_flag,very_high_cardinality_flag,potential_id_flag,...,high_cardinality,unseen_category_risk,unseen_category_count,unseen_category_ratio,unseen_categories,potential_typos,train_category_count,test_category_count,train_top_categories,test_top_categories
0,AMT_ANNUITY,float64,307499,12,0.00,13672,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
1,AMT_CREDIT,float64,307511,0,0.00,5603,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
2,AMT_GOODS_PRICE,float64,307233,278,0.09,1002,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
3,AMT_INCOME_TOTAL,float64,307511,0,0.00,2548,very_high_cardinality,True,True,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,
4,AMT_REQ_CREDIT_BUREAU_DAY,float64,265992,41519,13.50,9,low_cardinality,False,False,False,...,NaN,False,0.0,0.0,None,None,0.0,0.0,,


In [56]:
categorical_feature_profile_report.to_csv(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\5.categorical_feature_profile.csv")

# Step 6 Numerical Feature Profile

### DataFrame 1: numerical_core_stats_report

* min_value
* max_value
* mean_value
* median_value
* std_dev

In [57]:
import warnings
from typing import Literal

import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis


def _safe_round(value, decimals: int = 4):
    """Round finite values; return NaN otherwise."""
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), decimals)


def _get_numeric_columns(df: pd.DataFrame) -> list[str]:
    """Return numeric (non-boolean) column names."""
    return [
        col for col in df.columns
        if pd.api.types.is_numeric_dtype(df[col])
        and not pd.api.types.is_bool_dtype(df[col])
    ]


def numerical_core_stats_report(
    df: pd.DataFrame,
    percentiles: tuple[float, float] = (0.25, 0.75),
    include_shape_stats: bool = True,
    sort_by: str | None = None,
    ascending: bool = True,
    decimals: int = 4,
) -> pd.DataFrame:
    """
    Vectorized core statistics report for all numeric columns.

    Output Columns
    --------------
    Basic:
        column, count, missing_count, missing_pct

    Range & Central Tendency:
        min_value, max_value, range_value, mean_value, median_value

    Dispersion:
        std_dev, coef_of_variation, p25, p75, iqr

    Shape (optional):
        skewness, kurtosis

    Value Counts:
        zero_count, negative_count

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame
    percentiles : tuple
        Lower and upper percentiles for IQR calculation (default: 0.25, 0.75)
    include_shape_stats : bool
        Whether to include skewness and kurtosis (slightly slower)
    sort_by : str | None
        Column name to sort results by (e.g., 'missing_pct', 'std_dev')
    ascending : bool
        Sort order (default True)
    decimals : int
        Rounding precision for float columns

    Returns
    -------
    pd.DataFrame
        One row per numeric column with comprehensive statistics

    Notes
    -----
    - Boolean columns are excluded
    - Uses vectorized pandas operations for 10-100× speedup
    - Skewness/kurtosis use scipy.stats (Fisher definition)
    """
    # ── Validation ──────────────────────────────────────────────────────
    if df.empty:
        warnings.warn("Empty DataFrame provided.", UserWarning)
        return pd.DataFrame()

    numeric_cols = _get_numeric_columns(df)

    if not numeric_cols:
        warnings.warn("No numeric columns found in DataFrame.", UserWarning)
        return pd.DataFrame()

    # ── Subset to numeric only ──────────────────────────────────────────
    num_df = df[numeric_cols]
    total_rows = len(df)

    # ── Vectorized core aggregations (SINGLE PASS) ──────────────────────
    stats = pd.DataFrame({
        "column":        numeric_cols,
        "count":         num_df.count().values,
        "missing_count": num_df.isna().sum().values,
        "min_value":     num_df.min().values,
        "max_value":     num_df.max().values,
        "mean_value":    num_df.mean().values,
        "median_value":  num_df.median().values,
        "std_dev":       num_df.std().values,
    })

    # ── Derived metrics ─────────────────────────────────────────────────
    stats["missing_pct"] = (
        (stats["missing_count"] / total_rows * 100) if total_rows > 0 else np.nan
    )
    stats["range_value"] = stats["max_value"] - stats["min_value"]

    # Coefficient of Variation (handle mean ≈ 0)
    stats["coef_of_variation"] = np.where(
        np.abs(stats["mean_value"]) > 1e-10,
        stats["std_dev"] / np.abs(stats["mean_value"]),
        np.nan,
    )

    # ── Percentiles & IQR ───────────────────────────────────────────────
    p_low, p_high = percentiles
    p_low_label = f"p{int(p_low * 100)}"
    p_high_label = f"p{int(p_high * 100)}"

    quantiles = num_df.quantile([p_low, p_high])
    stats[p_low_label] = quantiles.loc[p_low].values
    stats[p_high_label] = quantiles.loc[p_high].values
    stats["iqr"] = stats[p_high_label] - stats[p_low_label]

    # ── Shape statistics (optional) ─────────────────────────────────────
    if include_shape_stats:
        skew_vals = []
        kurt_vals = []

        for col in numeric_cols:
            s = num_df[col].dropna()
            if len(s) >= 3:
                skew_vals.append(skew(s, nan_policy="omit"))
                kurt_vals.append(kurtosis(s, nan_policy="omit"))
            else:
                skew_vals.append(np.nan)
                kurt_vals.append(np.nan)

        stats["skewness"] = skew_vals
        stats["kurtosis"] = kurt_vals

    # ── Value counts (vectorized) ───────────────────────────────────────
    stats["zero_count"] = (num_df == 0).sum().values
    stats["negative_count"] = (num_df < 0).sum().values

    # ── Rounding ────────────────────────────────────────────────────────
    float_cols = [
        "missing_pct", "min_value", "max_value", "range_value",
        "mean_value", "median_value", "std_dev", "coef_of_variation",
        p_low_label, p_high_label, "iqr",
    ]
    if include_shape_stats:
        float_cols.extend(["skewness", "kurtosis"])

    for col in float_cols:
        if col in stats.columns:
            stats[col] = stats[col].apply(lambda x: _safe_round(x, decimals))

    # ── Integer columns as nullable Int64 ───────────────────────────────
    int_cols = ["count", "missing_count", "zero_count", "negative_count"]
    for col in int_cols:
        stats[col] = stats[col].astype("Int64")

    # ── Sorting ─────────────────────────────────────────────────────────
    if sort_by and sort_by in stats.columns:
        stats = (
            stats
            .sort_values(sort_by, ascending=ascending, na_position="last")
            .reset_index(drop=True)
        )

    # ── Column ordering ─────────────────────────────────────────────────
    ordered_cols = [
        "column", "count", "missing_count", "missing_pct",
        "min_value", "max_value", "range_value",
        "mean_value", "median_value",
        "std_dev", "coef_of_variation",
        p_low_label, p_high_label, "iqr",
    ]
    if include_shape_stats:
        ordered_cols.extend(["skewness", "kurtosis"])
    ordered_cols.extend(["zero_count", "negative_count"])

    stats = stats[[c for c in ordered_cols if c in stats.columns]]

    return stats


# ── Lightweight alternative ─────────────────────────────────────────────────

def numerical_quick_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Minimal 5-number summary using pandas describe() — fastest option.

    Returns: column, count, mean, std, min, p25, median, p75, max
    """
    numeric_cols = _get_numeric_columns(df)

    if not numeric_cols:
        return pd.DataFrame()

    desc = (
        df[numeric_cols]
        .describe(percentiles=[0.25, 0.5, 0.75])
        .T
        .reset_index()
        .rename(columns={
            "index": "column",
            "mean": "mean_value",
            "std": "std_dev",
            "min": "min_value",
            "25%": "p25",
            "50%": "median_value",
            "75%": "p75",
            "max": "max_value",
        })
    )

    return desc

In [58]:
train_numerical_core_stats_report = numerical_core_stats_report(train_df)
train_numerical_core_stats_report

,column,count,missing_count,missing_pct,min_value,max_value,range_value,mean_value,median_value,std_dev,coef_of_variation,p25,p75,iqr,skewness,kurtosis,zero_count,negative_count
0,SK_ID_CURR,307511,0,0.0000,100002.0,4.562550e+05,3.562530e+05,278180.5186,278202.0,102790.1753,0.3695,189145.5,367142.5,177997.0,-0.0012,-1.1990,0,0
1,TARGET,307511,0,0.0000,0.0,1.000000e+00,1.000000e+00,0.0807,0.0,0.2724,3.3745,0.0,0.0,0.0,3.0781,7.4750,282686,0
2,CNT_CHILDREN,307511,0,0.0000,0.0,1.900000e+01,1.900000e+01,0.4171,0.0,0.7221,1.7315,0.0,1.0,1.0,1.9746,7.9040,215371,0
3,AMT_INCOME_TOTAL,307511,0,0.0000,25650.0,1.170000e+08,1.169744e+08,168797.9193,147150.0,237123.1463,1.4048,112500.0,202500.0,90000.0,391.5577,191783.4360,0,0
4,AMT_CREDIT,307511,0,0.0000,45000.0,4.050000e+06,4.005000e+06,599025.9997,513531.0,402490.7770,0.6719,270000.0,808650.0,538650.0,1.2348,1.9340,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,CC_NAME_CONTRACT_STATUS_Demand_MEAN,86905,220606,71.7392,0.0,9.158000e-01,9.158000e-01,0.0001,0.0,0.0087,82.7849,0.0,0.0,0.0,86.6597,7747.5395,86890,0
564,CC_NAME_CONTRACT_STATUS_Refused_MEAN,86905,220606,71.7392,0.0,1.390000e-02,1.390000e-02,0.0000,0.0,0.0002,78.9785,0.0,0.0,0.0,79.3629,6328.2206,86891,0
565,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,86905,220606,71.7392,0.0,2.440000e-02,2.440000e-02,0.0001,0.0,0.0009,13.8106,0.0,0.0,0.0,14.0322,201.3481,86445,0
566,CC_NAME_CONTRACT_STATUS_Signed_MEAN,86905,220606,71.7392,0.0,1.000000e+00,1.000000e+00,0.0037,0.0,0.0319,8.5286,0.0,0.0,0.0,18.6819,463.4523,82614,0


### 6.2 numerical_quantile_report
* q1 
* q3 
* iqr 

In [59]:
import pandas as pd
import numpy as np

def numerical_quantile_report(df: pd.DataFrame) -> pd.DataFrame:
    """
    Highly Optimized Vectorized Quantile & Outlier Report.
    
    Output columns:
    - column
    - q1, q3, iqr
    - lower_bound, upper_bound (Tukey's fences for outliers)
    - outlier_count, outlier_percentage
    """
    
    # 1. Efficiently select numeric columns and exclude booleans
    num_df = df.select_dtypes(include=['number']).select_dtypes(exclude=['bool'])
    
    if num_df.empty:
        return pd.DataFrame()

    # 2. VECTORIZATION (100x Faster): Calculate quantiles for ALL columns at once
    # .quantile([0.25, 0.75]) returns a DF where rows are 0.25/0.75 and cols are the features.
    # We transpose (.T) it so features are rows, which matches your desired output.
    quantiles = num_df.quantile([0.25, 0.75]).T
    quantiles.columns = ['q1', 'q3']
    
    # 3. Vectorized Math for IQR and Outlier Bounds (Tukey's Fences)
    quantiles['iqr'] = quantiles['q3'] - quantiles['q1']
    quantiles['lower_bound'] = quantiles['q1'] - 1.5 * quantiles['iqr']
    quantiles['upper_bound'] = quantiles['q3'] + 1.5 * quantiles['iqr']
    
    # 4. Outlier Detection (Actionable Logic)
    # Broadcast comparison across the entire dataframe at once
    outlier_mask = (num_df < quantiles['lower_bound']) | (num_df > quantiles['upper_bound'])
    quantiles['outlier_count'] = outlier_mask.sum()
    
    # Calculate percentage based on valid (non-NaN) observations
    valid_counts = num_df.notna().sum()
    quantiles['outlier_percentage'] = np.where(
        valid_counts > 0,
        (quantiles['outlier_count'] / valid_counts) * 100,
        0
    )
    
    # 5. Formatting & Cleanup
    quantiles = quantiles.reset_index().rename(columns={'index': 'column'})
    
    # Round all float columns cleanly
    cols_to_round = ['q1', 'q3', 'iqr', 'lower_bound', 'upper_bound', 'outlier_percentage']
    quantiles[cols_to_round] = quantiles[cols_to_round].round(4)
    
    return quantiles

In [60]:
train_numerical_quantile_report = numerical_quantile_report(train_df)
train_numerical_quantile_report

,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,SK_ID_CURR,189145.5,367142.5,177997.0,-77850.0,634138.0,0,0.0000
1,TARGET,0.0,0.0,0.0,0.0,0.0,24825,8.0729
2,CNT_CHILDREN,0.0,1.0,1.0,-1.5,2.5,4272,1.3892
3,AMT_INCOME_TOTAL,112500.0,202500.0,90000.0,-22500.0,337500.0,14035,4.5641
4,AMT_CREDIT,270000.0,808650.0,538650.0,-537975.0,1616625.0,6562,2.1339
...,...,...,...,...,...,...,...,...
563,CC_NAME_CONTRACT_STATUS_Demand_MEAN,0.0,0.0,0.0,0.0,0.0,15,0.0173
564,CC_NAME_CONTRACT_STATUS_Refused_MEAN,0.0,0.0,0.0,0.0,0.0,14,0.0161
565,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,0.0,0.0,0.0,0.0,0.0,460,0.5293
566,CC_NAME_CONTRACT_STATUS_Signed_MEAN,0.0,0.0,0.0,0.0,0.0,4291,4.9376


### 6.3 numerical_shape_report
* skewness 
* kurtosis


In [61]:
import warnings
from typing import Iterable, Optional, Sequence

import numpy as np
import pandas as pd

try:
    from scipy.stats import normaltest
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False


def numerical_audit_report(
    df: pd.DataFrame,
    include_columns: Optional[Sequence[str]] = None,
    exclude_columns: Optional[Sequence[str]] = None,
    round_digits: int = 4,
    outlier_method: str = "iqr",
    normality_alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Build a numerical audit report for numeric, non-boolean columns.

    Output columns
    --------------
    - column
    - dtype
    - non_null_count
    - missing_count
    - missing_rate
    - unique_count
    - mean
    - median
    - std
    - min
    - q1
    - q3
    - max
    - iqr
    - skewness
    - kurtosis
    - outlier_count
    - outlier_rate
    - skew_label
    - variance_flag
    - normality_pvalue
    - normality_label

    Notes
    -----
    - Only numeric columns are included
    - Boolean columns are excluded
    - Normality test uses scipy if available and enough samples exist
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")

    include_columns = list(include_columns) if include_columns is not None else list(df.columns)
    exclude_columns = set(exclude_columns or [])

    candidate_cols = [c for c in include_columns if c in df.columns and c not in exclude_columns]

    numeric_cols = [
        c for c in candidate_cols
        if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])
    ]

    if not numeric_cols:
        return pd.DataFrame(columns=[
            "column", "dtype", "non_null_count", "missing_count", "missing_rate",
            "unique_count", "mean", "median", "std", "min", "q1", "q3", "max",
            "iqr", "skewness", "kurtosis", "outlier_count", "outlier_rate",
            "skew_label", "variance_flag", "normality_pvalue", "normality_label"
        ])

    data = df[numeric_cols]

    non_null_count = data.notna().sum()
    missing_count = data.isna().sum()
    missing_rate = missing_count / len(df) if len(df) else np.nan
    unique_count = data.nunique(dropna=True)

    mean = data.mean()
    median = data.median()
    std = data.std()
    min_ = data.min()
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    max_ = data.max()
    iqr = q3 - q1
    skewness = data.skew()
    kurtosis = data.kurt()

    report = pd.DataFrame({
        "column": numeric_cols,
        "dtype": [str(df[c].dtype) for c in numeric_cols],
        "non_null_count": [non_null_count[c] for c in numeric_cols],
        "missing_count": [missing_count[c] for c in numeric_cols],
        "missing_rate": [missing_rate[c] for c in numeric_cols],
        "unique_count": [unique_count[c] for c in numeric_cols],
        "mean": [mean[c] for c in numeric_cols],
        "median": [median[c] for c in numeric_cols],
        "std": [std[c] for c in numeric_cols],
        "min": [min_[c] for c in numeric_cols],
        "q1": [q1[c] for c in numeric_cols],
        "q3": [q3[c] for c in numeric_cols],
        "max": [max_[c] for c in numeric_cols],
        "iqr": [iqr[c] for c in numeric_cols],
        "skewness": [skewness[c] for c in numeric_cols],
        "kurtosis": [kurtosis[c] for c in numeric_cols],
    })

    def compute_outlier_info(series: pd.Series, method: str = "iqr"):
        s = series.dropna()
        if len(s) == 0:
            return 0, np.nan

        if method == "iqr":
            q1_ = s.quantile(0.25)
            q3_ = s.quantile(0.75)
            iqr_ = q3_ - q1_
            if pd.isna(iqr_) or iqr_ == 0:
                return 0, 0.0
            lower = q1_ - 1.5 * iqr_
            upper = q3_ + 1.5 * iqr_
            count = ((s < lower) | (s > upper)).sum()
            return int(count), count / len(s)

        raise ValueError(f"Unsupported outlier_method: {method}")

    outlier_counts = []
    outlier_rates = []
    for col in numeric_cols:
        count, rate = compute_outlier_info(df[col], method=outlier_method)
        outlier_counts.append(count)
        outlier_rates.append(rate)

    report["outlier_count"] = outlier_counts
    report["outlier_rate"] = outlier_rates

    def skew_label(x):
        if pd.isna(x):
            return "unknown"
        ax = abs(x)
        if ax < 0.5:
            return "approximately_symmetric"
        if ax < 1.0:
            return "moderately_skewed"
        return "highly_skewed"

    def variance_flag(unique_cnt, std_val):
        if pd.isna(std_val):
            return "unknown"
        if unique_cnt <= 1:
            return "constant"
        if std_val == 0:
            return "constant"
        return "variable"

    report["skew_label"] = report["skewness"].apply(skew_label)
    report["variance_flag"] = [
        variance_flag(u, s) for u, s in zip(report["unique_count"], report["std"])
    ]

    normality_pvalues = []
    normality_labels = []

    for col in numeric_cols:
        s = df[col].dropna()

        if not SCIPY_AVAILABLE:
            normality_pvalues.append(np.nan)
            normality_labels.append("scipy_not_installed")
            continue

        if len(s) < 8:
            normality_pvalues.append(np.nan)
            normality_labels.append("insufficient_samples")
            continue

        if s.nunique() <= 1:
            normality_pvalues.append(np.nan)
            normality_labels.append("constant")
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                _, pvalue = normaltest(s)
            normality_pvalues.append(pvalue)
            normality_labels.append("likely_normal" if pvalue >= normality_alpha else "non_normal")
        except Exception:
            normality_pvalues.append(np.nan)
            normality_labels.append("test_failed")

    report["normality_pvalue"] = normality_pvalues
    report["normality_label"] = normality_labels

    numeric_round_cols = [
        "missing_rate", "mean", "median", "std", "min", "q1", "q3", "max",
        "iqr", "skewness", "kurtosis", "outlier_rate", "normality_pvalue"
    ]
    existing_round_cols = [c for c in numeric_round_cols if c in report.columns]
    report[existing_round_cols] = report[existing_round_cols].round(round_digits)

    return report

In [62]:
train_numerical_shape_report = numerical_audit_report(train_df)
train_numerical_shape_report

,column,dtype,non_null_count,missing_count,missing_rate,unique_count,mean,median,std,min,...,max,iqr,skewness,kurtosis,outlier_count,outlier_rate,skew_label,variance_flag,normality_pvalue,normality_label
0,SK_ID_CURR,int64,307511,0,0.0000,307511,278180.5186,278202.0,102790.1753,100002.0,...,4.562550e+05,177997.0,-0.0012,-1.1990,0,0.0000,approximately_symmetric,variable,0.0,non_normal
1,TARGET,int64,307511,0,0.0000,2,0.0807,0.0,0.2724,0.0,...,1.000000e+00,0.0,3.0782,7.4751,0,0.0000,highly_skewed,variable,0.0,non_normal
2,CNT_CHILDREN,int64,307511,0,0.0000,15,0.4171,0.0,0.7221,0.0,...,1.900000e+01,1.0,1.9746,7.9041,4272,0.0139,highly_skewed,variable,0.0,non_normal
3,AMT_INCOME_TOTAL,float64,307511,0,0.0000,2548,168797.9193,147150.0,237123.1463,25650.0,...,1.170000e+08,90000.0,391.5597,191786.5544,14035,0.0456,highly_skewed,variable,0.0,non_normal
4,AMT_CREDIT,float64,307511,0,0.0000,5603,599025.9997,513531.0,402490.7770,45000.0,...,4.050000e+06,538650.0,1.2348,1.9340,6562,0.0213,highly_skewed,variable,0.0,non_normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,CC_NAME_CONTRACT_STATUS_Demand_MEAN,float64,86905,220606,0.7174,16,0.0001,0.0,0.0087,0.0,...,9.158000e-01,0.0,86.6612,7747.9854,0,0.0000,highly_skewed,variable,0.0,non_normal
564,CC_NAME_CONTRACT_STATUS_Refused_MEAN,float64,86905,220606,0.7174,14,0.0000,0.0,0.0002,0.0,...,1.390000e-02,0.0,79.3643,6328.5848,0,0.0000,highly_skewed,variable,0.0,non_normal
565,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,float64,86905,220606,0.7174,34,0.0001,0.0,0.0009,0.0,...,2.440000e-02,0.0,14.0324,201.3598,0,0.0000,highly_skewed,variable,0.0,non_normal
566,CC_NAME_CONTRACT_STATUS_Signed_MEAN,float64,86905,220606,0.7174,252,0.0037,0.0,0.0319,0.0,...,1.000000e+00,0.0,18.6822,463.4790,0,0.0000,highly_skewed,variable,0.0,non_normal


In [63]:
import pandas as pd
from functools import reduce

def merge_reports_on_column(reports, key="column", how="outer"):
    """
    Merge multiple report DataFrames on a shared key column.
    """
    if not reports:
        return pd.DataFrame()

    validated_reports = []
    for i, df in enumerate(reports, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Item {i} is not a pandas DataFrame")
        if key not in df.columns:
            raise ValueError(f"DataFrame {i} is missing required key column: '{key}'")
        validated_reports.append(df)

    return reduce(lambda left, right: left.merge(right, on=key, how=how), validated_reports)

numerical_feature_profile_report = merge_reports_on_column([
    train_numerical_core_stats_report,
    train_numerical_quantile_report,
    train_numerical_shape_report
])

In [64]:
numerical_feature_profile_report.head()

,column,count,missing_count_x,missing_pct,min_value,max_value,range_value,mean_value,median_value,std_dev,...,max,iqr,skewness_y,kurtosis_y,outlier_count_y,outlier_rate,skew_label,variance_flag,normality_pvalue,normality_label
0,AMT_ANNUITY,307499,12,0.0039,1615.5,258025.5,256410.0,27108.5739,24903.0,14493.7373,...,258025.5,18072.0,1.5798,7.7073,7504,0.0244,highly_skewed,variable,0.0,non_normal
1,AMT_CREDIT,307511,0,0.0000,45000.0,4050000.0,4005000.0,599025.9997,513531.0,402490.7770,...,4050000.0,538650.0,1.2348,1.9340,6562,0.0213,highly_skewed,variable,0.0,non_normal
2,AMT_GOODS_PRICE,307233,278,0.0904,40500.0,4050000.0,4009500.0,538396.2074,450000.0,369446.4605,...,4050000.0,441000.0,1.3490,2.4319,14728,0.0479,highly_skewed,variable,0.0,non_normal
3,AMT_INCOME_TOTAL,307511,0,0.0000,25650.0,117000000.0,116974350.0,168797.9193,147150.0,237123.1463,...,117000000.0,90000.0,391.5597,191786.5544,14035,0.0456,highly_skewed,variable,0.0,non_normal
4,AMT_REQ_CREDIT_BUREAU_DAY,265992,41519,13.5016,0.0,9.0,9.0,0.0070,0.0,0.1108,...,9.0,0.0,27.0435,1151.8676,0,0.0000,highly_skewed,variable,0.0,non_normal


In [65]:
numerical_feature_profile_report.to_csv(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\6.numerical_feature_report.csv")

# 7. Numerical Robustness / Transformation

### 7.1 numerical_outlier_report
* outlier_count
* outlier_percentage
* heavy_tail_flag

In [66]:
import warnings
from typing import Literal

import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, zscore
from scipy.stats import median_abs_deviation


# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def _safe_round(value, decimals: int = 4):
    """Round finite values; return NaN otherwise."""
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), decimals)


def _get_numeric_columns(df: pd.DataFrame) -> list[str]:
    """Return numeric (non-boolean) column names."""
    return [
        col for col in df.columns
        if pd.api.types.is_numeric_dtype(df[col])
        and not pd.api.types.is_bool_dtype(df[col])
    ]


def _calculate_iqr_bounds(
    s: pd.Series,
    multiplier: float = 1.5,
) -> tuple[float, float, float, float]:
    """
    Calculate IQR-based outlier bounds.
    
    Returns: (q1, q3, lower_bound, upper_bound)
    """
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    
    lower_bound = q1 - multiplier * iqr
    upper_bound = q3 + multiplier * iqr
    
    return q1, q3, lower_bound, upper_bound


def _count_iqr_outliers(
    s: pd.Series,
    multiplier: float = 1.5,
) -> dict:
    """
    Count outliers using IQR method with detailed breakdown.
    
    Returns dict with counts, bounds, and percentages.
    """
    if len(s) == 0:
        return {
            "lower_bound": np.nan,
            "upper_bound": np.nan,
            "lower_count": 0,
            "upper_count": 0,
            "total_count": 0,
            "lower_pct": np.nan,
            "upper_pct": np.nan,
            "total_pct": np.nan,
        }
    
    q1, q3, lower_bound, upper_bound = _calculate_iqr_bounds(s, multiplier)
    
    lower_outliers = s < lower_bound
    upper_outliers = s > upper_bound
    
    n = len(s)
    lower_count = int(lower_outliers.sum())
    upper_count = int(upper_outliers.sum())
    total_count = lower_count + upper_count
    
    return {
        "q1": q1,
        "q3": q3,
        "iqr": q3 - q1,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "lower_count": lower_count,
        "upper_count": upper_count,
        "total_count": total_count,
        "lower_pct": (lower_count / n) * 100,
        "upper_pct": (upper_count / n) * 100,
        "total_pct": (total_count / n) * 100,
    }


def _count_zscore_outliers(
    s: pd.Series,
    threshold: float = 3.0,
) -> dict:
    """
    Count outliers using Z-score method.
    
    Returns dict with counts and percentages.
    """
    if len(s) < 2:
        return {
            "lower_count": 0,
            "upper_count": 0,
            "total_count": 0,
            "total_pct": np.nan,
        }
    
    mean = s.mean()
    std = s.std()
    
    if std == 0 or pd.isna(std):
        return {
            "lower_count": 0,
            "upper_count": 0,
            "total_count": 0,
            "total_pct": 0.0,
        }
    
    z_scores = (s - mean) / std
    
    lower_outliers = z_scores < -threshold
    upper_outliers = z_scores > threshold
    
    n = len(s)
    lower_count = int(lower_outliers.sum())
    upper_count = int(upper_outliers.sum())
    total_count = lower_count + upper_count
    
    return {
        "lower_bound": mean - threshold * std,
        "upper_bound": mean + threshold * std,
        "lower_count": lower_count,
        "upper_count": upper_count,
        "total_count": total_count,
        "total_pct": (total_count / n) * 100,
    }


def _count_modified_zscore_outliers(
    s: pd.Series,
    threshold: float = 3.5,
) -> dict:
    """
    Count outliers using Modified Z-score (MAD-based).
    
    More robust than standard Z-score as MAD is resistant to outliers.
    
    Modified Z-score = 0.6745 * (x - median) / MAD
    
    Threshold of 3.5 is recommended by Iglewicz and Hoaglin (1993).
    """
    if len(s) < 2:
        return {
            "total_count": 0,
            "total_pct": np.nan,
        }
    
    median = s.median()
    mad = median_abs_deviation(s, nan_policy="omit")
    
    if mad == 0 or pd.isna(mad):
        # Fall back to standard deviation if MAD is zero
        mad = s.std() * 0.6745  # Approximate MAD from std for normal distribution
        if mad == 0 or pd.isna(mad):
            return {
                "total_count": 0,
                "total_pct": 0.0,
            }
    
    # Modified Z-score formula
    modified_z = 0.6745 * (s - median) / mad
    
    outliers = np.abs(modified_z) > threshold
    
    n = len(s)
    total_count = int(outliers.sum())
    
    return {
        "median": median,
        "mad": mad,
        "total_count": total_count,
        "total_pct": (total_count / n) * 100,
    }


def _count_percentile_outliers(
    s: pd.Series,
    lower_percentile: float = 0.01,
    upper_percentile: float = 0.99,
) -> dict:
    """
    Count outliers using percentile-based method.
    
    Useful for heavily skewed distributions.
    """
    if len(s) == 0:
        return {
            "lower_bound": np.nan,
            "upper_bound": np.nan,
            "total_count": 0,
            "total_pct": np.nan,
        }
    
    lower_bound = s.quantile(lower_percentile)
    upper_bound = s.quantile(upper_percentile)
    
    outliers = (s < lower_bound) | (s > upper_bound)
    
    n = len(s)
    total_count = int(outliers.sum())
    
    return {
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "total_count": total_count,
        "total_pct": (total_count / n) * 100,
    }


def _detect_distribution_characteristics(s: pd.Series) -> dict:
    """
    Detect distribution characteristics for outlier context.
    """
    if len(s) < 4:
        return {
            "skewness": np.nan,
            "kurtosis": np.nan,
            "distribution_type": "insufficient_data",
            "is_heavy_tailed": False,
            "is_skewed": False,
        }
    
    try:
        sk = skew(s, nan_policy="omit")
        kt = kurtosis(s, nan_policy="omit")  # Fisher (excess) kurtosis
    except Exception:
        return {
            "skewness": np.nan,
            "kurtosis": np.nan,
            "distribution_type": "unknown",
            "is_heavy_tailed": False,
            "is_skewed": False,
        }
    
    # Heavy-tailed: excess kurtosis > 1
    is_heavy_tailed = kt > 1
    
    # Skewed: |skewness| > 0.5
    is_skewed = abs(sk) > 0.5
    
    # Distribution type
    if abs(sk) < 0.5 and abs(kt) < 1:
        dist_type = "approximately_normal"
    elif sk > 1:
        dist_type = "right_skewed"
    elif sk < -1:
        dist_type = "left_skewed"
    elif kt > 3:
        dist_type = "heavy_tailed"
    elif kt < -1:
        dist_type = "light_tailed"
    else:
        dist_type = "moderate_deviation"
    
    return {
        "skewness": sk,
        "kurtosis": kt,
        "distribution_type": dist_type,
        "is_heavy_tailed": is_heavy_tailed,
        "is_skewed": is_skewed,
    }


def _suggest_outlier_treatment(
    outlier_pct: float,
    skewness: float,
    is_heavy_tailed: bool,
    has_lower_outliers: bool,
    has_upper_outliers: bool,
) -> str:
    """
    Suggest appropriate outlier treatment strategy.
    """
    if pd.isna(outlier_pct) or outlier_pct == 0:
        return "none_needed"
    
    if outlier_pct < 1:
        return "investigate_individually"
    
    if outlier_pct > 10:
        # Too many "outliers" — might be genuine distribution
        if is_heavy_tailed:
            return "consider_log_transform"
        return "review_data_source"
    
    if is_heavy_tailed and outlier_pct > 5:
        return "robust_scaling_or_transform"
    
    if has_lower_outliers and has_upper_outliers:
        return "winsorize_both_tails"
    elif has_lower_outliers:
        return "winsorize_lower_or_clip"
    elif has_upper_outliers:
        if not pd.isna(skewness) and skewness > 1:
            return "log_transform_or_winsorize"
        return "winsorize_upper_or_clip"
    
    return "winsorize_or_remove"


def _get_outlier_sample_values(
    s: pd.Series,
    lower_bound: float,
    upper_bound: float,
    max_samples: int = 5,
) -> dict:
    """
    Get sample outlier values for inspection.
    """
    lower_outliers = s[s < lower_bound].head(max_samples).tolist()
    upper_outliers = s[s > upper_bound].head(max_samples).tolist()
    
    return {
        "sample_lower_outliers": lower_outliers if lower_outliers else None,
        "sample_upper_outliers": upper_outliers if upper_outliers else None,
    }


# ══════════════════════════════════════════════════════════════════════════════
# MAIN REPORT FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def numerical_outlier_report(
    df: pd.DataFrame,
    # IQR parameters
    outlier_iqr_multiplier: float = 1.5,
    extreme_iqr_multiplier: float = 3.0,
    # Z-score parameters
    zscore_threshold: float = 3.0,
    modified_zscore_threshold: float = 3.5,
    # Percentile parameters
    percentile_lower: float = 0.01,
    percentile_upper: float = 0.99,
    # Heavy tail detection
    heavy_tail_outlier_pct_threshold: float = 5.0,
    heavy_tail_kurtosis_threshold: float = 3.0,
    # Feature flags
    include_zscore: bool = True,
    include_modified_zscore: bool = True,
    include_percentile: bool = True,
    include_extreme_outliers: bool = True,
    include_distribution_info: bool = True,
    include_recommendations: bool = True,
    include_sample_values: bool = False,
    # Output options
    sort_by: str | None = "iqr_total_pct",
    ascending: bool = False,
    decimals: int = 2,
) -> pd.DataFrame:
    """
    Comprehensive numerical outlier detection report using multiple methods.

    Detection Methods
    -----------------
    1. IQR (Interquartile Range): Q1 - k*IQR to Q3 + k*IQR
    2. Z-score: |z| > threshold (typically 3)
    3. Modified Z-score (MAD-based): Robust to existing outliers
    4. Percentile: Values below p1 or above p99
    5. Extreme IQR: 3×IQR for severe outliers

    Output Columns
    --------------
    Basic Info:
        column, count, missing_count
    
    IQR Method:
        iqr_q1, iqr_q3, iqr_value, iqr_lower_bound, iqr_upper_bound,
        iqr_lower_count, iqr_upper_count, iqr_total_count, iqr_total_pct
    
    Extreme IQR (optional):
        extreme_iqr_count, extreme_iqr_pct
    
    Z-score Method (optional):
        zscore_lower_count, zscore_upper_count, zscore_total_count, zscore_total_pct
    
    Modified Z-score (optional):
        mod_zscore_count, mod_zscore_pct
    
    Percentile Method (optional):
        pctl_lower_bound, pctl_upper_bound, pctl_count, pctl_pct
    
    Distribution Info (optional):
        skewness, kurtosis, distribution_type, is_heavy_tailed, is_skewed
    
    Flags:
        heavy_tail_flag, has_outliers, outlier_severity
    
    Recommendations (optional):
        suggested_treatment, winsorize_lower, winsorize_upper
    
    Sample Values (optional):
        sample_lower_outliers, sample_upper_outliers

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame
    outlier_iqr_multiplier : float
        IQR multiplier for standard outliers (default: 1.5)
    extreme_iqr_multiplier : float
        IQR multiplier for extreme outliers (default: 3.0)
    zscore_threshold : float
        Z-score threshold (default: 3.0)
    modified_zscore_threshold : float
        Modified Z-score threshold (default: 3.5)
    percentile_lower : float
        Lower percentile for percentile method (default: 0.01)
    percentile_upper : float
        Upper percentile for percentile method (default: 0.99)
    heavy_tail_outlier_pct_threshold : float
        Outlier % to flag heavy tail (default: 5.0)
    heavy_tail_kurtosis_threshold : float
        Kurtosis threshold for heavy tail (default: 3.0)
    include_* : bool
        Feature toggles for optional analyses
    sort_by : str | None
        Column to sort by
    ascending : bool
        Sort order
    decimals : int
        Rounding precision

    Returns
    -------
    pd.DataFrame
        Comprehensive outlier report
    """
    # ── Validation ──────────────────────────────────────────────────────
    if df.empty:
        warnings.warn("Empty DataFrame provided.", UserWarning)
        return pd.DataFrame()

    numeric_cols = _get_numeric_columns(df)

    if not numeric_cols:
        warnings.warn("No numeric columns found in DataFrame.", UserWarning)
        return pd.DataFrame()

    # ── Build report ────────────────────────────────────────────────────
    records = []

    for col in numeric_cols:
        s = df[col].dropna()
        n = len(s)
        missing = df[col].isna().sum()
        
        record = {
            "column": col,
            "count": n,
            "missing_count": missing,
        }

        # ════════════════════════════════════════════════════════════════
        # IQR METHOD (always included)
        # ════════════════════════════════════════════════════════════════
        
        iqr_results = _count_iqr_outliers(s, multiplier=outlier_iqr_multiplier)
        
        record.update({
            "iqr_q1": _safe_round(iqr_results.get("q1"), decimals),
            "iqr_q3": _safe_round(iqr_results.get("q3"), decimals),
            "iqr_value": _safe_round(iqr_results.get("iqr"), decimals),
            "iqr_lower_bound": _safe_round(iqr_results["lower_bound"], decimals),
            "iqr_upper_bound": _safe_round(iqr_results["upper_bound"], decimals),
            "iqr_lower_count": iqr_results["lower_count"],
            "iqr_upper_count": iqr_results["upper_count"],
            "iqr_total_count": iqr_results["total_count"],
            "iqr_total_pct": _safe_round(iqr_results["total_pct"], decimals),
        })

        # ════════════════════════════════════════════════════════════════
        # EXTREME IQR (3×IQR)
        # ════════════════════════════════════════════════════════════════
        
        if include_extreme_outliers:
            extreme_results = _count_iqr_outliers(s, multiplier=extreme_iqr_multiplier)
            record.update({
                "extreme_iqr_count": extreme_results["total_count"],
                "extreme_iqr_pct": _safe_round(extreme_results["total_pct"], decimals),
            })

        # ════════════════════════════════════════════════════════════════
        # Z-SCORE METHOD
        # ════════════════════════════════════════════════════════════════
        
        if include_zscore:
            zscore_results = _count_zscore_outliers(s, threshold=zscore_threshold)
            record.update({
                "zscore_lower_bound": _safe_round(zscore_results.get("lower_bound"), decimals),
                "zscore_upper_bound": _safe_round(zscore_results.get("upper_bound"), decimals),
                "zscore_lower_count": zscore_results["lower_count"],
                "zscore_upper_count": zscore_results["upper_count"],
                "zscore_total_count": zscore_results["total_count"],
                "zscore_total_pct": _safe_round(zscore_results["total_pct"], decimals),
            })

        # ════════════════════════════════════════════════════════════════
        # MODIFIED Z-SCORE (MAD-based)
        # ════════════════════════════════════════════════════════════════
        
        if include_modified_zscore:
            mod_z_results = _count_modified_zscore_outliers(
                s, threshold=modified_zscore_threshold
            )
            record.update({
                "mod_zscore_mad": _safe_round(mod_z_results.get("mad"), decimals),
                "mod_zscore_count": mod_z_results["total_count"],
                "mod_zscore_pct": _safe_round(mod_z_results["total_pct"], decimals),
            })

        # ════════════════════════════════════════════════════════════════
        # PERCENTILE METHOD
        # ════════════════════════════════════════════════════════════════
        
        if include_percentile:
            pctl_results = _count_percentile_outliers(
                s, lower_percentile=percentile_lower, upper_percentile=percentile_upper
            )
            record.update({
                "pctl_lower_bound": _safe_round(pctl_results["lower_bound"], decimals),
                "pctl_upper_bound": _safe_round(pctl_results["upper_bound"], decimals),
                "pctl_count": pctl_results["total_count"],
                "pctl_pct": _safe_round(pctl_results["total_pct"], decimals),
            })

        # ════════════════════════════════════════════════════════════════
        # DISTRIBUTION CHARACTERISTICS
        # ════════════════════════════════════════════════════════════════
        
        if include_distribution_info:
            dist_info = _detect_distribution_characteristics(s)
            record.update({
                "skewness": _safe_round(dist_info["skewness"], decimals),
                "kurtosis": _safe_round(dist_info["kurtosis"], decimals),
                "distribution_type": dist_info["distribution_type"],
                "is_heavy_tailed": dist_info["is_heavy_tailed"],
                "is_skewed": dist_info["is_skewed"],
            })
            
            skewness = dist_info["skewness"]
            kurtosis_val = dist_info["kurtosis"]
            is_heavy_tailed = dist_info["is_heavy_tailed"]
        else:
            skewness = np.nan
            kurtosis_val = np.nan
            is_heavy_tailed = False

        # ════════════════════════════════════════════════════════════════
        # FLAGS & SEVERITY
        # ════════════════════════════════════════════════════════════════
        
        outlier_pct = iqr_results["total_pct"]
        
        # Heavy tail flag (original logic + improvements)
        heavy_tail_flag = (
            (pd.notna(outlier_pct) and outlier_pct >= heavy_tail_outlier_pct_threshold)
            or (pd.notna(kurtosis_val) and kurtosis_val >= heavy_tail_kurtosis_threshold)
            or is_heavy_tailed
        )
        
        # Has outliers
        has_outliers = iqr_results["total_count"] > 0
        
        # Severity classification
        if pd.isna(outlier_pct) or outlier_pct == 0:
            severity = "none"
        elif outlier_pct < 1:
            severity = "low"
        elif outlier_pct < 5:
            severity = "moderate"
        elif outlier_pct < 10:
            severity = "high"
        else:
            severity = "severe"
        
        record.update({
            "heavy_tail_flag": heavy_tail_flag,
            "has_outliers": has_outliers,
            "outlier_severity": severity,
        })

        # ════════════════════════════════════════════════════════════════
        # RECOMMENDATIONS
        # ════════════════════════════════════════════════════════════════
        
        if include_recommendations:
            suggested = _suggest_outlier_treatment(
                outlier_pct=outlier_pct,
                skewness=skewness,
                is_heavy_tailed=is_heavy_tailed,
                has_lower_outliers=iqr_results["lower_count"] > 0,
                has_upper_outliers=iqr_results["upper_count"] > 0,
            )
            
            record.update({
                "suggested_treatment": suggested,
                "winsorize_lower": _safe_round(iqr_results["lower_bound"], decimals),
                "winsorize_upper": _safe_round(iqr_results["upper_bound"], decimals),
            })

        # ════════════════════════════════════════════════════════════════
        # SAMPLE VALUES
        # ════════════════════════════════════════════════════════════════
        
        if include_sample_values and n > 0:
            samples = _get_outlier_sample_values(
                s,
                iqr_results["lower_bound"],
                iqr_results["upper_bound"],
                max_samples=5,
            )
            record.update(samples)

        records.append(record)

    # ── Build DataFrame ─────────────────────────────────────────────────
    result = pd.DataFrame(records)

    # ── Integer columns ─────────────────────────────────────────────────
    int_cols = [
        "count", "missing_count",
        "iqr_lower_count", "iqr_upper_count", "iqr_total_count",
    ]
    
    if include_extreme_outliers:
        int_cols.append("extreme_iqr_count")
    if include_zscore:
        int_cols.extend(["zscore_lower_count", "zscore_upper_count", "zscore_total_count"])
    if include_modified_zscore:
        int_cols.append("mod_zscore_count")
    if include_percentile:
        int_cols.append("pctl_count")

    for col in int_cols:
        if col in result.columns:
            result[col] = result[col].astype("Int64")

    # ── Sorting ─────────────────────────────────────────────────────────
    if sort_by and sort_by in result.columns:
        result = (
            result
            .sort_values(sort_by, ascending=ascending, na_position="last")
            .reset_index(drop=True)
        )

    return result


# ══════════════════════════════════════════════════════════════════════════════
# QUICK SUMMARY FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def numerical_outlier_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Quick outlier summary — just the essentials.
    
    Returns: column, iqr_total_count, iqr_total_pct, severity, heavy_tail_flag
    """
    report = numerical_outlier_report(
        df,
        include_zscore=False,
        include_modified_zscore=False,
        include_percentile=False,
        include_extreme_outliers=False,
        include_distribution_info=False,
        include_recommendations=True,
        include_sample_values=False,
    )
    
    if report.empty:
        return report
    
    key_cols = [
        "column", "count",
        "iqr_total_count", "iqr_total_pct",
        "outlier_severity", "heavy_tail_flag",
        "suggested_treatment",
    ]
    
    return report[[c for c in key_cols if c in report.columns]]


def get_outlier_columns(
    df: pd.DataFrame,
    min_outlier_pct: float = 1.0,
    method: Literal["iqr", "zscore", "modified_zscore"] = "iqr",
) -> list[str]:
    """
    Get list of columns with outliers above threshold.
    
    Useful for quick filtering before detailed analysis.
    """
    report = numerical_outlier_report(
        df,
        include_zscore=(method == "zscore"),
        include_modified_zscore=(method == "modified_zscore"),
        include_percentile=False,
        include_extreme_outliers=False,
        include_distribution_info=False,
        include_recommendations=False,
    )
    
    if report.empty:
        return []
    
    pct_col = f"{method}_total_pct" if method != "modified_zscore" else "mod_zscore_pct"
    
    if pct_col not in report.columns:
        pct_col = "iqr_total_pct"
    
    mask = report[pct_col] >= min_outlier_pct
    return report.loc[mask, "column"].tolist()


def compare_outlier_methods(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compare outlier counts across all detection methods.
    
    Useful for understanding which method is most appropriate.
    """
    report = numerical_outlier_report(
        df,
        include_zscore=True,
        include_modified_zscore=True,
        include_percentile=True,
        include_extreme_outliers=True,
        include_distribution_info=True,
        include_recommendations=False,
        include_sample_values=False,
    )
    
    if report.empty:
        return report
    
    comparison_cols = [
        "column", "count", "distribution_type",
        "iqr_total_count", "iqr_total_pct",
        "extreme_iqr_count", "extreme_iqr_pct",
        "zscore_total_count", "zscore_total_pct",
        "mod_zscore_count", "mod_zscore_pct",
        "pctl_count", "pctl_pct",
        "is_heavy_tailed", "is_skewed",
    ]
    
    return report[[c for c in comparison_cols if c in report.columns]]

In [67]:
train_numerical_outlier_report = numerical_outlier_report(train_df)
train_numerical_outlier_report

,column,count,missing_count,iqr_q1,iqr_q3,iqr_value,iqr_lower_bound,iqr_upper_bound,iqr_lower_count,iqr_upper_count,...,kurtosis,distribution_type,is_heavy_tailed,is_skewed,heavy_tail_flag,has_outliers,outlier_severity,suggested_treatment,winsorize_lower,winsorize_upper
0,INS_LAST365_PAY_DIFF_MEAN,217478,90033,0.0,0.0,0.0,0.00,0.00,20997,43604,...,200.58,left_skewed,True,True,True,True,severe,consider_log_transform,0.00,0.00
1,INS_LAST365_PAY_DIFF_SUM,217508,90003,0.0,0.0,0.0,0.00,0.00,20997,43604,...,68.81,left_skewed,True,True,True,True,severe,consider_log_transform,0.00,0.00
2,INS_LAST365_PAY_PERC_MEAN,217478,90033,1.0,1.0,0.0,1.00,1.00,43329,21272,...,77662.38,right_skewed,True,True,True,True,severe,consider_log_transform,1.00,1.00
3,REGION_RATING_CLIENT,307511,0,2.0,2.0,0.0,2.00,2.00,32197,48330,...,0.80,approximately_normal,False,False,True,True,severe,review_data_source,2.00,2.00
4,REGION_RATING_CLIENT_W_CITY,307511,0,2.0,2.0,0.0,2.00,2.00,34167,43860,...,0.93,approximately_normal,False,False,True,True,severe,review_data_source,2.00,2.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,PREV_NAME_CONTRACT_STATUS_Approved_MEAN,291057,16454,0.5,1.0,0.5,-0.25,1.75,0,0,...,-0.88,moderate_deviation,False,True,False,False,none,none_needed,-0.25,1.75
564,FLAG_DOCUMENT_2,307511,0,0.0,0.0,0.0,0.00,0.00,0,13,...,23649.69,right_skewed,True,True,True,True,low,investigate_individually,0.00,0.00
565,FLAG_DOCUMENT_3,307511,0,0.0,1.0,1.0,-1.50,2.50,0,0,...,-1.14,light_tailed,False,True,False,False,none,none_needed,-1.50,2.50
566,SK_ID_CURR,307511,0,189145.5,367142.5,177997.0,-77850.00,634138.00,0,0,...,-1.20,light_tailed,False,False,False,False,none,none_needed,-77850.00,634138.00


### 7.2 numerical_transformation_report
* distribution_type
* transformation_recommendation
* scaling_needed
* clipping_recommendation
* winsorization_recommendation

In [68]:
import pandas as pd
import numpy as np

def numerical_transformation_report(
    df: pd.DataFrame,
    skew_threshold: float = 1.0,
    moderate_skew_threshold: float = 0.5,
    heavy_tail_kurtosis_threshold: float = 3.0, # Note: Pandas kurtosis is Excess Kurtosis
    outlier_pct_clip_threshold: float = 5.0,
    outlier_pct_winsor_threshold: float = 2.0,
    zero_inflation_threshold: float = 0.60,
    cv_scaling_threshold: float = 1.0
) -> pd.DataFrame:
    """
    Highly Optimized, Vectorized Numerical Transformation Report.
    
    Output columns:
    - column
    - distribution_type
    - transformation_recommendation (Integrated with scikit-learn terminology)
    - scaling_needed
    - clipping_recommendation
    - winsorization_recommendation
    """

    # 1. Efficiently select numeric columns
    num_df = df.select_dtypes(include=['number']).select_dtypes(exclude=['bool'])
    
    if num_df.empty:
        return pd.DataFrame()

    # 2. VECTORIZED STATS CALCULATION (100x Faster)
    # Calculate all base metrics for all columns at once
    n_valid = num_df.notna().sum()
    
    stats = pd.DataFrame({
        'nunique': num_df.nunique(),
        'min': num_df.min(),
        'max': num_df.max(),
        'mean': num_df.mean(),
        'std': num_df.std(),
        'skew': num_df.skew(),
        'kurt': num_df.kurt(), # Fisher's definition (Normal dist = 0)
        'zeros': (num_df == 0).sum(),
        'q1': num_df.quantile(0.25),
        'q3': num_df.quantile(0.75)
    })
    
    # 3. Vectorized Math for Derived Metrics
    stats['iqr'] = stats['q3'] - stats['q1']
    stats['lower_bound'] = stats['q1'] - 1.5 * stats['iqr']
    stats['upper_bound'] = stats['q3'] + 1.5 * stats['iqr']
    
    # Broadcast comparison over entire dataframe to find outliers
    outlier_mask = (num_df < stats['lower_bound']) | (num_df > stats['upper_bound'])
    stats['outlier_pct'] = np.where(n_valid > 0, (outlier_mask.sum() / n_valid) * 100, 0)
    
    stats['zero_ratio'] = np.where(n_valid > 0, stats['zeros'] / n_valid, 0)
    
    # Safe Coefficient of Variation (CV)
    stats['cv'] = np.where((stats['mean'].notna()) & (stats['mean'] != 0), 
                           (stats['std'] / stats['mean']).abs(), 
                           np.inf)
    stats['range'] = stats['max'] - stats['min']

    # 4. LOGIC ENGINE (Vectorized AI-like rules using np.select)
    
    # --- Distribution Type ---
    dist_conditions = [
        stats['nunique'] <= 1,
        stats['zero_ratio'] >= zero_inflation_threshold,
        (stats['kurt'].notna()) & (stats['kurt'] >= heavy_tail_kurtosis_threshold),
        (stats['skew'].notna()) & (stats['skew'] >= skew_threshold),
        (stats['skew'].notna()) & (stats['skew'] <= -skew_threshold)
    ]
    dist_choices = [
        "constant_or_near_constant",
        "zero_inflated",
        "heavy_tailed",
        "right_skewed",
        "left_skewed"
    ]
    stats['distribution_type'] = np.select(dist_conditions, dist_choices, default="approximately_normal")

    # --- Transformation Recommendation ---
    trans_conditions = [
        stats['distribution_type'] == "constant_or_near_constant",
        
        # Zero Inflated
        (stats['distribution_type'] == "zero_inflated") & (stats['min'] >= 0),
        (stats['distribution_type'] == "zero_inflated") & (stats['min'] < 0),
        
        # Right Skewed
        (stats['distribution_type'] == "right_skewed") & (stats['min'] >= 0),
        (stats['distribution_type'] == "right_skewed") & (stats['min'] < 0),
        
        # Left Skewed (Upgraded Logic)
        (stats['distribution_type'] == "left_skewed"),
        
        # Heavy Tailed
        (stats['distribution_type'] == "heavy_tailed") & (stats['min'] >= 0) & (stats['skew'] > moderate_skew_threshold),
        (stats['distribution_type'] == "heavy_tailed")
    ]
    trans_choices = [
        "drop_column",
        "log1p_or_tweedie", # Standard for ML models dealing with zero-inflation
        "yeo_johnson_power_transform",
        "log1p",
        "yeo_johnson_power_transform",
        "yeo_johnson_or_square_transform", # Upgraded from "review_manually"
        "log1p_or_quantile_transform", # QuantileTransformer is Sklearn's best tool for extreme tails
        "quantile_transform_or_robust_scaler" 
    ]
    stats['transformation_recommendation'] = np.select(trans_conditions, trans_choices, default="none_or_standard_scaler")

    # --- Scaling Needed ---
    stats['scaling_needed'] = (
        (stats['cv'].notna() & (stats['cv'] >= cv_scaling_threshold)) |
        (stats['range'].notna() & (stats['range'] > 100)) |
        (stats['distribution_type'].isin(["heavy_tailed", "right_skewed", "left_skewed"]))
    )

    # --- Clipping & Winsorization ---
    stats['clipping_recommendation'] = (
        stats['outlier_pct'].notna() & 
        (stats['outlier_pct'] >= outlier_pct_clip_threshold)
    )

    stats['winsorization_recommendation'] = (
        stats['outlier_pct'].notna() & 
        (stats['outlier_pct'] >= outlier_pct_winsor_threshold) & 
        (stats['outlier_pct'] < outlier_pct_clip_threshold)
    )

    # 5. Format Output to match requested structure exactly
    report_df = stats[[
        'distribution_type', 
        'transformation_recommendation', 
        'scaling_needed', 
        'clipping_recommendation', 
        'winsorization_recommendation'
    ]].copy()
    
    report_df = report_df.reset_index().rename(columns={'index': 'column'})

    return report_df

In [69]:
train_numerical_transformation_report = numerical_transformation_report(train_df)
train_numerical_transformation_report

,column,distribution_type,transformation_recommendation,scaling_needed,clipping_recommendation,winsorization_recommendation
0,SK_ID_CURR,approximately_normal,none_or_standard_scaler,True,False,False
1,TARGET,zero_inflated,log1p_or_tweedie,True,True,False
2,CNT_CHILDREN,zero_inflated,log1p_or_tweedie,True,False,False
3,AMT_INCOME_TOTAL,heavy_tailed,log1p_or_quantile_transform,True,False,True
4,AMT_CREDIT,right_skewed,log1p,True,False,True
...,...,...,...,...,...,...
563,CC_NAME_CONTRACT_STATUS_Demand_MEAN,zero_inflated,log1p_or_tweedie,True,False,False
564,CC_NAME_CONTRACT_STATUS_Refused_MEAN,zero_inflated,log1p_or_tweedie,True,False,False
565,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,zero_inflated,log1p_or_tweedie,True,False,False
566,CC_NAME_CONTRACT_STATUS_Signed_MEAN,zero_inflated,log1p_or_tweedie,True,False,True


### 7.3 numerical_structure_report
* binning_opportunity
* zero_inflation_flag

In [70]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Optional, List


@dataclass
class NumericalStructureConfig:
    zero_inflation_threshold: float = 0.60
    min_unique_for_binning: int = 10
    skew_threshold: float = 0.8
    outlier_percentage_threshold: float = 5.0
    near_constant_threshold: float = 0.95   # New: flag near-constant columns
    min_samples: int = 100


def numerical_structure_report(
    df: pd.DataFrame,
    config: Optional[NumericalStructureConfig] = None
) -> pd.DataFrame:
    """
    Enhanced numerical structure report with better logic and richer insights.
    
    Added features:
    - Better binning heuristic with clear reasoning
    - Near-constant column detection
    - Discrete vs Continuous classification
    - Recommended action + explanation
    - Structure quality score
    """
    if config is None:
        config = NumericalStructureConfig()

    numeric_cols = [
        col for col in df.columns
        if pd.api.types.is_numeric_dtype(df[col]) 
        and not pd.api.types.is_bool_dtype(df[col])
    ]

    rows = []

    for col in numeric_cols:
        s = df[col].dropna()

        if len(s) < config.min_samples:
            rows.append({
                "column": col,
                "is_numeric": True,
                "is_discrete": False,
                "is_near_constant": True,
                "zero_inflation_flag": False,
                "binning_opportunity": False,
                "structure_score": 0.0,
                "recommended_action": "skip",
                "reason": "insufficient_samples",
                "zero_ratio": 0.0,
                "unique_count": 0,
                "skewness": 0.0,
                "outlier_percentage": 0.0,
            })
            continue

        zero_ratio = (s == 0).mean()
        unique_count = s.nunique()
        skewness = abs(s.skew()) if len(s) > 2 else 0.0

        # Near constant check
        is_near_constant = unique_count / len(s) < 0.01 or s.value_counts(normalize=True).max() >= config.near_constant_threshold

        # Outlier percentage using IQR
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        outlier_percentage = 0.0
        if iqr > 0:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outlier_percentage = ((s < lower) | (s > upper)).mean() * 100

        is_discrete = unique_count <= config.min_unique_for_binning or unique_count / len(s) < 0.05

        # Improved binning opportunity logic
        binning_reasons = []
        if unique_count < config.min_unique_for_binning:
            binning_opportunity = False
        else:
            if skewness >= config.skew_threshold:
                binning_reasons.append("high_skew")
            if outlier_percentage >= config.outlier_percentage_threshold:
                binning_reasons.append("high_outliers")
            if zero_ratio >= config.zero_inflation_threshold:
                binning_reasons.append("zero_inflated")
            if is_discrete:
                binning_reasons.append("discrete")

            binning_opportunity = len(binning_reasons) > 0

        reason = ", ".join(binning_reasons) if binning_reasons else "no_strong_signal"
        if is_near_constant:
            reason = "near_constant"

        # Structure score (0 to 1) - higher = more structured / interesting
        structure_score = round(
            (0.3 * (skewness > 0.5) +
             0.3 * (outlier_percentage > 3) +
             0.2 * (zero_ratio > 0.3) +
             0.2 * (unique_count > 15)), 2
        )

        recommended_action = "bin" if binning_opportunity else "keep_continuous"
        if is_near_constant:
            recommended_action = "drop_or_encode"
        elif is_discrete:
            recommended_action = "treat_as_categorical"

        rows.append({
            "column": col,
            "is_numeric": True,
            "is_discrete": is_discrete,
            "is_near_constant": is_near_constant,
            "zero_inflation_flag": zero_ratio >= config.zero_inflation_threshold,
            "binning_opportunity": binning_opportunity,
            "structure_score": structure_score,
            "recommended_action": recommended_action,
            "reason": reason,
            "zero_ratio": round(zero_ratio, 4),
            "unique_count": unique_count,
            "skewness": round(skewness, 4),
            "outlier_percentage": round(outlier_percentage, 2),
        })

    return pd.DataFrame(rows).sort_values(by="structure_score", ascending=False)


# Example usage:
# config = NumericalStructureConfig(zero_inflation_threshold=0.5, skew_threshold=0.75)
# structure_report = numerical_structure_report(df, config)

In [71]:
train_numerical_structure_report = numerical_structure_report(train_df)
train_numerical_structure_report

,column,is_numeric,is_discrete,is_near_constant,zero_inflation_flag,binning_opportunity,structure_score,recommended_action,reason,zero_ratio,unique_count,skewness,outlier_percentage
541,CC_CNT_INSTALMENT_MATURE_CUM_MEAN,True,False,False,False,True,1.0,bin,"high_skew, high_outliers",0.3152,13923,1.4590,6.48
540,CC_CNT_INSTALMENT_MATURE_CUM_MAX,True,True,True,False,True,1.0,drop_or_encode,near_constant,0.3152,120,1.7570,7.22
539,CC_CNT_DRAWINGS_POS_CURRENT_SUM,True,True,True,True,True,1.0,drop_or_encode,near_constant,0.6065,532,8.4642,17.34
538,CC_CNT_DRAWINGS_POS_CURRENT_MEAN,True,False,False,False,True,1.0,bin,"high_skew, high_outliers",0.4407,5024,5.3480,14.72
537,CC_CNT_DRAWINGS_POS_CURRENT_MAX,True,True,True,False,True,1.0,drop_or_encode,near_constant,0.4407,123,3.1902,9.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,BUREAU_CLOSED_DAYS_CREDIT_MIN,True,True,False,False,True,0.2,treat_as_categorical,discrete,0.0000,2912,0.3955,0.00
0,SK_ID_CURR,True,False,False,False,False,0.2,keep_continuous,no_strong_signal,0.0000,307511,0.0012,0.00
567,CC_NAME_CONTRACT_STATUS_nan_MEAN,True,True,True,True,False,0.2,drop_or_encode,near_constant,1.0000,1,0.0000,0.00
21,REGION_RATING_CLIENT_W_CITY,True,True,True,False,False,0.0,drop_or_encode,near_constant,0.0000,3,0.0597,0.00


In [72]:
import pandas as pd
from functools import reduce

numerical_robustness_transformation_report = reduce(
    lambda left, right: pd.merge(left, right, on="column", how="outer"),
    [
        train_numerical_outlier_report,
        train_numerical_transformation_report,
        train_numerical_structure_report
    ]
)

numerical_robustness_transformation_report

,column,count,missing_count,iqr_q1,iqr_q3,iqr_value,iqr_lower_bound,iqr_upper_bound,iqr_lower_count,iqr_upper_count,...,is_near_constant,zero_inflation_flag,binning_opportunity,structure_score,recommended_action,reason,zero_ratio,unique_count,skewness_y,outlier_percentage
0,AMT_ANNUITY,307499,12,16524.00,34596.00,18072.00,-10584.00,61704.00,0,7504,...,False,False,True,0.5,treat_as_categorical,"high_skew, discrete",0.0000,13672,1.5798,2.44
1,AMT_CREDIT,307511,0,270000.00,808650.00,538650.00,-537975.00,1616625.00,0,6562,...,False,False,True,0.5,treat_as_categorical,"high_skew, discrete",0.0000,5603,1.2348,2.13
2,AMT_GOODS_PRICE,307233,278,238500.00,679500.00,441000.00,-423000.00,1341000.00,0,14728,...,True,False,True,0.8,drop_or_encode,near_constant,0.0000,1002,1.3490,4.79
3,AMT_INCOME_TOTAL,307511,0,112500.00,202500.00,90000.00,-22500.00,337500.00,0,14035,...,True,False,True,0.8,drop_or_encode,near_constant,0.0000,2548,391.5597,4.56
4,AMT_REQ_CREDIT_BUREAU_DAY,265992,41519,0.00,0.00,0.00,0.00,0.00,0,1489,...,True,True,False,0.5,drop_or_encode,near_constant,0.9944,9,27.0435,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
563,YEARS_BEGINEXPLUATATION_MEDI,157504,150007,0.98,0.99,0.01,0.96,1.00,4762,0,...,True,False,True,0.8,drop_or_encode,near_constant,0.0035,245,15.5731,3.02
564,YEARS_BEGINEXPLUATATION_MODE,157504,150007,0.98,0.99,0.01,0.96,1.00,5074,0,...,True,False,True,0.8,drop_or_encode,near_constant,0.0009,221,14.7553,3.22
565,YEARS_BUILD_AVG,103023,204488,0.69,0.82,0.14,0.48,1.03,2154,0,...,True,False,True,0.5,drop_or_encode,near_constant,0.0010,149,0.9625,2.09
566,YEARS_BUILD_MEDI,103023,204488,0.69,0.83,0.13,0.49,1.03,2274,0,...,True,False,True,0.5,drop_or_encode,near_constant,0.0010,151,0.9628,2.21


In [73]:
numerical_robustness_transformation_report.to_csv(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\7.numerical_robustness_report.csv")

In [74]:
numerical_robustness_transformation_report.head()

,column,count,missing_count,iqr_q1,iqr_q3,iqr_value,iqr_lower_bound,iqr_upper_bound,iqr_lower_count,iqr_upper_count,...,is_near_constant,zero_inflation_flag,binning_opportunity,structure_score,recommended_action,reason,zero_ratio,unique_count,skewness_y,outlier_percentage
0,AMT_ANNUITY,307499,12,16524.0,34596.0,18072.0,-10584.0,61704.0,0,7504,...,False,False,True,0.5,treat_as_categorical,"high_skew, discrete",0.0000,13672,1.5798,2.44
1,AMT_CREDIT,307511,0,270000.0,808650.0,538650.0,-537975.0,1616625.0,0,6562,...,False,False,True,0.5,treat_as_categorical,"high_skew, discrete",0.0000,5603,1.2348,2.13
2,AMT_GOODS_PRICE,307233,278,238500.0,679500.0,441000.0,-423000.0,1341000.0,0,14728,...,True,False,True,0.8,drop_or_encode,near_constant,0.0000,1002,1.3490,4.79
3,AMT_INCOME_TOTAL,307511,0,112500.0,202500.0,90000.0,-22500.0,337500.0,0,14035,...,True,False,True,0.8,drop_or_encode,near_constant,0.0000,2548,391.5597,4.56
4,AMT_REQ_CREDIT_BUREAU_DAY,265992,41519,0.0,0.0,0.0,0.0,0.0,0,1489,...,True,True,False,0.5,drop_or_encode,near_constant,0.9944,9,27.0435,0.00


# Step 8 Numerical Target-Aware Audit

### 8.1 numerical_correlation_signal_report
* pearson_corr_with_target
* spearman_corr_with_target
* linear_signal_strength

In [75]:
import warnings
from typing import Literal

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from scipy.stats import pearsonr, spearmanr, kendalltau, pointbiserialr
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
from sklearn.preprocessing import LabelEncoder


def _safe_round(value, decimals: int = 4):
    """Round finite values; return NaN otherwise."""
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), decimals)


def _get_numeric_columns(df: pd.DataFrame, exclude_cols: list = None) -> list:
    """Return numeric (non-boolean) column names, excluding specified columns."""
    exclude_cols = exclude_cols or []
    return [
        col for col in df.columns
        if col not in exclude_cols
        and pd.api.types.is_numeric_dtype(df[col])
        and not pd.api.types.is_bool_dtype(df[col])
    ]


def _correlation_strength_label(value: float, thresholds: dict = None) -> str:
    if pd.isna(value):
        return "unknown"
    thresholds = thresholds or {
        "negligible": 0.1,
        "weak": 0.3,
        "moderate": 0.5,
        "strong": 0.7,
    }
    abs_val = abs(value)
    if abs_val < thresholds["negligible"]:
        return "negligible"
    elif abs_val < thresholds["weak"]:
        return "weak"
    elif abs_val < thresholds["moderate"]:
        return "moderate"
    elif abs_val < thresholds["strong"]:
        return "strong"
    else:
        return "very_strong"


def _correlation_direction(value: float) -> str:
    if pd.isna(value):
        return "unknown"
    if value > 0.01:
        return "positive"
    elif value < -0.01:
        return "negative"
    else:
        return "none"


def _safe_correlation(x, y, method="pearson"):
    try:
        mask = ~(x.isna() | y.isna())
        x_clean = x[mask]
        y_clean = y[mask]
        if len(x_clean) < 3:
            return np.nan, np.nan
        if x_clean.nunique() <= 1 or y_clean.nunique() <= 1:
            return np.nan, np.nan
        if method == "pearson":
            corr, pval = pearsonr(x_clean, y_clean)
        elif method == "spearman":
            corr, pval = spearmanr(x_clean, y_clean)
        elif method == "kendall":
            corr, pval = kendalltau(x_clean, y_clean)
        else:
            return np.nan, np.nan
        return corr, pval
    except Exception:
        return np.nan, np.nan


def _calculate_point_biserial(x, y):
    try:
        mask = ~(x.isna() | y.isna())
        x_clean = x[mask]
        y_clean = y[mask]
        if len(x_clean) < 3:
            return np.nan, np.nan
        if y_clean.nunique() != 2:
            return np.nan, np.nan
        corr, pval = pointbiserialr(y_clean, x_clean)
        return corr, pval
    except Exception:
        return np.nan, np.nan


def _calculate_mutual_information(x, y, is_classification=False, n_neighbors=3, random_state=42):
    try:
        mask = ~(x.isna() | y.isna())
        x_clean = x[mask].values.reshape(-1, 1)
        y_clean = y[mask].values
        if len(x_clean) < 10:
            return np.nan
        if is_classification:
            if not np.issubdtype(y_clean.dtype, np.number):
                y_clean = LabelEncoder().fit_transform(y_clean)
            mi = mutual_info_classif(x_clean, y_clean, n_neighbors=n_neighbors, random_state=random_state)[0]
        else:
            mi = mutual_info_regression(x_clean, y_clean, n_neighbors=n_neighbors, random_state=random_state)[0]
        return mi
    except Exception:
        return np.nan


def _calculate_correlation_confidence_interval(r, n, confidence=0.95):
    if pd.isna(r) or n < 4:
        return np.nan, np.nan
    try:
        z = 0.5 * np.log((1 + r) / (1 - r))
        se = 1 / np.sqrt(n - 3)
        z_crit = scipy_stats.norm.ppf((1 + confidence) / 2)
        z_lower = z - z_crit * se
        z_upper = z + z_crit * se
        r_lower = (np.exp(2 * z_lower) - 1) / (np.exp(2 * z_lower) + 1)
        r_upper = (np.exp(2 * z_upper) - 1) / (np.exp(2 * z_upper) + 1)
        return r_lower, r_upper
    except Exception:
        return np.nan, np.nan


def _detect_target_type(y):
    nunique = y.nunique()
    if nunique == 2:
        return "binary"
    elif nunique <= 10 or not pd.api.types.is_numeric_dtype(y):
        return "multiclass"
    else:
        return "continuous"


def _calculate_eta_squared(x, y):
    try:
        mask = ~(x.isna() | y.isna())
        x_clean = x[mask]
        y_clean = y[mask]
        if len(x_clean) < 10:
            return np.nan
        groups = [x_clean[y_clean == cat].values for cat in y_clean.unique()]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            return np.nan
        f_stat, p_val = scipy_stats.f_oneway(*groups)
        grand_mean = x_clean.mean()
        ss_total = ((x_clean - grand_mean) ** 2).sum()
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        if ss_total == 0:
            return np.nan
        eta_sq = ss_between / ss_total
        return eta_sq
    except Exception:
        return np.nan

In [76]:
def numerical_correlation_signal_report(
    df: pd.DataFrame,
    target_col: str,
    include_pearson: bool = True,
    include_spearman: bool = True,
    include_kendall: bool = False,
    include_point_biserial: bool = True,
    include_mutual_info: bool = True,
    include_eta_squared: bool = True,
    include_pvalues: bool = True,
    include_confidence_intervals: bool = True,
    confidence_level: float = 0.95,
    significance_threshold: float = 0.05,
    strength_thresholds: dict = None,
    sort_by: str = "abs_max_correlation",
    ascending: bool = False,
    top_n: int = None,
    filter_significant_only: bool = False,
    decimals: int = 4,
) -> pd.DataFrame:
    """
    Comprehensive numerical correlation signal report.
    
    KEY OUTPUT COLUMNS (used downstream):
    - pearson_corr_with_target   (not 'pearson_corr')
    - spearman_corr_with_target  (not 'spearman_corr')
    """

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe.")

    if df.empty:
        warnings.warn("Empty DataFrame provided.", UserWarning)
        return pd.DataFrame()

    numeric_cols = _get_numeric_columns(df, exclude_cols=[target_col])

    if not numeric_cols:
        warnings.warn("No numeric columns found (excluding target).", UserWarning)
        return pd.DataFrame()

    target = df[target_col]
    target_type = _detect_target_type(target)
    is_binary = target_type == "binary"
    is_classification = target_type in ["binary", "multiclass"]

    records = []

    for col in numeric_cols:
        feature = df[col]
        mask = ~(feature.isna() | target.isna())
        sample_size = mask.sum()

        record = {
            "column": col,
            "sample_size": sample_size,
            "target_type": target_type,
        }

        correlations = {}
        pvalues = {}

        # ── PEARSON ──────────────────────────────────────────────────
        if include_pearson:
            pearson_r, pearson_p = _safe_correlation(feature, target, method="pearson")
            # FIX: renamed to pearson_corr_with_target
            record["pearson_corr_with_target"] = _safe_round(pearson_r, decimals)
            if include_pvalues:
                record["pearson_pvalue"] = _safe_round(pearson_p, decimals + 2)
            if include_confidence_intervals:
                ci_lower, ci_upper = _calculate_correlation_confidence_interval(
                    pearson_r, sample_size, confidence_level
                )
                record["pearson_ci_lower"] = _safe_round(ci_lower, decimals)
                record["pearson_ci_upper"] = _safe_round(ci_upper, decimals)
            if pd.notna(pearson_r):
                correlations["pearson"] = pearson_r
                pvalues["pearson"] = pearson_p

        # ── SPEARMAN ─────────────────────────────────────────────────
        if include_spearman:
            spearman_r, spearman_p = _safe_correlation(feature, target, method="spearman")
            # FIX: renamed to spearman_corr_with_target
            record["spearman_corr_with_target"] = _safe_round(spearman_r, decimals)
            if include_pvalues:
                record["spearman_pvalue"] = _safe_round(spearman_p, decimals + 2)
            if pd.notna(spearman_r):
                correlations["spearman"] = spearman_r
                pvalues["spearman"] = spearman_p

        # ── KENDALL ──────────────────────────────────────────────────
        if include_kendall:
            kendall_r, kendall_p = _safe_correlation(feature, target, method="kendall")
            record["kendall_corr"] = _safe_round(kendall_r, decimals)
            if include_pvalues:
                record["kendall_pvalue"] = _safe_round(kendall_p, decimals + 2)
            if pd.notna(kendall_r):
                correlations["kendall"] = kendall_r
                pvalues["kendall"] = kendall_p

        # ── POINT-BISERIAL (binary target) ───────────────────────────
        if include_point_biserial and is_binary:
            pb_r, pb_p = _calculate_point_biserial(feature, target)
            record["point_biserial_corr"] = _safe_round(pb_r, decimals)
            if include_pvalues:
                record["point_biserial_pvalue"] = _safe_round(pb_p, decimals + 2)
            if pd.notna(pb_r):
                correlations["point_biserial"] = pb_r
                pvalues["point_biserial"] = pb_p

        # ── MUTUAL INFORMATION ───────────────────────────────────────
        if include_mutual_info:
            mi = _calculate_mutual_information(
                feature, target, is_classification=is_classification,
            )
            record["mutual_info_score"] = _safe_round(mi, decimals)
            if pd.notna(mi) and mi > 0:
                mi_norm = min(1.0, mi / 2.0)
                record["mutual_info_normalized"] = _safe_round(mi_norm, decimals)
            else:
                record["mutual_info_normalized"] = 0.0 if mi == 0 else np.nan

        # ── ETA-SQUARED (categorical target) ─────────────────────────
        if include_eta_squared and is_classification:
            eta_sq = _calculate_eta_squared(feature, target)
            record["eta_squared"] = _safe_round(eta_sq, decimals)

        # ── SUMMARY METRICS ──────────────────────────────────────────
        if correlations:
            abs_corrs = {k: abs(v) for k, v in correlations.items() if pd.notna(v)}
            if abs_corrs:
                best_method = max(abs_corrs, key=abs_corrs.get)
                max_abs_corr = abs_corrs[best_method]
                best_corr_value = correlations[best_method]
                best_pvalue = pvalues.get(best_method, np.nan)

                record["abs_max_correlation"] = _safe_round(max_abs_corr, decimals)
                record["best_method"] = best_method
                record["best_correlation"] = _safe_round(best_corr_value, decimals)
                record["best_pvalue"] = _safe_round(best_pvalue, decimals + 2)
                record["correlation_direction"] = _correlation_direction(best_corr_value)
                record["signal_strength"] = _correlation_strength_label(
                    best_corr_value, strength_thresholds
                )
                record["is_significant"] = (
                    pd.notna(best_pvalue) and best_pvalue < significance_threshold
                )
                record["is_strong_signal"] = (
                    record["is_significant"] and max_abs_corr >= 0.3
                )
            else:
                record.update({
                    "abs_max_correlation": np.nan, "best_method": "none",
                    "best_correlation": np.nan, "best_pvalue": np.nan,
                    "correlation_direction": "unknown", "signal_strength": "unknown",
                    "is_significant": False, "is_strong_signal": False,
                })
        else:
            record.update({
                "abs_max_correlation": np.nan, "best_method": "none",
                "best_correlation": np.nan, "best_pvalue": np.nan,
                "correlation_direction": "unknown", "signal_strength": "unknown",
                "is_significant": False, "is_strong_signal": False,
            })

        records.append(record)

    result = pd.DataFrame(records)

    if "sample_size" in result.columns:
        result["sample_size"] = result["sample_size"].astype("Int64")

    if filter_significant_only and "is_significant" in result.columns:
        result = result[result["is_significant"]].copy()

    if sort_by and sort_by in result.columns and not result.empty:
        result = result.sort_values(sort_by, ascending=ascending, na_position="last").reset_index(drop=True)

    if top_n and top_n > 0:
        result = result.head(top_n)

    return result

In [77]:
train_numerical_correlation_signal_report = numerical_correlation_signal_report(
    df=train_df,
    target_col="TARGET"
)

train_numerical_correlation_signal_report

,column,sample_size,target_type,pearson_corr_with_target,pearson_pvalue,pearson_ci_lower,pearson_ci_upper,spearman_corr_with_target,spearman_pvalue,point_biserial_corr,...,mutual_info_normalized,eta_squared,abs_max_correlation,best_method,best_correlation,best_pvalue,correlation_direction,signal_strength,is_significant,is_strong_signal
0,APP_EXT_SOURCE_MEAN,307339,binary,-0.2221,0.0,-0.2254,-0.2187,-0.2043,0.0,-0.2221,...,0.0118,0.0493,0.2221,pearson,-0.2221,0.0,negative,weak,True,False
1,APP_EXT_SOURCE_MAX,307339,binary,-0.1969,0.0,-0.2003,-0.1935,-0.1757,0.0,-0.1969,...,0.0085,0.0388,0.1969,pearson,-0.1969,0.0,negative,weak,True,False
2,APP_EXT_SOURCE_MIN,307339,binary,-0.1853,0.0,-0.1887,-0.1818,-0.1803,0.0,-0.1853,...,0.0092,0.0343,0.1853,pearson,-0.1853,0.0,negative,weak,True,False
3,EXT_SOURCE_3,246546,binary,-0.1789,0.0,-0.1827,-0.1751,-0.1663,0.0,-0.1789,...,0.0083,0.0320,0.1789,pearson,-0.1789,0.0,negative,weak,True,False
4,EXT_SOURCE_2,306851,binary,-0.1605,0.0,-0.1639,-0.1570,-0.1473,0.0,-0.1605,...,0.0061,0.0258,0.1605,pearson,-0.1605,0.0,negative,weak,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
562,PREV_CHANNEL_TYPE_nan_MEAN,291057,binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,NaN,NaN,none,NaN,NaN,unknown,unknown,False,False
563,PREV_NAME_SELLER_INDUSTRY_nan_MEAN,291057,binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,NaN,NaN,none,NaN,NaN,unknown,unknown,False,False
564,PREV_NAME_YIELD_GROUP_nan_MEAN,291057,binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,NaN,NaN,none,NaN,NaN,unknown,unknown,False,False
565,POS_NAME_CONTRACT_STATUS_nan_MEAN,289444,binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0000,NaN,NaN,none,NaN,NaN,unknown,unknown,False,False


### 8.2 numerical_information_signal_report
* mutual_information_numeric
* nonlinear_signal_strength
* signal_to_noise_flag

In [78]:
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.impute import SimpleImputer


def numerical_information_signal_report(
    df: pd.DataFrame,
    target_col: str,
    random_state: int = 42,
    mi_noise_threshold: float = 0.005
) -> pd.DataFrame:
    """
    Highly Optimized, Matrix-Level Mutual Information Report.
    
    Output columns:
    - column
    - mutual_information_numeric
    - nonlinear_signal_strength
    - signal_to_noise_flag
    """

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe.")

    valid_df = df.dropna(subset=[target_col]).copy()
    if valid_df.empty:
        return pd.DataFrame()

    y = valid_df[target_col]

    X = valid_df.select_dtypes(include=['number']).select_dtypes(exclude=['bool'])
    if target_col in X.columns:
        X = X.drop(columns=[target_col])

    if X.empty:
        return pd.DataFrame()

    nunique = X.nunique()
    valid_cols = nunique[nunique > 1].index.tolist()
    invalid_cols = nunique[nunique <= 1].index.tolist()

    report_data = []

    if valid_cols:
        imputer = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X[valid_cols])

        is_regression = pd.api.types.is_float_dtype(y) and y.nunique() > 20

        if is_regression:
            mi_scores = mutual_info_regression(X_imputed, y, random_state=random_state)
        else:
            mi_scores = mutual_info_classif(X_imputed, y, random_state=random_state)

        for col, mi_val in zip(valid_cols, mi_scores):
            report_data.append({"column": col, "mutual_information_numeric": mi_val})

    for col in invalid_cols:
        report_data.append({"column": col, "mutual_information_numeric": np.nan})

    report_df = pd.DataFrame(report_data)

    conditions = [
        report_df['mutual_information_numeric'].isna(),
        report_df['mutual_information_numeric'] < 0.005,
        report_df['mutual_information_numeric'] < 0.02,
        report_df['mutual_information_numeric'] < 0.05,
        report_df['mutual_information_numeric'] >= 0.05
    ]
    choices = ["unknown", "negligible", "weak", "moderate", "strong"]

    report_df['nonlinear_signal_strength'] = np.select(conditions, choices, default="unknown")

    report_df['signal_to_noise_flag'] = (
        report_df['mutual_information_numeric'].notna() &
        (report_df['mutual_information_numeric'] >= mi_noise_threshold)
    )

    report_df['mutual_information_numeric'] = report_df['mutual_information_numeric'].round(6)

    return report_df[[
        "column",
        "mutual_information_numeric",
        "nonlinear_signal_strength",
        "signal_to_noise_flag"
    ]]

In [79]:
train_numerical_information_signal_report = numerical_information_signal_report(
    df=train_df,
    target_col="TARGET"
)

train_numerical_information_signal_report

,column,mutual_information_numeric,nonlinear_signal_strength,signal_to_noise_flag
0,SK_ID_CURR,0.000000,negligible,False
1,CNT_CHILDREN,0.004793,negligible,False
2,AMT_INCOME_TOTAL,0.002680,negligible,False
3,AMT_CREDIT,0.006944,weak,True
4,AMT_ANNUITY,0.012044,weak,True
...,...,...,...,...
562,PREV_CHANNEL_TYPE_nan_MEAN,NaN,unknown,False
563,PREV_NAME_SELLER_INDUSTRY_nan_MEAN,NaN,unknown,False
564,PREV_NAME_YIELD_GROUP_nan_MEAN,NaN,unknown,False
565,POS_NAME_CONTRACT_STATUS_nan_MEAN,NaN,unknown,False


### 8.3 numerical_binned_target_report
* target_mean_by_bin_variation
* monotonic_trend_with_target
* bin_lift_max

In [80]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class TargetBinnedConfig:
    n_bins: int = 10
    min_samples_per_bin: int = 50
    min_valid_bins: int = 3
    monotonic_tolerance: float = 0.02
    variation_threshold: float = 0.05
    correlation_method: str = "spearman"


def numerical_binned_target_report(
    df: pd.DataFrame,
    target_col: str,
    config: Optional[TargetBinnedConfig] = None
) -> pd.DataFrame:
    """
    Enhanced numerical binned target behavior report.
    
    KEY OUTPUT COLUMNS (used downstream):
    - target_mean_by_bin_variation  (FIX: was 'target_mean_variation')
    - bin_lift_max
    - monotonic_trend_with_target   (FIX: was 'monotonic_trend')
    """
    if config is None:
        config = TargetBinnedConfig()

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found.")

    numeric_cols = [
        col for col in df.columns
        if col != target_col
        and pd.api.types.is_numeric_dtype(df[col])
        and not pd.api.types.is_bool_dtype(df[col])
    ]

    overall_target_mean = df[target_col].dropna().mean()
    rows = []

    for col in numeric_cols:
        temp = df[[col, target_col]].dropna()

        if len(temp) < config.min_samples_per_bin * config.min_valid_bins or temp[col].nunique() < config.min_valid_bins:
            rows.append(_create_empty_row(col, reason="insufficient_data"))
            continue

        try:
            binned = pd.qcut(
                temp[col],
                q=config.n_bins,
                duplicates="drop",
                retbins=False
            )

            bin_stats = (
                temp.assign(bin=binned)
                .groupby("bin", observed=True)[target_col]
                .agg(mean="mean", count="size", sum="sum")
                .reset_index()
            )

            bin_stats = bin_stats[bin_stats["count"] >= config.min_samples_per_bin]

            if len(bin_stats) < config.min_valid_bins:
                rows.append(_create_empty_row(col, reason="too_few_valid_bins"))
                continue

            bin_means = bin_stats["mean"].values
            target_mean_by_bin_variation = bin_means.max() - bin_means.min()

            diffs = np.diff(bin_means)
            is_monotonic = (
                np.all(diffs >= -config.monotonic_tolerance) or
                np.all(diffs <= config.monotonic_tolerance)
            )

            bin_lift_max = bin_means.max() / overall_target_mean if overall_target_mean > 0 else np.nan

            correlation = temp[col].corr(temp[target_col], method=config.correlation_method)

            iv = _calculate_information_value(bin_stats, overall_target_mean)

            is_predictive = (
                target_mean_by_bin_variation >= config.variation_threshold and
                (abs(correlation) > 0.1 or iv > 0.02)
            )

            predictive_level = _get_predictive_level(target_mean_by_bin_variation, iv, abs(correlation))

            rows.append({
                "column": col,
                # FIX: consistent column name for downstream
                "target_mean_by_bin_variation": round(target_mean_by_bin_variation, 4),
                # FIX: consistent column name for downstream
                "monotonic_trend_with_target": is_monotonic,
                "bin_lift_max": round(bin_lift_max, 4),
                "correlation": round(correlation, 4),
                "information_value": round(iv, 4),
                "predictive_level": predictive_level,
                "is_predictive": is_predictive,
                "n_valid_bins": len(bin_stats),
                "unique_values": temp[col].nunique(),
            })

        except Exception as e:
            rows.append(_create_empty_row(col, reason="error", error=str(e)))

    return pd.DataFrame(rows).sort_values(
        by=["information_value", "target_mean_by_bin_variation"],
        ascending=False
    ).reset_index(drop=True)


def _calculate_information_value(bin_stats: pd.DataFrame, overall_mean: float) -> float:
    """Calculate Information Value (IV)."""
    bin_stats = bin_stats.copy()
    bin_stats["non_event"] = bin_stats["count"] - bin_stats["sum"]

    total_event = bin_stats["sum"].sum()
    total_non_event = bin_stats["non_event"].sum()

    if total_event == 0 or total_non_event == 0:
        return 0.0

    bin_stats["event_dist"] = bin_stats["sum"] / total_event
    bin_stats["non_event_dist"] = bin_stats["non_event"] / total_non_event

    bin_stats["woe"] = np.log(
        (bin_stats["event_dist"] + 1e-8) / (bin_stats["non_event_dist"] + 1e-8)
    )
    bin_stats["iv_contrib"] = (bin_stats["event_dist"] - bin_stats["non_event_dist"]) * bin_stats["woe"]

    return bin_stats["iv_contrib"].sum()


def _get_predictive_level(variation: float, iv: float, corr: float) -> str:
    if iv > 0.3 or (variation > 0.15 and abs(corr) > 0.25):
        return "strong"
    elif iv > 0.1 or (variation > 0.08 and abs(corr) > 0.15):
        return "medium"
    elif iv > 0.02 or variation > 0.04:
        return "weak"
    return "very_weak"


def _create_empty_row(col: str, reason: str = "error", error: str = "") -> dict:
    return {
        "column": col,
        # FIX: consistent column names
        "target_mean_by_bin_variation": np.nan,
        "monotonic_trend_with_target": False,
        "bin_lift_max": np.nan,
        "correlation": np.nan,
        "information_value": np.nan,
        "predictive_level": "none",
        "is_predictive": False,
        "n_valid_bins": 0,
        "unique_values": 0,
        "reason": reason,
    }

In [81]:
train_numerical_binned_target_report = numerical_binned_target_report(
    df=train_df,
    target_col="TARGET"
)

train_numerical_binned_target_report

,column,target_mean_by_bin_variation,monotonic_trend_with_target,bin_lift_max,correlation,information_value,predictive_level,is_predictive,n_valid_bins,unique_values,reason
0,APP_EXT_SOURCE_MEAN,0.2089,True,2.8407,-0.2043,0.6086,strong,True,10,300347,NaN
1,APP_EXT_SOURCE_MIN,0.1860,True,2.6298,-0.1803,0.4663,strong,True,10,139146,NaN
2,APP_EXT_SOURCE_MAX,0.1821,True,2.6089,-0.1757,0.4426,strong,True,10,121113,NaN
3,EXT_SOURCE_3,0.1678,True,2.4778,-0.1663,0.4101,strong,True,10,814,NaN
4,EXT_SOURCE_1,0.1499,True,2.1756,-0.1511,0.3465,strong,True,10,114584,NaN
...,...,...,...,...,...,...,...,...,...,...,...
562,CC_NAME_CONTRACT_STATUS_Demand_MEAN,NaN,False,NaN,NaN,NaN,none,False,0,0,too_few_valid_bins
563,CC_NAME_CONTRACT_STATUS_Refused_MEAN,NaN,False,NaN,NaN,NaN,none,False,0,0,too_few_valid_bins
564,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,NaN,False,NaN,NaN,NaN,none,False,0,0,too_few_valid_bins
565,CC_NAME_CONTRACT_STATUS_Signed_MEAN,NaN,False,NaN,NaN,NaN,none,False,0,0,too_few_valid_bins


### 8.4 numerical_predictive_summary_report
* predictive_power_band

In [82]:
import logging

logger = logging.getLogger(__name__)


# ─── Configuration ───────────────────────────────────────────────────────────

DEFAULT_SCORING_CONFIG = {
    "max_abs_correlation": [
        (0.30, 3),
        (0.15, 2),
        (0.05, 1),
    ],
    "mutual_information_numeric": [
        (0.05, 3),
        (0.02, 2),
        (0.005, 1),
    ],
    "target_mean_by_bin_variation": [
        (0.15, 3),
        (0.08, 2),
        (0.03, 1),
    ],
    "bin_lift_max": [
        (3.0, 3),
        (2.0, 2),
        (1.3, 1),
    ],
}

DEFAULT_BAND_THRESHOLDS = [
    (10, "very_strong"),
    (7,  "strong"),
    (4,  "moderate"),
    (2,  "weak"),
    (0,  "very_weak"),
]

# FIX: Column names now match actual report outputs
REQUIRED_COLUMNS = {
    "correlation_report": ["column", "pearson_corr_with_target", "spearman_corr_with_target"],
    "information_report": ["column", "mutual_information_numeric"],
    "binned_target_report": ["column", "target_mean_by_bin_variation", "bin_lift_max"],
}

OUTPUT_COLUMNS = [
    "column",
    "max_abs_correlation",
    "mutual_information_numeric",
    "target_mean_by_bin_variation",
    "bin_lift_max",
    "corr_score",
    "mi_score",
    "bin_var_score",
    "bin_lift_score",
    "total_score",
    "predictive_power_band",
]


def _validate_report(df, report_name):
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"'{report_name}' must be a pd.DataFrame, got {type(df).__name__}")
    required = REQUIRED_COLUMNS[report_name]
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(
            f"'{report_name}' is missing required columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )


def _score_signal(series, thresholds):
    score = pd.Series(0, index=series.index, dtype="int32")
    for threshold, points in thresholds:
        mask = (score == 0) & (series >= threshold)
        score = score.where(~mask, points)
    score = score.where(series.notna(), 0)
    return score


def _assign_band(total_score, band_thresholds):
    band = pd.Series("very_weak", index=total_score.index, dtype="object")
    for threshold, label in sorted(band_thresholds, key=lambda x: x[0]):
        band = band.where(~(total_score >= threshold), label)
    return band


def _safe_merge_reports(correlation_df, information_df, binned_df):
    corr_cols = REQUIRED_COLUMNS["correlation_report"]
    info_cols = REQUIRED_COLUMNS["information_report"]
    bin_cols  = REQUIRED_COLUMNS["binned_target_report"]

    merged = (
        correlation_df[corr_cols]
        .merge(information_df[info_cols], on="column", how="outer")
        .merge(binned_df[bin_cols], on="column", how="outer")
    )

    dup_count = merged["column"].duplicated().sum()
    if dup_count > 0:
        warnings.warn(
            f"Found {dup_count} duplicate 'column' entries after merging. Keeping first.",
            UserWarning,
            stacklevel=3,
        )
        merged = merged.drop_duplicates(subset="column", keep="first").reset_index(drop=True)

    return merged


def numerical_predictive_summary_report(
    numerical_correlation_signal_report_df: pd.DataFrame,
    numerical_information_signal_report_df: pd.DataFrame,
    numerical_binned_target_report_df: pd.DataFrame,
    scoring_config=None,
    band_thresholds=None,
    include_score_details: bool = True,
) -> pd.DataFrame:
    """
    Build a numerical predictive summary report with transparent scoring.
    """

    _validate_report(numerical_correlation_signal_report_df, "correlation_report")
    _validate_report(numerical_information_signal_report_df, "information_report")
    _validate_report(numerical_binned_target_report_df, "binned_target_report")

    config = scoring_config or DEFAULT_SCORING_CONFIG
    bands = band_thresholds or DEFAULT_BAND_THRESHOLDS

    merged = _safe_merge_reports(
        numerical_correlation_signal_report_df,
        numerical_information_signal_report_df,
        numerical_binned_target_report_df,
    )

    if merged.empty:
        logger.warning("Merged report is empty — returning empty DataFrame.")
        output_cols = OUTPUT_COLUMNS if include_score_details else ["column", "predictive_power_band"]
        return pd.DataFrame(columns=output_cols)

    logger.info(f"Scoring {len(merged)} numeric feature(s).")

    # Compute derived signal: max absolute correlation
    merged["max_abs_correlation"] = np.fmax(
        merged["pearson_corr_with_target"].abs(),
        merged["spearman_corr_with_target"].abs(),
    )

    # Vectorized scoring
    signal_to_score_col = {
        "max_abs_correlation":          "corr_score",
        "mutual_information_numeric":   "mi_score",
        "target_mean_by_bin_variation": "bin_var_score",
        "bin_lift_max":                 "bin_lift_score",
    }

    for signal_name, score_col in signal_to_score_col.items():
        thresholds = config.get(signal_name, [])
        if not thresholds:
            warnings.warn(
                f"No scoring thresholds for '{signal_name}'. Contributes 0.",
                UserWarning,
                stacklevel=2,
            )
        merged[score_col] = _score_signal(merged[signal_name], thresholds)

    score_cols = list(signal_to_score_col.values())
    merged["total_score"] = merged[score_cols].sum(axis=1)
    merged["predictive_power_band"] = _assign_band(merged["total_score"], bands)

    merged = merged.sort_values("total_score", ascending=False).reset_index(drop=True)

    if include_score_details:
        # Only select columns that exist
        available_output = [c for c in OUTPUT_COLUMNS if c in merged.columns]
        output = merged[available_output].copy()
    else:
        output = merged[["column", "predictive_power_band"]].copy()

    return output

In [83]:
# Quick patch — rename old column to expected name
train_numerical_binned_target_report = train_numerical_binned_target_report.rename(columns={
    "target_mean_variation": "target_mean_by_bin_variation",
    "monotonic_trend": "monotonic_trend_with_target"
})

# Now run the predictive summary
train_numerical_predictive_summary_report = numerical_predictive_summary_report(
    numerical_correlation_signal_report_df=train_numerical_correlation_signal_report,
    numerical_information_signal_report_df=train_numerical_information_signal_report,
    numerical_binned_target_report_df=train_numerical_binned_target_report
)

train_numerical_predictive_summary_report

,column,max_abs_correlation,mutual_information_numeric,target_mean_by_bin_variation,bin_lift_max,corr_score,mi_score,bin_var_score,bin_lift_score,total_score,predictive_power_band
0,APP_EXT_SOURCE_MEAN,0.2221,0.023564,0.2089,2.8407,2,2,3,2,9,strong
1,APP_EXT_SOURCE_MIN,0.1853,0.018027,0.1860,2.6298,2,1,3,2,8,strong
2,APP_EXT_SOURCE_MAX,0.1969,0.017253,0.1821,2.6089,2,1,3,2,8,strong
3,EXT_SOURCE_3,0.1789,0.015743,0.1678,2.4778,2,1,3,2,8,strong
4,EXT_SOURCE_1,0.1553,0.020401,0.1499,2.1756,2,2,2,2,8,strong
...,...,...,...,...,...,...,...,...,...,...,...
562,AMT_REQ_CREDIT_BUREAU_QRT,0.0085,0.001225,NaN,NaN,0,0,0,0,0,very_weak
563,AMT_INCOME_TOTAL,0.0181,0.002680,0.0286,1.1149,0,0,0,0,0,very_weak
564,BUREAU_ACTIVE_AMT_CREDIT_SUM_OVERDUE_SUM,0.0370,0.000194,NaN,NaN,0,0,0,0,0,very_weak
565,SK_ID_CURR,0.0021,0.000000,0.0038,1.0264,0,0,0,0,0,very_weak


In [84]:
from functools import reduce

numerical_target_aware_audit_report = reduce(
    lambda left, right: pd.merge(left, right, on="column", how="outer", suffixes=("", "_dup")),
    [
        train_numerical_correlation_signal_report,
        train_numerical_information_signal_report,
        train_numerical_binned_target_report,
        train_numerical_predictive_summary_report
    ]
)

# Drop any duplicate columns that appeared from the merge
dup_cols = [c for c in numerical_target_aware_audit_report.columns if c.endswith("_dup")]
if dup_cols:
    numerical_target_aware_audit_report = numerical_target_aware_audit_report.drop(columns=dup_cols)

numerical_target_aware_audit_report

,column,sample_size,target_type,pearson_corr_with_target,pearson_pvalue,pearson_ci_lower,pearson_ci_upper,spearman_corr_with_target,spearman_pvalue,point_biserial_corr,...,n_valid_bins,unique_values,reason,max_abs_correlation,corr_score,mi_score,bin_var_score,bin_lift_score,total_score,predictive_power_band
0,AMT_ANNUITY,307499,binary,-0.0128,0.000000,-0.0164,-0.0093,-0.0001,0.967562,-0.0128,...,10,13672,NaN,0.0128,0,1,1,0,2,weak
1,AMT_CREDIT,307511,binary,-0.0304,0.000000,-0.0339,-0.0268,-0.0175,0.000000,-0.0304,...,10,5603,NaN,0.0304,0,1,1,1,3,weak
2,AMT_GOODS_PRICE,307233,binary,-0.0396,0.000000,-0.0432,-0.0361,-0.0315,0.000000,-0.0396,...,10,1002,NaN,0.0396,0,1,2,1,4,moderate
3,AMT_INCOME_TOTAL,307511,binary,-0.0040,0.027238,-0.0075,-0.0004,-0.0181,0.000000,-0.0040,...,10,2548,NaN,0.0181,0,0,0,0,0,very_weak
4,AMT_REQ_CREDIT_BUREAU_DAY,265992,binary,0.0027,0.163084,-0.0011,0.0065,0.0049,0.011224,0.0027,...,0,0,too_few_valid_bins,0.0049,0,0,0,0,0,very_weak
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
562,YEARS_BEGINEXPLUATATION_MEDI,157504,binary,-0.0100,0.000073,-0.0149,-0.0051,-0.0275,0.000000,-0.0100,...,10,245,NaN,0.0275,0,1,0,0,1,very_weak
563,YEARS_BEGINEXPLUATATION_MODE,157504,binary,-0.0090,0.000335,-0.0140,-0.0041,-0.0271,0.000000,-0.0090,...,10,221,NaN,0.0271,0,1,0,0,1,very_weak
564,YEARS_BUILD_AVG,103023,binary,-0.0221,0.000000,-0.0283,-0.0160,-0.0232,0.000000,-0.0221,...,10,149,NaN,0.0232,0,2,0,0,2,weak
565,YEARS_BUILD_MEDI,103023,binary,-0.0223,0.000000,-0.0284,-0.0162,-0.0234,0.000000,-0.0223,...,10,151,NaN,0.0234,0,2,0,0,2,weak


In [85]:
numerical_target_aware_audit_report.to_csv(r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\8.numerical_target_aware.csv")

# Merge All Csv File into One single report 

In [86]:
import logging
from pathlib import Path
from typing import Dict, List, Optional
from functools import reduce

import pandas as pd

logger = logging.getLogger(__name__)


# ─── Configuration ───────────────────────────────────────────────────────────

REPORT_DIR = Path(
    r"C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE"
    r"\FSDS_BOOTCAMP\Machine_Learning_Projects"
    r"\home-credit-default-risk\Report"
)

REPORT_FILES = {
    "identity":                      "1.Identity_report.csv",
    "missingness":                   "2.missingness_report.csv",
    "uniqueness":                    "3.uniqueness_report.csv",
    "categorical_feature_profile":   "4.categorical_feature.csv",
    "categorical_target_audit":      "5.categorical_feature_profile.csv",
    "numerical_feature_profile":     "6.numerical_feature_report.csv",
    "numerical_robustness":          "7.numerical_robustness_report.csv",
    "numerical_target_audit":        "8.final_numerical_target_aware.csv",
}


# ─── Helper: Load all reports ────────────────────────────────────────────────

def load_all_reports(
    report_dir: Path = REPORT_DIR,
    report_files: Dict[str, str] = REPORT_FILES,
) -> Dict[str, pd.DataFrame]:
    """Load all CSV reports into a name → DataFrame dictionary."""

    reports = {}
    for name, filename in report_files.items():
        filepath = report_dir / filename
        if not filepath.exists():
            logger.warning(f"Report file not found, skipping: {filepath}")
            continue
        df = pd.read_csv(filepath)
        logger.info(f"Loaded '{name}': {df.shape[0]} rows × {df.shape[1]} cols")
        reports[name] = df

    return reports


# ─── Helper: Classify reports by granularity ─────────────────────────────────

def _classify_reports(
    reports: Dict[str, pd.DataFrame],
    merge_key: str = "column",
) -> Dict[str, Dict[str, pd.DataFrame]]:
    """
    Classify reports into:
      - 'column_level' : 1 row per column (safe to merge on 'column')
      - 'detail_level' : multiple rows per column (category/bin level)
      - 'no_key'       : doesn't have the merge key at all
    """
    classified = {"column_level": {}, "detail_level": {}, "no_key": {}}

    for name, df in reports.items():
        if merge_key not in df.columns:
            logger.warning(
                f"Report '{name}' has no '{merge_key}' column — cannot merge. "
                f"Columns: {df.columns.tolist()}"
            )
            classified["no_key"][name] = df
            continue

        # Check if it's 1 row per column
        is_unique = df[merge_key].nunique() == len(df)

        if is_unique:
            classified["column_level"][name] = df
            logger.info(f"  '{name}' → column-level (1 row per {merge_key})")
        else:
            classified["detail_level"][name] = df
            logger.info(f"  '{name}' → detail-level (multiple rows per {merge_key})")

    return classified


# ─── Core: Safe merge ────────────────────────────────────────────────────────

def _resolve_duplicate_columns(
    left: pd.DataFrame,
    right: pd.DataFrame,
    merge_key: str,
    right_name: str,
) -> pd.DataFrame:
    """
    Before merging, drop columns from `right` that already exist in `left`
    (except the merge key) to avoid _x / _y suffix pollution.
    """
    overlap = set(left.columns) & set(right.columns) - {merge_key}
    if overlap:
        logger.warning(
            f"Dropping overlapping columns from '{right_name}' "
            f"to avoid duplicates: {overlap}"
        )
        right = right.drop(columns=list(overlap))
    return right


def merge_column_level_reports(
    reports: Dict[str, pd.DataFrame],
    merge_key: str = "column",
    how: str = "outer",
) -> pd.DataFrame:
    """
    Safely merge all column-level (1 row per column) reports.

    Parameters
    ----------
    reports : dict of {name: DataFrame}
        Each DataFrame must have a unique `merge_key` per row.
    merge_key : str, default 'column'
        Column to join on.
    how : str, default 'outer'
        Merge strategy ('inner', 'outer', 'left', 'right').

    Returns
    -------
    pd.DataFrame
        Single merged DataFrame with all signals per column.
    """

    if not reports:
        logger.warning("No column-level reports to merge.")
        return pd.DataFrame()

    names = list(reports.keys())
    dfs = list(reports.values())

    logger.info(f"Merging {len(dfs)} column-level reports: {names}")

    # Start with the first report
    merged = dfs[0].copy()

    for name, df in zip(names[1:], dfs[1:]):
        df_clean = _resolve_duplicate_columns(merged, df, merge_key, name)
        merged = merged.merge(df_clean, on=merge_key, how=how)
        logger.info(
            f"  + merged '{name}' → {merged.shape[0]} rows × {merged.shape[1]} cols"
        )

    return merged


# ─── Main: Full pipeline ────────────────────────────────────────────────────

def build_master_report(
    report_dir: Path = REPORT_DIR,
    report_files: Dict[str, str] = REPORT_FILES,
    merge_key: str = "column",
    how: str = "outer",
    export_path: Optional[str] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Load, classify, and merge all reports into a master report.

    Returns
    -------
    dict with keys:
        'master'        : merged column-level DataFrame (main output)
        'detail_reports' : dict of detail-level DataFrames (not merged)
        'skipped'        : dict of reports without merge key

    Examples
    --------
    >>> result = build_master_report()
    >>> master = result["master"]
    >>> print(master.shape)
    (122, 45)
    >>> master.head()
    """

    # ── Load ─────────────────────────────────────────────────────────────
    reports = load_all_reports(report_dir, report_files)

    if not reports:
        raise FileNotFoundError(f"No report files found in {report_dir}")

    # ── Classify ─────────────────────────────────────────────────────────
    classified = _classify_reports(reports, merge_key)

    column_level = classified["column_level"]
    detail_level = classified["detail_level"]
    skipped = classified["no_key"]

    # ── Merge column-level reports ───────────────────────────────────────
    master = merge_column_level_reports(column_level, merge_key, how)

    logger.info(
        f"\n{'='*60}\n"
        f"Master report: {master.shape[0]} rows × {master.shape[1]} cols\n"
        f"Detail reports (not merged): {list(detail_level.keys())}\n"
        f"Skipped reports (no key): {list(skipped.keys())}\n"
        f"{'='*60}"
    )

    # ── Warn about detail-level reports ──────────────────────────────────
    if detail_level:
        print(
            f"\n⚠️  {len(detail_level)} report(s) have MULTIPLE rows per column "
            f"and were NOT merged into the master report:\n"
        )
        for name, df in detail_level.items():
            n_cols = df[merge_key].nunique()
            print(f"   • {name}: {len(df)} rows across {n_cols} columns")
        print(
            f"\n   Access them via: result['detail_reports']['{list(detail_level.keys())[0]}']"
        )

    # ── Optional export ──────────────────────────────────────────────────
    if export_path:
        master.to_csv(export_path, index=False)
        logger.info(f"Master report exported to: {export_path}")
        print(f"\n✅ Master report saved: {export_path}")

    return {
        "master": master,
        "detail_reports": detail_level,
        "skipped": skipped,
    }


# ─── Usage ───────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(message)s")

    result = build_master_report(
        export_path=str(REPORT_DIR / "9.master_consolidated_report.csv")
    )

    master = result["master"]
    print(f"\n📊 Master Report Shape: {master.shape}")
    print(f"📋 Columns:\n{master.columns.tolist()}")
    print(f"\n{master.head()}")

    # Access detail reports separately
    for name, df in result["detail_reports"].items():
        print(f"\n📂 Detail report '{name}': {df.shape}")

Report file not found, skipping: C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\1.Identity_report.csv
Report file not found, skipping: C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\2.missingness_report.csv
Loaded 'uniqueness': 584 rows × 22 cols
Loaded 'categorical_feature_profile': 584 rows × 34 cols
Loaded 'categorical_target_audit': 584 rows × 34 cols
Loaded 'numerical_feature_profile': 568 rows × 47 cols
Loaded 'numerical_robustness': 568 rows × 56 cols
Report file not found, skipping: C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\8.final_numerical_target_aware.csv
  'uniqueness' → column-level (1 row per column)
  'categorical_feature_profile' → column-level (1 row per column)
  'categorical_target_audit' → column-level (1 row per column)
  'numerical_


✅ Master report saved: C:\Users\USER\DATA-SCINCE-AI\INCEPTION_BD_WORKSPACE\FSDS_BOOTCAMP\Machine_Learning_Projects\home-credit-default-risk\Report\9.master_consolidated_report.csv

📊 Master Report Shape: (584, 140)
📋 Columns:
['Unnamed: 0', 'column', 'dtype', 'non_null_count_x', 'unique_count', 'unique_percentage', 'duplicate_count', 'duplicate_value_ratio', 'constant_flag', 'top_value_percentage', 'quasi_constant_flag', 'top_value', 'top_value_count', 'top_values_preview', 'non_null_count_y', 'dominant_value', 'dominant_value_pct', 'second_dominant_value', 'second_dominant_pct', 'top_values', 'suspicious_uniformity_flag', 'dominance_note', 'non_null_count', 'missing_count_x', 'missing_pct', 'cardinality_count', 'cardinality_group', 'high_cardinality_flag', 'very_high_cardinality_flag', 'potential_id_flag', 'numeric_categorical_flag', 'encoding_recommendation', 'sample_values', 'missing_count_y', 'missing_percentage', 'top_category', 'top_category_percentage', 'rare_category_count', '

In [87]:
master.shape

(584, 140)